# Orbit Wars -- v12 A100 run  *(GENERATED -- do not edit cells here)*

This notebook is **generated from the `orbit_wars_v12/` package** by `scripts/make_v12_nb.py`.
The package is the single source of truth: **edit the modules under `orbit_wars_v12/`, then run
`python scripts/make_v12_nb.py` to regenerate this notebook.** Hand-edits to the cells below are
overwritten on the next regen.

Layout: one shared **imports** cell, then **one cell per module** (relative imports stripped --
every definition shares one global namespace, exactly as the package does once imported), then the
**run** cells (settings -> optional BC warm-start -> train -> save league -> plot) that call the
package's public API.

## Imports

In [ ]:
import math
import os
import random
from dataclasses import dataclass, field, fields, replace
from typing import Optional
import numpy as np
import torch
import torch.nn.functional as F
import contextlib
import torch.nn as nn
import torch.utils.checkpoint as _ckpt
import copy
import glob
import re
import json
import time

## `orbit_wars_v12.config`

Run-tunable hyperparameters for the v12 stack, as a single frozen :class:`Config`.

This replaces the notebook's flat module-level globals (cell 5). Field NAMES are kept UPPER to
match the notebook 1:1 (and the checkpoint ``config`` blob keys), so a port is a uniform
``NAME -> cfg.NAME`` rewrite and existing ``.pt`` files round-trip unchanged. Fixed game/feature
constants are NOT here -- see :mod:`orbit_wars_v12.constants`.

Use :meth:`Config.create` (mirrors the notebook config cell: applies the SMOKE branch, derives
``B``/paths/``device``, and seeds ``random``/``numpy``/``torch``). Plain ``Config()`` yields the
full-run defaults with no global side effects.

In [ ]:
from __future__ import annotations
# Model/optim fields that differ between a full run and a fast SMOKE sanity run. `create(smoke=True)`
# applies these on top of the full-run defaults declared on the dataclass.
_SMOKE_FIELDS = dict(
    HIDDEN=64, N_RES_BLOCKS=2, GRAD_CHECKPOINT=False, USE_COMPILE=False, VALUE_RES_BLOCKS=1,
    NUM_GROUPS=4, GROUP_SIZE=4, EPISODE_STEPS=200, FLEET_CAP=512, TOTAL_ITERS=4,
    SELFPLAY_REFRESH=5, N_WORLDS=256, WORLD_RESAMPLE_EVERY=0, ELO_RECAL_EVERY=0,
    LEARNER_RECAL_EVERY=0, LEAGUE_MAX_SNAPSHOTS=4, MINIBATCHES=16,
)

_DEFAULT_INVITE_CFG = {"HIDDEN": 256, "N_TX_LAYERS": 4, "N_HEADS": 8, "TX_MLP_RATIO": 4,
                       "N_STEM_RES": 1, "N_HEAD_RES": 1, "N_RES_BLOCKS": 6, "VALUE_RES_BLOCKS": 2,
                       "ARCH": "transformer", "USE_GLU": True, "USE_ATTENTION": True, "D_G": 32}


@dataclass(frozen=True)
class Config:
    """All v12 hyperparameters. Defaults = the delivered FULL run (SMOKE=False)."""

    # ---- record of the SMOKE flag used to build this config --------------------------------
    SMOKE: bool = False

    # ---- model size (full spec: width 512, deep ResNet trunk) ------------------------------
    HIDDEN: int = 512              # d: trunk width ("model dim")
    N_RES_BLOCKS: int = 86         # residual MLP blocks in the legacy TRUNK [512x86]
    GRAD_CHECKPOINT: bool = True   # activation-checkpoint trunk blocks (the 512x86 OOM fix)
    USE_COMPILE: bool = True       # torch.compile the LEARNER forward (fuses launch-bound kernels)
    COMPILE_MODE: str = "default"  # "default" = fuse, low VRAM | "reduce-overhead" = CUDA graphs
    ACT_FN: str = "tanh"           # ResNet-MLP activation: 'relu'|'tanh'|'gelu'|'silu'
    # ---- trunk architecture (stacked transformer w/ ResNet-MLP ends) -----------------------
    ARCH: str = "trunk"            # "trunk" = legacy (1x attn + N res-MLP) | "transformer" = stack |
    #                                "blockseq" = arbitrary ordered res/attn sequence from TRUNK_SPEC
    TRUNK_SPEC: str = ""           # [blockseq] e.g. "res16,attn1,res64": res = pre-LN ResNet-MLP block,
    #                                attn = PURE cross-planet multi-head self-attention (uses N_HEADS)
    N_TX_LAYERS: int = 6           # [transformer] encoder layers
    N_HEADS: int = 12              # [transformer] attention heads (HIDDEN must divide by this)
    TX_MLP_RATIO: int = 4          # [transformer] per-layer MLP hidden = ratio x HIDDEN
    N_STEM_RES: int = 1            # [transformer] ResNet-MLP blocks BEFORE the stack
    N_HEAD_RES: int = 4            # [transformer] ResNet-MLP blocks AFTER the stack
    VALUE_RES_BLOCKS: int = 4      # residual MLP blocks in the VALUE/critic head
    D_G: int = 32                  # board-globals embedding dim
    USE_GLU: bool = True
    USE_ATTENTION: bool = False    # cross-planet self-attention block (legacy trunk; A/B off)

    # ---- env / batch (B = NUM_GROUPS * GROUP_SIZE) -----------------------------------------
    NUM_GROUPS: int = 16
    GROUP_SIZE: int = 16
    B: int = 256                   # parallel envs (derived; create() recomputes from groups)
    EPISODE_STEPS: int = 500
    FLEET_CAP: int = 1024          # max simultaneous in-flight fleets/env
    COMETS_ENABLED: bool = True
    COMET_OFFICIAL: bool = True    # official elliptical waypoint comets (CPU precompute)
    COMET_MAX_LEN: int = 40        # official visible-path cap (5..40 waypoints)
    COMET_SPAWN_STEPS: tuple = (50, 150, 250, 350, 450)
    COMET_SPEED: float = 4.0
    COMET_PAIRS: int = 1
    COMET_SPAWN_RADIUS: float = 50.0
    COMET_PERI_MIN: float = 15.0
    COMET_PERI_MAX: float = 30.0
    COMET_EXPIRE_RADIUS: float = 56.0

    # ---- PPO + GAE -------------------------------------------------------------------------
    LR: float = 1e-4
    GAMMA: float = 0.997
    GAE_LAMBDA: float = 0.98
    CLIP: float = 0.2
    CLIP_HI: float = 0.0           # [D3/DAPO clip-higher] asymmetric UPPER clip 1+CLIP_HI (0 -> symmetric = CLIP)
    RATIO_LENGTHNORM: bool = False # [D1/GSPO] divide the per-step log-ratio by #owned planets (geom-mean ratio)
    VF_COEF: float = 0.5
    ENT_COEF: float = 0.005
    MINIBATCHES: int = 64
    UPDATE_EPOCHS: int = 3
    MAX_GRAD_NORM: float = 0.5
    KL_TARGET: float = 0.05        # stop the epoch once a minibatch KL exceeds this
    LOGRATIO_CLAMP: float = 4.0    # clamp log importance-ratio before exp
    LOGIT_CLAMP: float = 8.0       # clamp dest_logits before softplus
    ADAM_EPS: float = 1e-5

    # ---- auxiliary reward-prediction head (UNREAL-style; PPO path) --------------------------
    # A shallow head off the SHARED trunk regresses the immediate per-step reward r_t as a supervised
    # aux loss. It does NOT change the reward, the advantage, or the critic -- it only adds gradient to
    # the trunk representation, which helps when the RL signal is terminal-dominated. Default OFF, so the
    # net (and every existing checkpoint) is byte-identical until flipped; when on, the head is persisted
    # in the ckpt config blob and rebuilt per-member, so pre-aux league snapshots still load unchanged.
    AUX_REWARD_PRED: bool = False   # build + train the reward-prediction head (consumed by ppo_update only)
    AUX_REWARD_COEF: float = 0.25   # weight of the aux reward MSE loss (shared-trunk regularizer)
    # ---- auxiliary WIN-BET head (replaces the naive reward predictor; PPO path) -----------------
    # The SAME shallow scalar head instead bets b_t = tanh(head) in [-1,1] on the eventual game outcome
    # z in [-1,1] (2p: +-1 win/loss; 4p: placement). The bet earns b_t * z (bet +1 & win -> +1; bet +1 &
    # lose -> -1; bet -1 & lose -> +1), so the trunk is pressured to discriminate winning vs losing states
    # -- a far more decision-relevant aux than predicting the immediate reward. Pure representation aux:
    # NOT in the reward / advantage / critic. Mutually exclusive with AUX_REWARD_PRED (win-bet takes
    # precedence). Default OFF; flag persisted in the ckpt blob so league members rebuild correctly.
    AUX_WIN_BET: bool = False        # build + train the win-bet head (maximize bet*outcome; ppo_update only)
    AUX_WIN_BET_COEF: float = 0.25   # weight of the win-bet aux loss (shared-trunk regularizer)
    # ---- ONE-SIDED win-bet REWARD (behaviour shaping; PPO + GRPO + mc) --------------------------
    # Optionally promote the bet from a pure representation aux to an actual per-step REWARD fed into
    # the advantage -- but ONE-SIDED: pay coef * relu(tanh(bet)) * max(z,0). It rewards CONFIDENT WINS
    # and pays EXACTLY ZERO on a draw/loss. The two-sided bet*z reward is a passivity trap (it pays
    # coef/step for confidently LOSING -> the policy learns to bet against itself and throw games, the
    # documented passivity ratchet); rectifying to the winning side removes that gradient entirely. The
    # bet is DETACHED into the reward (no grad through it); the head is still trained as a calibrated bet
    # by the AUX_WIN_BET loss above. REQUIRES AUX_WIN_BET=True. Default OFF; flows through PPO+GAE and the
    # per-step GRPO/mc returns (bypassed only under GRPO_OUTCOME_ONLY, which keeps the terminal payoff).
    AUX_WIN_BET_REWARD: bool = False     # add the one-sided confident-win bet reward into the advantage
    AUX_WIN_BET_REWARD_COEF: float = 0.5 # raw per-step weight (pre PPO_REWARD_SCALE); ~0.5 -> ~75 raw over a won game (<<WIN_BONUS 1000)

    # ---- GRPO (group-relative PO; critic-free alternative to PPO+GAE) -----------------------
    # ALGO selects the optimizer at rollout+update time. "grpo" rolls each world out GRPO_GROUP
    # times and standardizes each rollout's return WITHIN its group as the advantage (no learned
    # value baseline) -- same clipped surrogate + entropy, no value loss, optional KL-to-reference.
    ALGO: str = "ppo"               # "ppo" = PPO+GAE (learned critic) | "grpo" = group baseline (no critic) |
    #                                 "mc" = critic-free, advantage = discounted Monte-Carlo return-to-go G_t
    #                                 (no GAE, no value loss; whitened; uses the grpo clipped-surrogate update)
    GRPO_GROUP: int = 0             # rollouts sharing one world for the relative baseline (0 -> GROUP_SIZE)
    GRPO_ADV_EPS: float = 1e-4      # std floor when standardizing returns within a group
    GRPO_WHITEN: bool = False       # also batch-whiten advantages on top of the per-group baseline
    GRPO_OUTCOME_ONLY: bool = False # baseline on the terminal outcome only (else the full shaped return)
    GRPO_KL_COEF: float = 0.0       # beta: KL(pi||pi_ref) penalty vs a frozen reference (0 -> off, no ref built)
    # ---- GRPO v12 variance-reduction tricks (docs/grpo_v12.tex). ALL default to the legacy path, so
    # an existing run is byte-for-byte unchanged until a flag is flipped; ablate one at a time.
    GRPO_STD_NORM: bool = True      # [B] divide the centered return by the PER-GROUP std (DeepSeek). False ->
    #                                 center-only + ONE global scale (Dr.GRPO: removes the difficulty-inversion bias)
    GRPO_LOO: bool = False          # [B] leave-one-out group mean baseline (RLOO: unbiased, decorrelates b from sample i)
    GRPO_ANALYTIC_ADV: bool = False # [A] 2p binary closed-form advantage at a Beta-shrunk win-rate (kills the
    #                                 divide-by-eps corner + the within-loss-group length signal); 4p falls back to [B]
    GRPO_ANALYTIC_ALPHA: float = 1.0  # [A] Beta(alpha,alpha) pseudocount shrinking p_hat off {0,1}
    GRPO_PHI_VALUE: bool = False    # [C] use the shaping potential Phi as an OLS-fit GAE value (per-step credit);
    #                                 turns Phi from reward shaping (cancelled by the group mean) into a baseline
    GRPO_PHI_A_MAX: float = 3.0     # [C] clamp on the fitted Phi->outcome slope (guards a degenerate OLS fit)
    GRPO_CRN: bool = False          # [D] common random numbers: couple NEURAL-opponent sampling noise within a
    #                                 world-group so the baseline isolates ego-action variance (lower-variance adv)
    # ---- v12 GRPO advantage-estimator + variance flags (Tier 0-2; docs/grpo_v12.tex). WARNING: these
    #      "anti-collapse" estimators (rank/cvar/sibling + center-only/Dr.GRPO) all caused PASSIVITY in the
    #      3070 Ti ablation (docs/grpo_v12.tex Section "Empirical result"); the KEEPER is the LEGACY default
    #      (GRPO_STD_NORM=True + symmetric decay). The ESTIMATOR flags are MUTUALLY EXCLUSIVE (train.py prints
    #      the active one + warns); all default off. Re-ablate on the 3070 Ti before any A100 use.
    ADV_TARGET_RMS: float = 0.0     # [Tier0/D4] pin the (center-only) advantage RMS to this (0 -> legacy whiten).
    #                                 Stops league-difficulty from becoming a stealth LR schedule via Adam's eps.
    GRPO_RANK_ADV: bool = False     # [C1] within-group RANK advantage (van der Waerden / NES rank-shaping): bounded
    #                                 in [-sqrt3,sqrt3] -- but EMPIRICALLY went passive (bounded != good; discards margin)
    GRPO_RB_GATE: bool = False      # [A1] Rao-Blackwellize the launch gate: de-noise the WHERE gradient with p_i
    #                                 (expected fire) instead of the sampled 0/1 fire_i (exact, zero extra fwd)
    GRPO_OPP_BASELINE_W: float = 0.0  # [E1] subtract the league's Elo-expected outcome vs THIS opponent before
    #                                 centering (a no-op under 1-opponent-per-iter centering; enables E2 stratification)
    GRPO_CVAR_ALPHA: float = 0.0    # [C2] risk-sensitive: baseline on the worst-alpha quantile (0 -> off). 0.5 = mild
    GRPO_CVAR_LAMBDA: float = 1.0   # [C2] extra gradient weight on the tail (worst-alpha) rollouts
    GRPO_AUX_VALUE: bool = False    # [B2/C5] train the (else-unused) value head by MC regression to the return as a
    #                                 representation aux (does NOT enter the critic-free advantage; weights stay PPO-compat)
    GRPO_AUX_COEF: float = 0.1      # [B2] weight of the value aux loss (kept small: shared-trunk regularizer)
    GRPO_SIBLING_BASELINE: bool = False  # [B1] per-step baseline from matched-prefix siblings (shared-root VinePPO):
    #                                 phi_ship-bucketed group mean at each step -> per-step credit (critic-free GAE)
    GRPO_SIBLING_BUCKET_DELTA: float = 0.5  # [B1] phi_ship bucket width (log-ship units) for sibling matching
    GRPO_SIBLING_MIN_PEERS: int = 4         # [B1] fall back to the whole-group mean when a bucket is thinner than this
    GRPO_OPP_STRATA: int = 1        # [E2] RESERVED (not yet wired): would sample this many opponents/iter, one per
    #                                 block of groups, so the baseline averages opponent strength. Needs a per-block
    #                                 opponent rollout (invasive hot-loop change); E1 above is its no-op-until-then companion.

    # ---- training length (the Elo/PFSP league IS the curriculum) ---------------------------
    TOTAL_ITERS: int = 600
    SELFPLAY_REFRESH: int = 25     # snapshot the learner into the pool every N iters
    SNAPSHOT_WARMUP: int = 0       # anchors-only warmup before the first learner snapshot

    # ---- self-play Elo LEAGUE (PFSP pool -- the sole opponent source) ----------------------
    LEAGUE_MAX_SNAPSHOTS: int = 16
    LEAGUE_INIT_CKPTS: list = field(default_factory=list)
    ELO_INIT: float = 0.0
    ELO_RANDOM: float = 0.0        # ANCHORED rating of the random bot -> pins the Elo scale
    ELO_STARTER: float = 1350.0
    ELO_MEDIUM: float = 1560.0
    ELO_GREEDY: float = 1500.0
    ELO_INTERMEDIATE: float = 1425.0
    ADVANCE_TRIGGER_ELO: float = 1500.0
    ADVANCE_CONFIRM_ELO: float = 1350.0
    ELO_K: float = 32.0
    ELO_SCALE: float = 300.0 / math.log10(9.0)   # 314.38: a 300-Elo gap = 90% expected
    ELO_GROUND_KINDS: tuple = ("greedy", "medium")
    MASTER_EVICT_2P_WR: float = 0.90
    MASTER_EVICT_4P_WR: float = 0.75
    MASTER_KEEP_KINDS: tuple = ("random", "greedy")
    PFSP_MODE: str = "even"        # "even" (p*(1-p)) / "hard" ((1-p)^P) / "uniform"
    PFSP_POWER: float = 2.0
    PFSP_FLOOR: float = 0.05
    MATCH_ELO_W: float = 0.7
    MATCH_FP_W: float = 0.3
    SCRIPT_MATCH_BOOST: float = 2.0
    SCRIPT_BOOST_DROP_WR: float = 0.90
    SCRIPT_BOOST_MIN_GAMES: int = 5
    WR_EMA_BETA: float = 0.1
    CAPTURE_RADIUS: float = 25.0
    CAPTURE_MARGIN: float = 2.0
    REBALANCE_PCT: float = 0.4
    REBALANCE_RADIUS: float = 30.0

    # ---- gated allocation action space + sun-reachability ----------------------------------
    REACH_MASK: bool = True
    LEAD_TARGET: bool = True
    WHERE_DIST: str = "categorical"   # 'categorical' (v7 default) | 'dirichlet' (v6 A/B)
    ALLOC_KAPPA: float = 1.0
    MIN_LAUNCH_SHIPS: int = 2
    GATE_TRIM_LO: float = 0.2
    GATE_TRIM_HI: float = 0.8
    GATE_EPS: float = 0.02
    GREEDY_SAMPLE_GATE: bool = True
    INIT_GATE_BIAS: float = -1.0
    POLICY_MODE: str = "ppo"
    LAUNCH_REWARD: float = 0.001
    SELF_LAUNCH_REWARD: float = 0.001
    SELF_LAUNCH_CAP: float = 10.0
    LAUNCH_STEP_CAP: float = 0.5
    LAUNCH_WINDOW: int = 150
    LAUNCH_GAME_CAP: float = 30.0

    # ---- reward (v5; docs/set-ups/1.md) ----------------------------------------------------
    # v13: terminal W/L halved to +-500 and the residual ~500 of outcome magnitude is moved into a
    # DENSE per-step survival reward (ALIVE_REWARD below) -- every alive step pays, so credit is spread
    # across the trajectory instead of one sparse +-1000 spike at the end (easier value learning / lower
    # return variance). NOTE this is a survival bonus, NOT Ng et al. potential shaping (it is not
    # policy-invariant): it rewards staying alive, so watch lnch/st for a passivity drift.
    WIN_BONUS: float = 500.0
    LOSS_PENALTY: float = 500.0
    ALIVE_REWARD: float = 2.0      # raw reward per step the ego is alive (pre-PPO_REWARD_SCALE); ~2*len over a game
    WIN_DECAY: float = 1.0
    LOSS_DECAY: float = 0.9995     # asymmetric "lose-slower": a late loss is penalized slightly less than an
    #                                early one (a survival incentive); winning is undecayed (WIN_DECAY=1.0)
    # ---- v13 submission: WIN POOL terminal + dense shared cap (all default OFF -> legacy unchanged) ----
    # USE_WIN_POOL recasts the win side as a FIXED POOL of WIN_POOL raw reward: the ego draws ALIVE_REWARD
    # per alive step (the survival drip) and, on a WIN, collects the REST of the pool (WIN_POOL -
    # ALIVE_REWARD*len, clamped >=0). So a won game pays EXACTLY WIN_POOL total regardless of length -- the
    # drip just spreads it across the trajectory for credit assignment (discounting still rewards winning
    # FAST: the big remainder arrives earlier = less discounted). On a NON-win the drip is CLAWED BACK at
    # terminal so a long-surviving loss can't net positive (the passivity ratchet): a loser nets exactly
    # -decayed(LOSS_PENALTY). Replaces WIN_DECAY^len * WIN_BONUS for the win side ONLY; the loss side keeps
    # LOSS_DECAY^len_eff * LOSS_PENALTY. Pair with LOSS_PENALTY=1000 for the documented 1200/-1000 design.
    USE_WIN_POOL: bool = False
    WIN_POOL: float = 1200.0       # total raw win reward (drip + terminal remainder); >= ALIVE_REWARD*max_len
    # USE_DENSE_GAME_CAP applies ONE shared hard cap (DENSE_GAME_CAP raw) to the SUM of the legacy dense
    # channels over a game -- capture + prod-milestone + launch share it (their per-step ratio is preserved
    # when the cap binds). Supersedes the launch-only LAUNCH_GAME_CAP as the binding ceiling. Keeps the small
    # shaped channels from ever rivalling the +-WIN_POOL/LOSS_PENALTY terminal outcome.
    USE_DENSE_GAME_CAP: bool = False
    DENSE_GAME_CAP: float = 250.0  # shared game cap (raw) on capture + prod-milestone + launch combined
    # DENSE_WITH_SHAPING lets the legacy dense channels (capture/prod-milestone/launch) COEXIST with Ng et al.
    # potential shaping instead of being replaced by it: when USE_POTENTIAL_SHAPING=True AND this is True,
    # BOTH are summed into the per-step reward (shaping is policy-invariant + telescoping, so it is NOT inside
    # the DENSE_GAME_CAP). WARNING: ship-margin shaping overlaps the capture channel -- watch for double-count.
    DENSE_WITH_SHAPING: bool = False
    DECAY_START_STEP: float = 100.0
    CAPTURE_REWARD: float = 30.0
    CAPTURE_LOSS_FRAC: float = 0.9
    CAPTURE_PROD_SCALE: float = 0.2
    PROD_MILESTONE_REWARD: float = 20.0
    PROD_MILESTONE_BASE: float = 100.0
    PPO_REWARD_SCALE: float = 100.0
    DENSE_REWARD_SCALE: float = 1.0   # multiplier on the capture + prod-milestone channels ONLY (launch is EXEMPT
    #   -- it is the anti-passivity activity floor). <1 makes the terminal W/L (+-WIN_BONUS) dominate the
    #   integrated return. CAUTION under critic-free ALGO mc/grpo: too low (e.g. 0.2) starves per-step credit
    #   assignment (no critic baseline -> every step of a losing game gets the same advantage) and triggers the
    #   passivity ratchet. ~0.5 keeps enough dense signal; watch lnch/st as the canary. Use ppo for true sparse-W/L.

    # ---- v6: compute-bound A100 + gate/entropy/popart/shaping ------------------------------
    USE_AMP: bool = True
    AMP_DTYPE: torch.dtype = torch.bfloat16
    GREEDY_TAU: float = 0.5
    INIT_DEST_SCALE: float = 0.02
    DEST_HEAD: str = "bilinear"    # 'linear' (per-source) | 'bilinear' (query=src, key=dst)
    ENT_DIR_COEF: float = 0.0
    ENT_COEF_START: float = 0.005
    ENT_COEF_END: float = 0.002
    ENT_DECAY_ITERS: int = 300
    KL_EARLYSTOP: bool = True
    USE_POPART: bool = True
    POPART_BETA: float = 3e-4
    USE_POTENTIAL_SHAPING: bool = False   # default = legacy SMALL dense reward (capture + prod-milestone +
    #   launch) so the terminal W/L (+-10 after PPO_REWARD_SCALE) dominates. True -> policy-invariant
    #   potential shaping (ship-margin + prod-share, Ng et al.) via SHAPE_SHIP/SHAPE_PROD.
    SHAPE_SHIP: float = 25.0
    SHAPE_PROD: float = 25.0

    # ---- checkpointing (filled by create() from OW_CKPT_DIR) -------------------------------
    CKPT_DIR: str = "."
    CKPT_PATH: str = "setup1_target_policy.pt"
    BEST_CKPT_PATH: str = "setup1_target_policy_best.pt"
    TRAIN_STATE_PATH: str = "setup1_target_train_state.pt"
    CKPT_EVERY: int = 20

    # ---- BC warm-start + PPO stability guardrails (v7) -------------------------------------
    BC_ENABLED: bool = False
    BC_ROUNDS: int = 12
    BC_EPOCHS: int = 2
    BC_LR: float = 3e-4
    BC_MINIBATCH: int = 4096
    BC_CKPT_PATH: str = "bc_medium_init.pt"
    VALUE_WARMUP_ITERS: int = 5
    LR_WARMUP_ITERS: int = 16
    KL_STOP_MINIBATCH: bool = True
    TRIPWIRE_FRAC: float = 0.4
    TRIPWIRE_BASE_ITERS: int = 10
    KL_HARD_MULT: float = 1.5

    # ---- world pool ------------------------------------------------------------------------
    N_WORLDS: int = 2048
    SEED: int = 0
    WORLD_RESAMPLE_EVERY: int = 100
    ELO_RECAL_EVERY: int = 500
    RESUME_FROM: Optional[str] = None
    ELO_RECAL_ENVS: int = 64

    # ---- v8: mixed 2p/4p + dual Elo + invited league ---------------------------------------
    FOURP_ENABLED: bool = True
    B_4P: int = 256                # parallel envs per 4p iter (derived = B)
    OUTCOME_4P: str = "placement"  # "placement" (linear) | "winner" (official top-only)
    SEAT_SWAP: bool = True
    S4_BASE: float = 0.25          # [pin 3:1 2p:4p] baseline share of 4p iters
    S4_MIN: float = 0.25
    S4_MAX: float = 0.25           # upper == lower -> share_4p pinned at 0.25
    S4_GAIN: float = 1.0
    ELO_K4: float = 16.0
    ELO_COUPLING: float = 0.25
    INVITE_DIRS: list = field(default_factory=lambda: [
        os.environ.get("OW_LEAGUE_DIR", ""), "/content/drive/MyDrive/league_agents",
        "league_invited", "league_agents", os.path.join("notebooks", "league_agents")])
    INVITE_MAX: int = 4
    INVITE_DEFAULT_ELO: float = 750.0
    INVITE_ELO4_OFFSET: float = 0.0
    INVITE_MATCH_BOOST: float = 2.0
    INVITE_MASTER_WR: float = 0.75
    INVITE_MASTERED_W: float = 0.2
    INVITE_CFG: dict = field(default_factory=lambda: dict(_DEFAULT_INVITE_CFG))
    # ---- v9.2: scripted-exposure caps + rating-noise control -------------------------------
    SCRIPT_SHARE_CAP: float = 0.30
    MAX_SCRIPT_SEATS_4P: int = 1
    LEARNER_RECAL_EVERY: int = 50
    RECAL_BLEND: float = 0.5
    S4_EMA_BETA: float = 0.10
    GAUNTLET_NEURAL_W: float = 0.5
    INVITE_MEASURE: bool = True
    GAUNTLET_4P_W: float = 0.5
    # ---- v12: dormant calibration watchdogs ------------------------------------------------
    WATCHDOG_KINDS: tuple = ("random", "greedy")
    WATCHDOG_TOL: float = 0.10

    # ---- runtime ---------------------------------------------------------------------------
    device: torch.device = field(
        default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu"))

    # -----------------------------------------------------------------------------------------
    @classmethod
    def create(cls, smoke: bool = False, **overrides) -> "Config":
        """Build a config the way the notebook config cell does: apply the SMOKE branch, then
        any `overrides`, derive `B`/`B_4P`/checkpoint paths/`device`, seed RNGs, set matmul
        precision, and create the checkpoint dir. Overrides win over the SMOKE branch (so e.g.
        `B=8` directly is honoured, matching the notebook smoke harness)."""
        vals: dict = {"SMOKE": smoke}
        if smoke:
            vals.update(_SMOKE_FIELDS)
        vals.update(overrides)

        defaults = {f.name: f for f in fields(cls)}

        # B / B_4P (derive from groups unless given explicitly)
        ng = vals.get("NUM_GROUPS", defaults["NUM_GROUPS"].default)
        gs = vals.get("GROUP_SIZE", defaults["GROUP_SIZE"].default)
        vals.setdefault("B", ng * gs)
        vals.setdefault("B_4P", vals["B"])

        # checkpoint paths (Colab/Kaggle-friendly, like the notebook)
        ckpt_dir = (vals.get("CKPT_DIR") or os.environ.get("OW_CKPT_DIR")
                    or ("/kaggle/working" if os.path.isdir("/kaggle/working")
                        else "/content" if os.path.isdir("/content") else "."))
        vals["CKPT_DIR"] = ckpt_dir
        vals.setdefault("CKPT_PATH", os.path.join(ckpt_dir, "setup1_target_policy.pt"))
        vals.setdefault("BEST_CKPT_PATH", os.path.join(ckpt_dir, "setup1_target_policy_best.pt"))
        vals.setdefault("TRAIN_STATE_PATH", os.path.join(ckpt_dir, "setup1_target_train_state.pt"))
        vals.setdefault("BC_CKPT_PATH", os.path.join(ckpt_dir, "bc_medium_init.pt"))
        vals.setdefault("INVITE_DIRS", [os.environ.get("OW_LEAGUE_DIR", ""),
                                        "/content/drive/MyDrive/league_agents",
                                        os.path.join(ckpt_dir, "league_invited"), "league_agents",
                                        os.path.join("notebooks", "league_agents")])
        vals.setdefault("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

        # global side effects matching the notebook config cell
        os.makedirs(ckpt_dir, exist_ok=True)
        torch.set_float32_matmul_precision("high")
        seed = int(vals.get("SEED", defaults["SEED"].default))
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        valid = {f.name for f in fields(cls)}
        unknown = set(vals) - valid
        if unknown:
            raise TypeError("unknown Config fields: %s" % ", ".join(sorted(unknown)))
        if vals.get("AUX_WIN_BET_REWARD") and not vals.get("AUX_WIN_BET"):
            raise ValueError("AUX_WIN_BET_REWARD requires AUX_WIN_BET=True (the bet head must be trained)")
        return cls(**vals)

    def replace(self, **changes) -> "Config":
        """Return a copy with `changes` applied (does not re-seed or recompute derived fields)."""
        return replace(self, **changes)

## `orbit_wars_v12.constants`

Fixed game/feature constants for the v12 stack.

These mirror the official Orbit Wars engine + the C++ core (`core/state.hpp`,
`core/encode.hpp`) and the feature layout the policy is trained on. They are PHYSICS, not
configuration: they never vary between runs. Run-tunable hyperparameters live in
:class:`orbit_wars_v12.config.Config`.

Ported verbatim from `notebooks/setup1_v12_a100.ipynb` cells 3 (DTYPE), 7 (board + feature
layout), and 11 (world-generator group bounds).

In [ ]:
# ---- numeric convention --------------------------------------------------------------------
DTYPE = torch.float32  # integer-in-float32 convention (mirrors the native env)

# ---- board constants (core/state.hpp) ------------------------------------------------------
BOARD_SIZE = 100.0
CENTER = BOARD_SIZE / 2.0
SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
COMET_RADIUS = 1.0
COMET_PRODUCTION = 1
PI = math.pi

# ---- planet / ship structural dims (fixed across every version) ----------------------------
PLANET_CAP = 48   # E: planet slots/env == dest-categorical size (comets omitted)
SHIP_SPEED = 6.0

# ---- feature layout (core/encode.hpp) ------------------------------------------------------
# N_SOON/N_BIG are LEGACY names from the old "15 soonest + 15 biggest" threat split; the encoder now
# takes the TOP-30 inbound fleets BY SIZE (no soonest/biggest split), so these are just a 15+15
# decomposition of the 30 slots and carry no separate meaning.
N_SOON = 15
N_BIG = 15
N_THREAT_FLEETS = N_SOON + N_BIG            # 30 inbound-fleet slots per planet (top-30 by size)
# per-fleet threat descriptor: a 4-channel EGO-RELATIVE seat ONE-HOT owner [self, enemy+1, +2, +3]
# (v12: replaces the single self/enemy SIGN, which collapsed all opponents into one channel in 4p)
# + eta + raw ships (per-fleet ship-log compression removed; un-normalized integer count).
N_THREAT_FEATS = 6
# v13: body = the 14 v9/v12 channels + 4 MOTION channels (vx, vy, ax, ay): the per-planet velocity
# and curvature, backward finite-diff of position, UNIFORM across static/orbital/comet bodies (the new
# info is comet motion -- orbital already had vmag/cw; static is 0). Appended AFTER the v12 channels so
# indices 0..13 are unchanged (only the threat-block offset shifts). See docs/v13_motion.md.
N_BODY_FEATURES_V12 = 14   # pre-v13 body width (no motion vectors) -> legacy down-convert target
N_BODY_FEATURES = 18   # v13: + vx,vy,ax,ay  (v9: ownership = 4-channel seat one-hot, was 1 scalar -> +3)
N_ENTITY_FEATURES = N_BODY_FEATURES + N_THREAT_FEATS * N_THREAT_FLEETS   # 18 + 6*30 = 198
N_GLOBAL_FEATURES = 10
F_DIM = N_ENTITY_FEATURES
G_DIM = N_GLOBAL_FEATURES
# Older observation layouts kept playable as league members via checkpoint.LegacyObsAdapter, which
# down-converts the CURRENT obs to the member's layout each forward:
#   F_DIM_NO_MOTION    (194): pre-v13 v12 layout (14 body, no motion vectors; SAME 6-feat threat) ->
#                             drop the 4 motion channels, threat passes through uncollapsed.
#   F_DIM_BINARY_THREAT (104): v9-v12 single self/enemy fleet SIGN (collapse the 4-ch owner one-hot).
#   F_DIM_SCALAR_OWNER  (101): v7/v8, additionally SCALAR body ownership (collapse the body one-hot).
# All pinned to N_BODY_FEATURES_V12 (the historical 14-body width), NOT the live N_BODY_FEATURES.
F_DIM_NO_MOTION = N_BODY_FEATURES_V12 + N_THREAT_FEATS * N_THREAT_FLEETS   # 194: 14 body + 30x6 threat
F_DIM_BINARY_THREAT = N_BODY_FEATURES_V12 + 3 * N_THREAT_FLEETS           # 104: 14 body + 30x3 threat
F_DIM_SCALAR_OWNER = (N_BODY_FEATURES_V12 - 3) + 3 * N_THREAT_FLEETS      # 101: 11 body + 30x3 threat

SHIP_LOG_DENOM = math.log(1000.0)
DIAG_HALF = math.sqrt(BOARD_SIZE * BOARD_SIZE + BOARD_SIZE * BOARD_SIZE) / 2.0
THREAT_MAX_SPEED = 6.0      # == ship speed; fleet speed cap for the ETA model
THREAT_ETA_SCALE = 100.0
# v13 motion-feature normalisers (per-planet body channels 14..17 = vx,vy,ax,ay).
V_SCALE = THREAT_MAX_SPEED  # velocity ~ COMET_SPEED(4)/orbital chord(<=3.5) -> ~0.6 normalised
A_SCALE = THREAT_MAX_SPEED  # curvature (2nd diff) is small; placeholder == V_SCALE, tune from measured |accel|
BIG = 1e18

# ---- world-generator group bounds (REFERENCE_orbit_wars.py::generate_planets) --------------
MIN_PLANET_GROUPS = 5
MAX_PLANET_GROUPS = 10
MIN_STATIC_GROUPS = 3
PLANET_CLEARANCE = 7


def ship_log_t(x):
    return torch.log1p(x.clamp_min(0.0)) / SHIP_LOG_DENOM


def fleet_speed_t(ships, vmax=SHIP_SPEED):
    # 1 + (vmax-1)*(log(ships)/log(1000))^1.5, capped, ships>=1  (core/encode.hpp)
    n = ships.clamp_min(1.0)
    v = 1.0 + (vmax - 1.0) * torch.pow(torch.log(n) / math.log(1000.0), 1.5)
    return v.clamp_max(vmax)

## `orbit_wars_v12.distributions`

Gated per-source action distributions (notebook cell 9).

The actor emits, per owned source planet, a WHERE distribution over destinations plus a launch
GATE. On fire, the planet dispatches its FULL garrison routed by the WHERE row; on no-fire it
holds. Two variants share the same ``(B, E, E+1)`` action bundle (``[...,:E]`` = allocation rows,
``[..., E]`` = fire in ``{0,1}``):

* :class:`GatedAllocDist` -- Dirichlet rows (v6 A/B).
* :class:`GatedCatDist`   -- categorical destinations (v7 default; bounded log-prob, BC-stable).

Both take the :class:`~orbit_wars_v12.config.Config` so the clamps / gate trims / entropy weights
that were notebook globals are read off ``cfg``.

In [ ]:
def _rb_gate_where(cfg, fire, where_lp, p):
    """[A1] The WHERE term of the per-planet log-prob. Default = sampled ``fire * where_lp``.
    With ``cfg.GRPO_RB_GATE`` it is Rao-Blackwellized over the launch Bernoulli: a straight-through
    that keeps the FORWARD value at ``fire * where_lp`` (so the importance ratio stays an honest
    on-policy ratio) while routing the GRADIENT through ``p.detach() * where_lp`` -- the conditional
    expectation of the WHERE score over the gate, removing the gate's 0/1 sampling variance exactly
    (E[fire]=p) without injecting any spurious gate-direction gradient."""
    if not cfg.GRPO_RB_GATE:
        return fire * where_lp
    pc = p.detach()
    return (fire * where_lp).detach() + pc * where_lp - (pc * where_lp).detach()


class GatedAllocDist:
    """Gated per-source Dirichlet ALLOCATION actor (v6).
      WHERE  dest_logits (B,E,E) -> alpha = softplus(logits)*kappa + eps -> Dirichlet rows over dests.
      GATE   gate_logits (B,E)   -> p = clamp((sigmoid-LO)/(HI-LO),0,1), trimmed to [eps,1-eps]; fire ~ Bernoulli(p).
    On fire the planet dispatches its FULL garrison routed by its WHERE row; on no-fire it holds.
    Action (B,E,E+1): [...,:E]=allocation rows, [...,E]=fire in {0,1}. Log-prob/entropy SUM over owned sources;
    the WHERE term only counts when firing.
    v6: greedy()/DEPLOY uses a TEMPERATURE-SHARPENED softmax over the LIVE+REACHABLE off-diagonal dest logits
    (concentration-aware) so the deployed policy lands a CONCENTRATED strike -- the Dirichlet MEAN spreads the
    full garrison thin and the decode floor then dropped every launch (the residual greedy-passive bug)."""
    def __init__(self, cfg, dest_logits, gate_logits, owned, alive=None, reach=None, ships=None,
                 kappa=None, deploy=False):
        self.cfg = cfg
        kappa = cfg.ALLOC_KAPPA if kappa is None else kappa
        self.owned = owned if owned.dim() == 2 else owned.squeeze(-1)        # (B,E) legal sources
        logits = dest_logits.clamp(-cfg.LOGIT_CLAMP, cfg.LOGIT_CLAMP)
        self.alpha = F.softplus(logits) * kappa + 1e-3                       # (B,E,E)
        self.dist = torch.distributions.Dirichlet(self.alpha)
        g = torch.sigmoid(gate_logits if gate_logits.dim() == 2 else gate_logits.squeeze(-1))   # (B,E)
        if deploy:                                                          # DEPLOY: hard deadzone (fire/hold trim)
            p = ((g - cfg.GATE_TRIM_LO) / (cfg.GATE_TRIM_HI - cfg.GATE_TRIM_LO)).clamp(0.0, 1.0)
        else:                                                              # TRAIN: SMOOTH fire-prob so the gate head
            p = g                                                          #   keeps a full-support gradient (the
                                                                           #   deadzone clamp zeroed it -> grad vanish)
        self.p = p.clamp(cfg.GATE_EPS, 1.0 - cfg.GATE_EPS)                  # (B,E) trainable fire-prob
        E = logits.shape[-1]
        eye = torch.eye(E, dtype=torch.bool, device=logits.device).unsqueeze(0)
        dead = eye.expand_as(logits).clone()                                # drop self (decode drops it)
        if alive is not None:
            dead = dead | (alive < 0.5).unsqueeze(1)                        # drop dead dests
        if reach is not None:
            dead = dead | (reach < 0.5)                                     # drop sun-blocked dests
        self.glogits = logits.masked_fill(dead, -1e9)                       # masked logits for the greedy WHERE

    def _bundle(self, alloc, fire):
        return torch.cat([alloc, fire.unsqueeze(-1)], -1)                   # (B,E,E+1)

    def sample(self, crn_group=0):    # crn_group accepted for call-site parity; CRN coupling (docs/grpo_v12.tex [D])
        #                               is implemented for the (default) categorical WHERE, not the Dirichlet
        return self._bundle(self.dist.rsample(), torch.bernoulli(self.p))  # WHERE simplex + Bernoulli fire

    def greedy(self):
        alloc = torch.softmax(self.glogits / self.cfg.GREEDY_TAU, -1)      # sharpened, mask-aware -> concentrated
        fire = torch.bernoulli(self.p) if self.cfg.GREEDY_SAMPLE_GATE else (self.p >= 0.5).float()
        return self._bundle(alloc, fire.to(alloc.dtype))                   # DEPLOY: gate fires at the TRAINED rate

    def log_prob(self, a):
        E = self.alpha.shape[-1]
        alloc = a[..., :E].clamp_min(1e-6); alloc = alloc / alloc.sum(-1, keepdim=True)   # project to simplex
        fire = a[..., E]                                                   # (B,E) in {0,1}
        where_lp = self.dist.log_prob(alloc)                              # (B,E)
        gate_lp = fire * torch.log(self.p.clamp_min(1e-8)) + (1.0 - fire) * torch.log((1.0 - self.p).clamp_min(1e-8))
        lp = gate_lp + _rb_gate_where(self.cfg, fire, where_lp, self.p)   # [A1] Rao-Blackwell de-noise (or fire*where_lp)
        return (lp * self.owned).sum(1)                                   # (B,)

    def entropy(self):
        p = self.p
        gate_e = -(p * torch.log(p.clamp_min(1e-8)) + (1.0 - p) * torch.log((1.0 - p).clamp_min(1e-8)))
        # Dirichlet differential entropy is unboundedly NEGATIVE when peaked -> as a bonus it
        # blurs a BC-sharpened WHERE head (measured: loss +20, gn 1380, kl 4.8). Gate-only by default.
        ent = gate_e + self.cfg.ENT_DIR_COEF * p * self.dist.entropy()
        return (ent * self.owned).sum(1)                                  # (B,)


class GatedCatDist:
    """Gated per-source CATEGORICAL destination actor (v7 default).
      WHERE  masked logits glogits (B,E,E) -> Categorical over dests; action row = one-hot(dest).
      GATE   identical to GatedAllocDist (smooth p at train, hard trim at deploy).
    Same (B,E,E+1) action bundle as the Dirichlet -> decode/buffers/league untouched.
    log_prob is bounded-sensitivity (log_softmax), which is what makes PPO-after-BC stable."""
    def __init__(self, cfg, dest_logits, gate_logits, owned, alive=None, reach=None, ships=None, deploy=False):
        self.cfg = cfg
        self.owned = owned if owned.dim() == 2 else owned.squeeze(-1)
        logits = dest_logits.clamp(-cfg.LOGIT_CLAMP, cfg.LOGIT_CLAMP)
        g = torch.sigmoid(gate_logits if gate_logits.dim() == 2 else gate_logits.squeeze(-1))
        if deploy:
            p = ((g - cfg.GATE_TRIM_LO) / (cfg.GATE_TRIM_HI - cfg.GATE_TRIM_LO)).clamp(0.0, 1.0)
        else:
            p = g
        self.p = p.clamp(cfg.GATE_EPS, 1.0 - cfg.GATE_EPS)
        E = logits.shape[-1]
        eye = torch.eye(E, dtype=torch.bool, device=logits.device).unsqueeze(0)
        dead = eye.expand_as(logits).clone()
        if alive is not None:
            dead = dead | (alive < 0.5).unsqueeze(1)
        if reach is not None:
            dead = dead | (reach < 0.5)
        self.glogits = logits.masked_fill(dead, -1e9)
        self.lsm = torch.log_softmax(self.glogits, -1)                      # (B,E,E)

    def _bundle(self, dest, fire):
        alloc = torch.zeros_like(self.lsm)
        alloc.scatter_(2, dest.unsqueeze(-1), 1.0)                          # one-hot WHERE row
        return torch.cat([alloc, fire.unsqueeze(-1)], -1)

    def sample(self, crn_group=0):
        B_, E, _ = self.lsm.shape
        if crn_group and crn_group > 1 and B_ % crn_group == 0:
            # CRN (docs/grpo_v12.tex [D]): share the WHERE Gumbel + GATE uniform noise across the
            # crn_group rollouts of one world-group (envs are laid out in contiguous groups), so only
            # the ACTING agent's own randomness varies within a group -> lower-variance group baseline.
            G = crn_group; nW = B_ // G
            u = torch.rand(nW, 1, E, E, device=self.lsm.device).clamp_(1e-9, 1.0)
            gumbel = -torch.log(-torch.log(u))                              # Gumbel-max == Categorical(lsm)
            dest = (self.lsm.view(nW, G, E, E) + gumbel).argmax(-1).reshape(B_, E)
            ug = torch.rand(nW, 1, E, device=self.p.device)
            fire = (ug < self.p.view(nW, G, E)).to(self.lsm.dtype).reshape(B_, E)
            return self._bundle(dest, fire)
        dest = torch.distributions.Categorical(logits=self.lsm.reshape(-1, E)).sample().view(B_, E)
        return self._bundle(dest, torch.bernoulli(self.p))

    def greedy(self):
        fire = torch.bernoulli(self.p) if self.cfg.GREEDY_SAMPLE_GATE else (self.p >= 0.5).float()
        return self._bundle(self.glogits.argmax(-1), fire.to(self.lsm.dtype))

    def log_prob(self, a):
        E = self.lsm.shape[-1]
        dest = a[..., :E].argmax(-1)                                        # rows are one-hot
        fire = a[..., E]
        where_lp = self.lsm.gather(2, dest.unsqueeze(-1)).squeeze(-1)
        gate_lp = fire * torch.log(self.p.clamp_min(1e-8)) + (1.0 - fire) * torch.log((1.0 - self.p).clamp_min(1e-8))
        lp = gate_lp + _rb_gate_where(self.cfg, fire, where_lp, self.p)    # [A1] Rao-Blackwell de-noise (or fire*where_lp)
        return (lp * self.owned).sum(1)

    def entropy(self):
        p = self.p
        gate_e = -(p * torch.log(p.clamp_min(1e-8)) + (1.0 - p) * torch.log((1.0 - p).clamp_min(1e-8)))
        cat_e = -(self.lsm.exp() * self.lsm.clamp_min(-30.0)).sum(-1)       # bounded (<= ln E)
        ent = gate_e + self.cfg.ENT_DIR_COEF * p * cat_e
        return (ent * self.owned).sum(1)

## `orbit_wars_v12.worldgen`

Pure-Python world generator (notebook cell 11).

Faithful mirror of the official ``REFERENCE_orbit_wars.py::generate_planets`` /
``generate_comet_paths`` (bit-exact-verified by ``scripts/test_official_comets.py``). World dicts
hold ``planets`` rows ``[id, owner, x, y, radius, ships, production]``, ``angular_velocity``,
``home_base`` and (when ``cfg.COMET_OFFICIAL``) padded comet-path arrays.

In [ ]:
def _dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def generate_planets(rng):
    '''Faithful mirror of REFERENCE_orbit_wars.py::generate_planets.
    Returns rows [id, owner, x, y, radius, ships, production].'''
    planets = []
    num_q1 = rng.randint(MIN_PLANET_GROUPS, MAX_PLANET_GROUPS)
    idc = 0
    # Phase 1: guaranteed static groups (polar sampling).
    static_groups = 0
    for _ in range(5000):
        if static_groups >= MIN_STATIC_GROUPS:
            break
        prod = rng.randint(1, 5)
        r = 1 + math.log(prod)
        angle = rng.uniform(0, math.pi / 2)
        min_orbital = ROTATION_RADIUS_LIMIT - r
        max_orbital = (BOARD_SIZE - CENTER - r) / max(math.cos(angle), math.sin(angle))
        if min_orbital > max_orbital:
            continue
        orbital_r = rng.uniform(min_orbital, max_orbital)
        x = CENTER + orbital_r * math.cos(angle)
        y = CENTER + orbital_r * math.sin(angle)
        if x + r > BOARD_SIZE or x - r < 0 or y + r > BOARD_SIZE or y - r < 0:
            continue
        if (BOARD_SIZE - x) - r < 0 or (BOARD_SIZE - y) - r < 0:
            continue
        if (x - CENTER) < r + 5 or (y - CENTER) < r + 5:
            continue
        ships = min(rng.randint(5, 99), rng.randint(5, 99))
        # NOTE: the reference stores rows as [id, owner, y, x, r, ...] (x/y swapped naming),
        # which is just a relabel of the symmetric copies; we keep it identical.
        tps = [
            [idc, -1, y, x, r, ships, prod],
            [idc + 1, -1, BOARD_SIZE - x, y, r, ships, prod],
            [idc + 2, -1, x, BOARD_SIZE - y, r, ships, prod],
            [idc + 3, -1, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod],
        ]
        valid = True
        for tp in tps:
            for p in planets:
                if _dist((p[2], p[3]), (tp[2], tp[3])) < p[4] + tp[4] + PLANET_CLEARANCE:
                    valid = False; break
            if not valid:
                break
        if valid:
            planets.extend(tps); idc += 4; static_groups += 1
    # Phase 2: fill remaining groups (normal random loop).
    attempts = 0
    max_attempts = 5000
    has_orbiting = False
    while len(planets) < num_q1 * 4 or (not has_orbiting and attempts < max_attempts):
        attempts += 1
        if attempts >= max_attempts:
            break
        prod = rng.randint(1, 5)
        r = 1 + math.log(prod)
        x = rng.uniform(CENTER + 15, BOARD_SIZE - r - 5)
        y = rng.uniform(CENTER + 15, BOARD_SIZE - r - 5)
        orbital_radius = _dist((x, y), (CENTER, CENTER))
        if orbital_radius < SUN_RADIUS + r + 10:
            continue
        if orbital_radius + r >= ROTATION_RADIUS_LIMIT:
            if x + r > BOARD_SIZE or x - r < 0 or y + r > BOARD_SIZE or y - r < 0:
                continue
        valid = True
        ships = rng.randint(5, 30)
        tps = [
            [idc, -1, y, x, r, ships, prod],
            [idc + 1, -1, BOARD_SIZE - x, y, r, ships, prod],
            [idc + 2, -1, x, BOARD_SIZE - y, r, ships, prod],
            [idc + 3, -1, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod],
        ]
        for tp in tps:
            tp_orb = _dist((tp[2], tp[3]), (CENTER, CENTER))
            tp_rot = tp_orb + tp[4] < ROTATION_RADIUS_LIMIT
            for p in planets:
                p_orb = _dist((p[2], p[3]), (CENTER, CENTER))
                p_rot = p_orb + p[4] < ROTATION_RADIUS_LIMIT
                if _dist((p[2], p[3]), (tp[2], tp[3])) < p[4] + tp[4] + PLANET_CLEARANCE:
                    valid = False; break
                if tp_rot != p_rot:
                    if abs(tp_orb - p_orb) < tp[4] + p[4] + PLANET_CLEARANCE:
                        valid = False; break
            if not valid:
                break
        if valid:
            if orbital_radius + r < ROTATION_RADIUS_LIMIT:
                has_orbiting = True
            planets.extend(tps); idc += 4
    return planets


def generate_world(cfg, seed):
    '''Mirror of native_worldgen.generate_world (comets dropped unless cfg.COMET_OFFICIAL).'''
    rng = random.Random(seed)
    angular_velocity = rng.uniform(0.025, 0.05)
    planets = generate_planets(rng)
    num_groups = len(planets) // 4
    base = -1
    if num_groups > 0:
        base = rng.randint(0, num_groups - 1) * 4
        planets[base][1] = 0;      planets[base][5] = 10       # player 0 home
        planets[base + 3][1] = 1;  planets[base + 3][5] = 10   # player 1 home
    # v8: remember the home group; env.reset(n_players=4) re-seats owners 0..3 on base..base+3
    # exactly like the official engine's 4-player branch (same group => fair under 4-fold symmetry).
    w = {"planets": planets, "angular_velocity": angular_velocity, "home_base": base}
    if cfg.COMET_OFFICIAL:
        attach_official_comets(cfg, w, seed)
    return w


# ---- OFFICIAL comet paths (numpy-vectorized port of REFERENCE_orbit_wars.generate_comet_paths;
# ---- same math/draw-order/reject-semantics, bit-exact-verified by scripts/test_official_comets.py)
def _comet_paths_official(initial_planets, angular_velocity, spawn_step, comet_speed, rng):
    """Returns 4 symmetric waypoint paths (list of [x,y], one waypoint per tick) or None."""
    stat, orb = [], []
    for p in initial_planets:
        pr = math.sqrt((p[2] - CENTER) ** 2 + (p[3] - CENTER) ** 2)
        (orb if pr + p[4] < ROTATION_RADIUS_LIMIT else stat).append(p)
    stat_xy = np.array([[p[2], p[3]] for p in stat], np.float64).reshape(-1, 2)
    stat_rad = np.array([p[4] for p in stat], np.float64)
    orb_r = np.array([math.sqrt((p[2] - CENTER) ** 2 + (p[3] - CENTER) ** 2) for p in orb], np.float64)
    orb_a0 = np.array([math.atan2(p[3] - CENTER, p[2] - CENTER) for p in orb], np.float64)
    orb_rad = np.array([p[4] for p in orb], np.float64)
    num = 5000
    t_arr = 0.3 * math.pi + 1.4 * math.pi * np.arange(num) / (num - 1)
    cos_t, sin_t = np.cos(t_arr), np.sin(t_arr)
    for _ in range(300):
        e = rng.uniform(0.75, 0.93)
        a = rng.uniform(60, 150)
        if a * (1 - e) < SUN_RADIUS + COMET_RADIUS:
            continue
        b = a * math.sqrt(1 - e ** 2)
        c_val = a * e
        phi = rng.uniform(math.pi / 6, math.pi / 3)
        ex = c_val + a * cos_t; ey = b * sin_t
        cp, sp = math.cos(phi), math.sin(phi)
        x = CENTER + ex * cp - ey * sp
        y = CENTER + ex * sp + ey * cp
        dx = x[1:] - x[:-1]; dy = y[1:] - y[:-1]
        cum = np.cumsum(np.sqrt(dx * dx + dy * dy))            # cum[i-1] = arc length at dense[i]
        nk = int(cum[-1] // comet_speed) + 1
        sel = np.searchsorted(cum, comet_speed * np.arange(1, nk + 1), side="left") + 1
        sel = sel[sel < num]
        px = np.concatenate(([x[0]], x[sel])); py = np.concatenate(([y[0]], y[sel]))
        on = (px >= 0) & (px <= BOARD_SIZE) & (py >= 0) & (py <= BOARD_SIZE)
        if not on.any():
            continue
        i0 = int(np.argmax(on)); i1 = int(len(on) - 1 - np.argmax(on[::-1]))
        vx = px[i0:i1 + 1]; vy = py[i0:i1 + 1]                  # contiguous on-board SPAN (incl. interior)
        K = len(vx)
        if not (5 <= K <= 40):
            continue
        if (np.sqrt((vx - CENTER) ** 2 + (vy - CENTER) ** 2) < SUN_RADIUS + COMET_RADIUS).any():
            continue
        sym_x = np.stack([vy, BOARD_SIZE - vx, vx, BOARD_SIZE - vy])          # (4,K)
        sym_y = np.stack([vx, vy, BOARD_SIZE - vy, BOARD_SIZE - vx])
        if len(stat_xy):
            d = np.sqrt((sym_x[:, :, None] - stat_xy[None, None, :, 0]) ** 2
                        + (sym_y[:, :, None] - stat_xy[None, None, :, 1]) ** 2)
            if (d < stat_rad[None, None, :] + (COMET_RADIUS + 0.5)).any():
                continue
        if len(orb_r):
            gs = spawn_step - 1 + np.arange(K)                                # (K,)
            ang = orb_a0[:, None] + angular_velocity * gs[None, :]            # (P,K)
            ox = CENTER + orb_r[:, None] * np.cos(ang)
            oy = CENTER + orb_r[:, None] * np.sin(ang)
            d = np.sqrt((sym_x[:, :, None] - ox.T[None, :, :]) ** 2
                        + (sym_y[:, :, None] - oy.T[None, :, :]) ** 2)        # (4,K,P)
            if (d < orb_rad[None, None, :] + COMET_RADIUS).any():
                continue
        return [
            [[float(vy[k]), float(vx[k])] for k in range(K)],
            [[float(BOARD_SIZE - vx[k]), float(vy[k])] for k in range(K)],
            [[float(vx[k]), float(BOARD_SIZE - vy[k])] for k in range(K)],
            [[float(BOARD_SIZE - vy[k]), float(BOARD_SIZE - vx[k])] for k in range(K)],
        ]
    return None


def attach_official_comets(cfg, world, seed):
    """Precompute the official comet schedule for one world (CPU). Stores padded arrays:
    comet_paths (NS,4,COMET_MAX_LEN,2) f32 / comet_len (NS,) / comet_ships (NS,) where
    NS = len(COMET_SPAWN_STEPS). Draw order matches the engine exactly (paths, then ships)."""
    NS = len(cfg.COMET_SPAWN_STEPS)
    cp = np.zeros((NS, 4, cfg.COMET_MAX_LEN, 2), np.float32)
    cl = np.zeros((NS,), np.int64)
    cs = np.zeros((NS,), np.float32)
    for e, s in enumerate(cfg.COMET_SPAWN_STEPS):
        rng = random.Random(f"orbit_wars-comet-{seed}-{s}")
        paths = _comet_paths_official(world["planets"], world["angular_velocity"], s, cfg.COMET_SPEED, rng)
        if not paths:
            continue
        ships = min(rng.randint(1, 99), rng.randint(1, 99), rng.randint(1, 99), rng.randint(1, 99))
        L = len(paths[0])
        for m in range(4):
            for k, (x, y) in enumerate(paths[m][:cfg.COMET_MAX_LEN]):
                cp[e, m, k, 0] = x; cp[e, m, k, 1] = y
        cl[e] = min(L, cfg.COMET_MAX_LEN)
        cs[e] = float(ships)
    world["comet_paths"] = cp; world["comet_len"] = cl; world["comet_ships"] = cs
    return world


def make_world_pool(cfg, n, base_seed=0):
    return [generate_world(cfg, base_seed + i) for i in range(n)]


# alias matching the plan's public-API name
build_world_pool = make_world_pool

## `orbit_wars_v12.policy`

v5 target actor-critic policy + the act/evaluate wrappers (notebook cell 21).

Per-planet trunk (legacy attn+ResNet-MLP, or a stacked transformer) -> WHERE/GATE heads producing
the gated-allocation action, plus a masked-mean-pooled value head with PopArt. The runtime knobs
that were notebook globals (``DEST_HEAD``/``INIT_DEST_SCALE``/``GRAD_CHECKPOINT`` for the net;
``WHERE_DIST``/``REACH_MASK``/``ALLOC_KAPPA``/``USE_AMP``/``USE_POPART`` for the wrappers) are read
off the :class:`~orbit_wars_v12.config.Config` passed in.

In [ ]:
def _amp_ctx(cfg):
    if cfg.USE_AMP and cfg.device.type == 'cuda':
        return torch.autocast('cuda', dtype=cfg.AMP_DTYPE)
    return contextlib.nullcontext()


class PopArt:
    '''Output-preserving running normalization of value targets (Hessel et al. 2018). The value head
    predicts NORMALIZED values; raw V = denormalize(y); on each stats update val_out is rescaled so
    the raw predictions are preserved.'''
    def __init__(self, beta):
        self.mu = 0.0; self.nu = 1.0; self.sigma = 1.0; self.beta = beta; self.init = False

    def normalize(self, y):
        return (y - self.mu) / self.sigma

    def denormalize(self, yn):
        return yn * self.sigma + self.mu

    def update(self, targets, val_out):
        with torch.no_grad():
            old_mu, old_sigma = self.mu, self.sigma
            m = targets.mean().item(); v = (targets * targets).mean().item()
            if not self.init:
                self.mu, self.nu, self.init = m, v, True
            else:
                b = self.beta
                self.mu = (1 - b) * self.mu + b * m
                self.nu = (1 - b) * self.nu + b * v
            self.sigma = max((self.nu - self.mu * self.mu) ** 0.5, 1e-4)
            if old_sigma > 0:                       # preserve raw outputs of the value head
                val_out.weight.mul_(old_sigma / self.sigma)
                val_out.bias.copy_((old_sigma * val_out.bias + old_mu - self.mu) / self.sigma)


def _resolve_act(name):
    """Map an ACT_FN name to a pointwise activation fn (default relu = the historical block act)."""
    return {"relu": torch.relu, "tanh": torch.tanh, "gelu": F.gelu, "silu": F.silu}.get(name, torch.relu)


class TxEncoderLayer(nn.Module):
    '''Pre-LN transformer encoder block: x = x + MHA(LN(x)); x = x + MLP(LN(x)).
    Multi-head; masks dead-planet KEYS (matches the legacy single-head attention).'''
    def __init__(self, h, n_heads, mlp_ratio, act='relu'):
        super().__init__()
        assert h % n_heads == 0, "HIDDEN (%d) not divisible by N_HEADS (%d)" % (h, n_heads)
        self.h, self.nh, self.hd = h, n_heads, h // n_heads
        self.act = _resolve_act(act)
        self.ln1 = nn.LayerNorm(h)
        self.q = nn.Linear(h, h); self.k = nn.Linear(h, h); self.v = nn.Linear(h, h); self.o = nn.Linear(h, h)
        self.ln2 = nn.LayerNorm(h)
        self.mlp_in = nn.Linear(h, mlp_ratio * h); self.mlp_out = nn.Linear(mlp_ratio * h, h)

    def forward(self, x, entity_mask):
        B, E, _ = x.shape
        xn = self.ln1(x)
        q = self.q(xn).view(B, E, self.nh, self.hd).transpose(1, 2)        # (B,nh,E,hd)
        k = self.k(xn).view(B, E, self.nh, self.hd).transpose(1, 2)
        v = self.v(xn).view(B, E, self.nh, self.hd).transpose(1, 2)
        attn_mask = (entity_mask < 0.5).view(B, 1, 1, E).to(q.dtype) * (-1e9)   # additive key mask
        ctx = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)      # flash/mem-efficient (B,nh,E,hd)
        ctx = ctx.transpose(1, 2).reshape(B, E, self.h)
        x = x + self.o(ctx)
        x = x + self.mlp_out(self.act(self.mlp_in(self.ln2(x))))
        return x


class TransformerTrunk(nn.Module):
    '''proj -> n_stem pre-LN ResNet-MLP -> n_layers transformer blocks -> n_head pre-LN
    ResNet-MLP -> LN. Emits per-planet features (B,E,h); drop-in for the legacy trunk.'''
    def __init__(self, F, h, n_heads, n_layers, mlp_ratio, n_stem, n_head, act='relu'):
        super().__init__()
        self.act = _resolve_act(act)
        self.proj = nn.Linear(F, h)
        self.stem_a = nn.ModuleList([nn.Linear(h, h) for _ in range(n_stem)])
        self.stem_b = nn.ModuleList([nn.Linear(h, h) for _ in range(n_stem)])
        self.stem_ln = nn.ModuleList([nn.LayerNorm(h) for _ in range(n_stem)])
        self.layers = nn.ModuleList([TxEncoderLayer(h, n_heads, mlp_ratio, act) for _ in range(n_layers)])
        self.head_a = nn.ModuleList([nn.Linear(h, h) for _ in range(n_head)])
        self.head_b = nn.ModuleList([nn.Linear(h, h) for _ in range(n_head)])
        self.head_ln = nn.ModuleList([nn.LayerNorm(h) for _ in range(n_head)])
        self.ln_out = nn.LayerNorm(h)

    def forward(self, entities, entity_mask):
        tok = self.proj(entities)
        for i in range(len(self.stem_a)):
            tok = tok + self.stem_b[i](self.act(self.stem_a[i](self.stem_ln[i](tok))))
        for layer in self.layers:
            tok = layer(tok, entity_mask)
        for i in range(len(self.head_a)):
            tok = tok + self.head_b[i](self.act(self.head_a[i](self.head_ln[i](tok))))
        return self.ln_out(tok)


def parse_trunk_spec(spec):
    """'res16,attn1,res64' -> [('res',16),('attn',1),('res',64)]. Block kinds: 'res' (pre-LN
    ResNet-MLP) and 'attn' (PURE cross-planet multi-head self-attention, no MLP)."""
    out = []
    for tok in str(spec).replace(" ", "").split(","):
        if not tok:
            continue
        i = 0
        while i < len(tok) and tok[i].isalpha():
            i += 1
        kind, n = tok[:i], int(tok[i:])
        assert kind in ("res", "attn", "seq"), "unknown block %r in TRUNK_SPEC (use res/attn/seq)" % kind
        out.append((kind, n))
    return out


class ResMLPBlock(nn.Module):
    '''One pre-LN ResNet-MLP block: x = x + W2(act(W1(LN(x)))). entity_mask arg ignored (uniform call).'''
    def __init__(self, h, act='relu'):
        super().__init__()
        self.ln = nn.LayerNorm(h); self.a = nn.Linear(h, h); self.b = nn.Linear(h, h)
        self.act = _resolve_act(act)

    def forward(self, x, entity_mask=None):
        return x + self.b(self.act(self.a(self.ln(x))))


class CrossAttnBlock(nn.Module):
    '''PURE cross-planet multi-head self-attention (pre-LN, residual, NO MLP). Masks dead-planet keys.'''
    def __init__(self, h, n_heads):
        super().__init__()
        assert h % n_heads == 0, "HIDDEN (%d) not divisible by N_HEADS (%d)" % (h, n_heads)
        self.h, self.nh, self.hd = h, n_heads, h // n_heads
        self.ln = nn.LayerNorm(h)
        self.q = nn.Linear(h, h); self.k = nn.Linear(h, h); self.v = nn.Linear(h, h); self.o = nn.Linear(h, h)

    def forward(self, x, entity_mask):
        B, E, _ = x.shape
        xn = self.ln(x)
        q = self.q(xn).view(B, E, self.nh, self.hd).transpose(1, 2)        # (B,nh,E,hd)
        k = self.k(xn).view(B, E, self.nh, self.hd).transpose(1, 2)
        v = self.v(xn).view(B, E, self.nh, self.hd).transpose(1, 2)
        am = (entity_mask < 0.5).view(B, 1, 1, E).to(q.dtype) * (-1e9)     # additive key mask
        ctx = F.scaled_dot_product_attention(q, k, v, attn_mask=am)
        ctx = ctx.transpose(1, 2).reshape(B, E, self.h)
        return x + self.o(ctx)


class BlockSeqTrunk(nn.Module):
    '''proj -> an arbitrary ordered SEQUENCE of blocks (parsed from TRUNK_SPEC) -> LN. Block kinds:
    'res' = pre-LN ResNet-MLP; 'attn' = PURE cross-planet MHA (no MLP); 'seq' = full transformer
    encoder layer (MHA + MLP, TX_MLP_RATIO). Emits per-planet features (B,E,h); drop-in trunk.'''
    def __init__(self, F, h, spec, n_heads, act='relu', grad_checkpoint=False, mlp_ratio=4):
        super().__init__()
        self.proj = nn.Linear(F, h)
        self.grad_checkpoint = bool(grad_checkpoint)
        blocks = []
        for kind, count in spec:
            for _ in range(count):
                if kind == 'res':
                    blocks.append(ResMLPBlock(h, act))
                elif kind == 'attn':
                    blocks.append(CrossAttnBlock(h, n_heads))
                else:                                   # 'seq' = transformer encoder layer (MHA + MLP)
                    blocks.append(TxEncoderLayer(h, n_heads, mlp_ratio, act))
        self.blocks = nn.ModuleList(blocks)
        self.ln_out = nn.LayerNorm(h)

    def forward(self, entities, entity_mask):
        tok = self.proj(entities)
        for blk in self.blocks:
            if self.grad_checkpoint and tok.requires_grad:
                tok = _ckpt.checkpoint(blk, tok, entity_mask, use_reentrant=False)
            else:
                tok = blk(tok, entity_mask)
        return self.ln_out(tok)


class PolicyNet(nn.Module):
    '''v5 TARGET actor-critic (mirrors model/policy_net.cpp, target_actor branch).

    Trunk (UNCHANGED from the continuous actor): per-planet proj + masked cross-planet
    self-attention + GLU + n residual blocks + a separate board-globals embedding.
    Heads: dest_head -> (B,E,E) per-source WHERE allocation logits (-> Dirichlet rows over the
    E=PLANET_CAP dests; dest_head out dim == max_entities); gate_head -> (B,E,1) per-source launch
    gate logit. See GatedAllocDist for how WHERE x GATE compose into the (B,E,E+1) action.

    `cfg` supplies DEST_HEAD / INIT_DEST_SCALE / GRAD_CHECKPOINT (read once here); the remaining
    dims are explicit so league snapshots / invited members can rebuild at their own sizes.
    '''
    def __init__(self, cfg, F, G, hidden, d_g, max_entities, n_res_blocks, use_glu, init_gate_bias,
                 use_attention=True,
                 value_res_blocks=2, arch="trunk", n_heads=8, n_tx_layers=4, tx_mlp_ratio=4,
                 n_stem_res=1, n_head_res=1, act='relu', aux_reward_pred=False, aux_win_bet=False,
                 trunk_spec=""):
        super().__init__()
        self.F, self.G, self.h, self.d_g, self.E = F, G, hidden, d_g, max_entities
        self.use_glu = use_glu
        self.use_attention = use_attention
        self.arch = arch
        self.grad_checkpoint = bool(cfg.GRAD_CHECKPOINT)   # read once (avoids cfg on the compiled module)
        self.act_fn = _resolve_act(act)   # ResNet-MLP block activation (g_embed below stays relu)
        h = hidden
        if arch == "transformer":
            self.trunk = TransformerTrunk(F, h, n_heads, n_tx_layers, tx_mlp_ratio, n_stem_res, n_head_res, act)
        elif arch == "blockseq":                            # ordered res/attn block sequence from TRUNK_SPEC
            self.trunk = BlockSeqTrunk(F, h, parse_trunk_spec(trunk_spec), n_heads, act,
                                       self.grad_checkpoint, tx_mlp_ratio)
        else:
            self.proj = nn.Linear(F, h)
            if use_attention:                               # cross-planet self-attention (toggleable)
                self.attn_q = nn.Linear(h, h); self.attn_k = nn.Linear(h, h)
                self.attn_v = nn.Linear(h, h); self.attn_o = nn.Linear(h, h)
                self.ln_attn = nn.LayerNorm(h)
            self.ln_out = nn.LayerNorm(h)
            if use_glu:
                self.glu_gate = nn.Linear(h, h); self.glu_val = nn.Linear(h, h); self.glu_out = nn.Linear(h, h)
            self.res_a = nn.ModuleList([nn.Linear(h, h) for _ in range(n_res_blocks)])
            self.res_b = nn.ModuleList([nn.Linear(h, h) for _ in range(n_res_blocks)])
            self.ln_res = nn.ModuleList([nn.LayerNorm(h) for _ in range(n_res_blocks)])
        self.g_embed = nn.Linear(G, d_g)
        # --- gated allocation heads: WHERE (per-source dest simplex) + GATE (per-source fire logit) ---
        self.dest_mode = cfg.DEST_HEAD
        if cfg.DEST_HEAD == 'bilinear':                    # pointer score: query=source, key=dest
            self.dest_q = nn.Linear(h + d_g, h); self.dest_k = nn.Linear(h + d_g, h)
        else:
            self.dest_head = nn.Linear(h + d_g, max_entities)  # (B,E,E) per-source WHERE allocation logits
        self.gate_head = nn.Linear(h + d_g, 1)             # (B,E,1) per-source launch-gate logit
        # value head (PPO critic): input proj -> pre-LN RESIDUAL MLP (same block as the trunk) -> readout
        self.val_in = nn.Linear(h + d_g, h)
        self.val_res_a = nn.ModuleList([nn.Linear(h, h) for _ in range(value_res_blocks)])
        self.val_res_b = nn.ModuleList([nn.Linear(h, h) for _ in range(value_res_blocks)])
        self.val_ln = nn.ModuleList([nn.LayerNorm(h) for _ in range(value_res_blocks)])
        self.val_out = nn.Linear(h, 1)
        # auxiliary reward-prediction head (UNREAL): a SHALLOW head off the same pooled trunk features
        # regresses the immediate per-step reward r_t. Deliberately shallow -- the point is to pressure
        # the TRUNK, not to fit r_t with a deep head. Built only when enabled, so the state_dict (and every
        # existing checkpoint) is unchanged when off; this flag is persisted in the ckpt config blob.
        self.aux_reward_pred = bool(aux_reward_pred)
        self.aux_win_bet = bool(aux_win_bet)
        if self.aux_reward_pred or self.aux_win_bet:      # ONE shallow scalar head, two possible objectives:
            self.rew_in = nn.Linear(h + d_g, h)           #   reward-pred (raw -> MSE to r_t) OR
            self.rew_out = nn.Linear(h, 1)                #   win-bet (raw logit -> bet=tanh, maximize bet*outcome)
        # calm init: few planets fire at start (negative gate bias); WHERE small-init -> ~uniform
        with torch.no_grad():
            self.gate_head.bias.fill_(init_gate_bias)
            if cfg.DEST_HEAD == 'bilinear':
                self.dest_q.weight.mul_(cfg.INIT_DEST_SCALE); self.dest_q.bias.zero_()
                self.dest_k.weight.mul_(cfg.INIT_DEST_SCALE); self.dest_k.bias.zero_()
            else:
                self.dest_head.weight.mul_(cfg.INIT_DEST_SCALE); self.dest_head.bias.zero_()

    def _res_block(self, i, tok):                  # one pre-LN residual MLP block (trunk)
        return tok + self.res_b[i](self.act_fn(self.res_a[i](self.ln_res[i](tok))))

    def forward(self, entities, entity_mask, action_mask, globals_):
        if self.arch in ("transformer", "blockseq"):
            tok = self.trunk(entities, entity_mask)
        else:
            tok = self.proj(entities)                       # (B,E,d)
            # masked single-head self-attention over planets (pre-norm) -- the only cross-planet mixing
            if self.use_attention:
                xn = self.ln_attn(tok)
                Q = self.attn_q(xn); Kt = self.attn_k(xn); Vt = self.attn_v(xn)
                scores = torch.matmul(Q, Kt.transpose(1, 2)) / math.sqrt(self.h)   # (B,E,E)
                dead_key = (entity_mask < 0.5).unsqueeze(1)     # (B,1,E)
                scores = scores.masked_fill(dead_key, -1e9)
                attn = torch.matmul(torch.softmax(scores, -1), Vt)
                tok = tok + self.attn_o(attn)
            if self.use_glu:
                tok = tok + self.glu_out(self.glu_val(tok) * torch.sigmoid(self.glu_gate(tok)))
            for i in range(len(self.res_a)):
                if self.grad_checkpoint and tok.requires_grad:   # recompute in backward -> deep stack fits VRAM
                    tok = _ckpt.checkpoint(self._res_block, i, tok, use_reentrant=False)
                else:                                       # rollout / eval (no_grad): identical to before
                    tok = self._res_block(i, tok)
            tok = self.ln_out(tok)
        B, E = tok.shape[0], tok.shape[1]
        gp = torch.relu(self.g_embed(globals_))         # (B,d_g)
        gpb = gp.unsqueeze(1).expand(B, E, self.d_g)
        hh = torch.cat([tok, gpb], -1)                  # (B,E,d+d_g)
        # --- v5 target heads ---
        if self.dest_mode == 'bilinear':               # pointer score q.k / sqrt(h)
            qd = self.dest_q(hh); kd = self.dest_k(hh)
            dest_logits = torch.einsum('bsh,bdh->bsd', qd, kd) / math.sqrt(self.h)
        else:
            dest_logits = self.dest_head(hh)           # (B,E,E) per-source WHERE allocation logits
        gate_logits = self.gate_head(hh).squeeze(-1)   # (B,E) per-source launch-gate logit
        # value: masked-mean pool of trunk tokens over live planets + g'
        em = entity_mask.unsqueeze(-1)
        pooled = (tok * em).sum(1) / em.sum(1).clamp_min(1.0)
        vh = self.val_in(torch.cat([pooled, gp], -1))                  # (B,d)
        for i in range(len(self.val_res_a)):                           # residual blocks (skip per block)
            vh = vh + self.val_res_b[i](self.act_fn(self.val_res_a[i](self.val_ln[i](vh))))
        value = self.val_out(vh).squeeze(-1)                           # (B,)
        reward_pred = None                                             # aux scalar head off the SAME pooled features:
        if self.aux_reward_pred or self.aux_win_bet:                   # raw output -> MSE to r_t (reward-pred), or the
            reward_pred = self.rew_out(self.act_fn(self.rew_in(torch.cat([pooled, gp], -1)))).squeeze(-1)  # bet logit
        return dest_logits, gate_logits, value, reward_pred            # (win-bet: bet = tanh(reward_pred) in the loss)


def build_policy(cfg):
    _net = PolicyNet(cfg, F_DIM, G_DIM, cfg.HIDDEN, cfg.D_G, PLANET_CAP, cfg.N_RES_BLOCKS,
                     cfg.USE_GLU, cfg.INIT_GATE_BIAS, use_attention=cfg.USE_ATTENTION, act=cfg.ACT_FN,
                     value_res_blocks=cfg.VALUE_RES_BLOCKS,
                     arch=cfg.ARCH, n_heads=cfg.N_HEADS, n_tx_layers=cfg.N_TX_LAYERS,
                     tx_mlp_ratio=cfg.TX_MLP_RATIO, n_stem_res=cfg.N_STEM_RES,
                     n_head_res=cfg.N_HEAD_RES, aux_reward_pred=cfg.AUX_REWARD_PRED,
                     aux_win_bet=cfg.AUX_WIN_BET, trunk_spec=getattr(cfg, "TRUNK_SPEC", "")).to(cfg.device)
    _net.popart = PopArt(cfg.POPART_BETA)
    return _net


def compute_reach(ent, em):
    """Per-source destination reachability (B,E,E): 1 unless a straight src->dest shot is absorbed
    by the sun. Recovered from encoded planet positions so it is identical in rollout and update.
    Diagonal forced to 1; the alive-mask is applied in the distribution."""
    px = ent[..., 0] * BOARD_SIZE
    py = ent[..., 1] * BOARD_SIZE
    B, E = px.shape
    sx = px.unsqueeze(2); sy = py.unsqueeze(2)         # source (B,E,1)
    dx = px.unsqueeze(1) - sx                           # src->dest (B,E,E)
    dy = py.unsqueeze(1) - sy
    dist = torch.sqrt(dx * dx + dy * dy).clamp_min(1e-6)
    ux = dx / dist; uy = dy / dist
    cx = CENTER - sx; cy = CENTER - sy                  # source -> sun centre
    t = cx * ux + cy * uy                               # closest-approach distance along the ray
    perp2 = (cx * cx + cy * cy) - t * t
    sr = SUN_RADIUS * SUN_RADIUS
    t_sun = t - torch.sqrt((sr - perp2).clamp_min(0.0))
    blocked = (t >= 0.0) & (perp2 <= sr) & (t_sun >= 0.0) & (t_sun < dist)
    reach = (~blocked).to(DTYPE)
    eye = torch.eye(E, device=ent.device, dtype=DTYPE).unsqueeze(0)
    return reach * (1.0 - eye) + eye                    # source -> itself is always reachable


def recover_ships(ent):
    """Integer ship count per planet, read directly from feature b6 (now the RAW garrison)."""
    return torch.round(ent[..., 6].clamp_min(0.0))


def _make_dist(cfg, net, ent, em, am, gl):
    '''Build the gated-allocation actor distribution. owned = legal source planets (= action_mask),
    alive = valid destination slots (= entity_mask), reach = sun-reachability mask.'''
    reach = compute_reach(ent, em) if cfg.REACH_MASK else None
    with _amp_ctx(cfg):
        dest_logits, gate_logits, value, reward_pred = net(ent, em, am, gl)
    if cfg.WHERE_DIST == 'categorical':
        dist = GatedCatDist(cfg, dest_logits.float(), gate_logits.float(), owned=am, alive=em, reach=reach)
    else:
        dist = GatedAllocDist(cfg, dest_logits.float(), gate_logits.float(), owned=am, alive=em,
                              reach=reach, kappa=cfg.ALLOC_KAPPA)
    return dist, value.float(), (reward_pred.float() if reward_pred is not None else None)


@torch.no_grad()
def act(cfg, net, ent, em, am, gl, greedy=False, crn_group=0, want_rpred=False):
    dist, value, rpred = _make_dist(cfg, net, ent, em, am, gl)   # rpred (bet logit) unused unless want_rpred
    # crn_group>1 couples the sampling noise across each contiguous block of crn_group envs (GRPO
    # common-random-numbers, docs/grpo_v12.tex [D]); 0 = independent draws (the default everywhere else).
    action = dist.greedy() if greedy else dist.sample(crn_group=crn_group)
    if cfg.USE_POPART and getattr(net, 'popart', None) is not None:
        value = net.popart.denormalize(value)
    if want_rpred:   # AUX_WIN_BET_REWARD path: also surface the per-step bet logit (-> one-sided win reward in rollout)
        return action, dist.log_prob(action), value, rpred
    return action, dist.log_prob(action), value


def evaluate(cfg, net, ent, em, am, gl, action):
    dist, value, reward_pred = _make_dist(cfg, net, ent, em, am, gl)
    return dist.log_prob(action), dist.entropy(), value, reward_pred

## `orbit_wars_v12.env`

GPU-batched Orbit Wars simulator (notebook cells 13/15/17/19 + settle/settle_n from 23).

Bit-exact to the official engine including official comets (see memory `gpuenv-kaggle-parity`).
All state lives as ``(B, *)`` tensors on ``cfg.device``; one :meth:`GpuEnv.step` advances every
parallel env one tick. The free functions ``env_encode`` / ``env_step`` / ``opponent_action`` etc.
operate on a :class:`GpuEnv`; functions that read run-tunable hyperparameters take ``cfg`` (the
pure geometry/feature helpers do not -- they read only :mod:`orbit_wars_v12.constants`).

In [ ]:
# opponent_action codes by league "kind" (the two notebook names are the same mapping).
_OPP_BY_KIND = {"random": 0, "starter": 1, "noop": 2, "medium": 3, "greedy": 4, "intermediate": 5}
_KIND2OPP = _OPP_BY_KIND

# seat-swap board rotations (k*90deg) mapping ego -> each seat; see _apply_rot_aug.
_SEAT_ROT = {2: (0, 2), 4: (0, 1, 2, 3)}
ROTATE_AUGMENT = True   # legacy per-env RANDOM rotation fallback (used if _rot_aug set w/o _rot_k)


class GpuEnv:
    def __init__(self, cfg, planet_cap=None, fleet_cap=None, episode_steps=None, ship_speed=None):
        self.cfg = cfg
        self.Ec = PLANET_CAP if planet_cap is None else planet_cap
        self.Fc = cfg.FLEET_CAP if fleet_cap is None else fleet_cap
        self.T = cfg.EPISODE_STEPS if episode_steps is None else episode_steps
        self.vmax = SHIP_SPEED if ship_speed is None else ship_speed
        self.dev = cfg.device
        self.B = 0
        self.n_players = 2

    def reset(self, worlds, n_players=2):
        cfg = self.cfg
        self.n_players = int(n_players)
        B = len(worlds)
        self.B = B
        Ec, Fc = self.Ec, self.Fc
        dev = self.dev
        z = lambda *s: torch.zeros(s, dtype=DTYPE, device=dev)
        # planets (B, Ec)
        self.p_alive = z(B, Ec)
        self.p_owner = torch.full((B, Ec), -1.0, dtype=DTYPE, device=dev)
        self.p_x = z(B, Ec); self.p_y = z(B, Ec); self.p_radius = z(B, Ec)
        self.p_ships = z(B, Ec); self.p_prod = z(B, Ec); self.p_is_comet = z(B, Ec)
        self.p_init_x = z(B, Ec); self.p_init_y = z(B, Ec); self.p_rotates = z(B, Ec)
        self.p_comet_vx = z(B, Ec); self.p_comet_vy = z(B, Ec)   # comet straight-line velocity
        # fill from worlds (CPU numpy then upload once)
        pa = np.zeros((B, Ec), np.float32); po = np.full((B, Ec), -1.0, np.float32)
        px = np.zeros((B, Ec), np.float32); py = np.zeros((B, Ec), np.float32)
        pr = np.zeros((B, Ec), np.float32); ps = np.zeros((B, Ec), np.float32)
        pp = np.zeros((B, Ec), np.float32); pix = np.zeros((B, Ec), np.float32)
        piy = np.zeros((B, Ec), np.float32); prot = np.zeros((B, Ec), np.float32)
        angv = np.zeros((B,), np.float32)
        for b, w in enumerate(worlds):
            pls = w["planets"][:Ec]
            for i, pl in enumerate(pls):
                pid, owner, x, y, rad, sh, prod = pl
                pa[b, i] = 1.0; po[b, i] = float(owner)
                px[b, i] = x; py[b, i] = y; pr[b, i] = rad
                ps[b, i] = sh; pp[b, i] = prod
                pix[b, i] = x; piy[b, i] = y
                rr = math.hypot(x - CENTER, y - CENTER)
                prot[b, i] = 1.0 if (rr + rad < ROTATION_RADIUS_LIMIT) else 0.0
            angv[b] = w["angular_velocity"]
            if self.n_players >= 4:               # official 4p seating: owners 0..3, 10 ships each
                hb = int(w.get("home_base", -1))
                if 0 <= hb and hb + 3 < Ec:
                    for j in range(4):
                        po[b, hb + j] = float(j); ps[b, hb + j] = 10.0
        t = lambda a: torch.from_numpy(a).to(dev)
        self.p_alive = t(pa); self.p_owner = t(po); self.p_x = t(px); self.p_y = t(py)
        self.p_radius = t(pr); self.p_ships = t(ps); self.p_prod = t(pp)
        self.p_init_x = t(pix); self.p_init_y = t(piy); self.p_rotates = t(prot)
        self.p_is_comet = z(B, Ec)
        # fleets (B, Fc)
        self.f_alive = z(B, Fc); self.f_owner = z(B, Fc); self.f_x = z(B, Fc)
        self.f_y = z(B, Fc); self.f_angle = z(B, Fc); self.f_ships = z(B, Fc)
        self.f_seq = z(B, Fc)
        # official comet schedule (padded waypoint playback)
        NS = len(cfg.COMET_SPAWN_STEPS)
        if cfg.COMET_OFFICIAL and worlds and ('comet_paths' in worlds[0]):
            cp = np.stack([w['comet_paths'] for w in worlds])           # (B,NS,4,L,2)
            cl = np.stack([w['comet_len'] for w in worlds])             # (B,NS)
            cs = np.stack([w['comet_ships'] for w in worlds])           # (B,NS)
            self.c_paths = torch.from_numpy(cp).to(dev)
            self.c_len = torch.from_numpy(cl).to(dev)
            self.c_ships = torch.from_numpy(cs).to(dev)
            self.c_slot = torch.full((B, NS, 4), -1, dtype=torch.long, device=dev)
        else:
            self.c_paths = None
        # per-env
        self.ang_vel = t(angv)
        self.step_ct = torch.zeros(B, dtype=DTYPE, device=dev)
        self.done = torch.zeros(B, dtype=DTYPE, device=dev)
        if getattr(self, '_rot_aug', False):       # seat-swap aug (training env only); set by collect_ppo
            _apply_rot_aug(self)
        self.p_x_prev = self.p_x.clone(); self.p_y_prev = self.p_y.clone()    # 1-step prev pos -> finite-diff velocity
        self.p_x_prev2 = self.p_x.clone(); self.p_y_prev2 = self.p_y.clone()  # 2-step prev pos -> finite-diff curvature (v13)


# ---- seat-swap augmentation helpers (training env only) --------------------------------------
# collect_ppo tiles the worlds P-fold and stamps a per-env rotation index env._rot_k (seat-block
# b -> _SEAT_ROT[P][b]); GpuEnv.reset's hook (above) rotates ALL absolute-coord state by k*90deg
# about CENTER. The board is 4-fold rotationally symmetric, so this exactly moves the ego (pid 0)
# onto each seat's geometry -- the absolute-coord seat overfit is averaged out. eval/recal/gauntlet
# build their own canonical envs (no _rot_aug), so ratings stay comparable.
def _rot90_about_center(x, y, k):
    """Rotate (x,y) about CENTER by k*90deg (k int tensor broadcastable to x). +90: (x,y)->(-y,x)."""
    dx = x - CENTER; dy = y - CENTER
    nx = torch.where(k == 0, dx, torch.where(k == 1, -dy, torch.where(k == 2, -dx, dy)))
    ny = torch.where(k == 0, dy, torch.where(k == 1, dx, torch.where(k == 2, -dy, -dx)))
    return CENTER + nx, CENTER + ny


def _apply_rot_aug(env):
    """Rotate ALL absolute-coord state by env._rot_k (per-env k*90deg about CENTER): planet
    positions + orbit anchors + comet waypoints. radius/ownership/ang_vel are rotation-invariant;
    fleets are empty at reset. If _rot_k is unset, fall back to a per-env RANDOM k (legacy aug)."""
    B = env.B
    k = getattr(env, "_rot_k", None)
    if k is None:
        k = torch.randint(0, 4, (B, 1), device=env.dev)
    env.p_x, env.p_y = _rot90_about_center(env.p_x, env.p_y, k)
    env.p_init_x, env.p_init_y = _rot90_about_center(env.p_init_x, env.p_init_y, k)
    if getattr(env, "c_paths", None) is not None:          # comet waypoints (B,NS,4,L,2)
        kk = k.view(B, 1, 1, 1)
        rx, ry = _rot90_about_center(env.c_paths[..., 0], env.c_paths[..., 1], kk)
        env.c_paths = torch.stack([rx, ry], dim=-1)


def _fleet_target_core(fx, fy, fang, fships, vmax, p_x, p_y, p_radius, p_alive, p_x_prev, p_y_prev):
    '''Pure-tensor core of fleet_target_batch (NO env object) -> torch.compile-friendly. Identical
    math; the planet state + previous positions (for the finite-diff planet velocity) arrive as
    tensor args instead of being read off the env (which dynamo cannot trace through cleanly).'''
    spd = fleet_speed_t(fships, vmax)                            # (B,M) fleet speed
    fvx = (torch.cos(fang) * spd).unsqueeze(2)                   # (B,M,1) fleet velocity vector
    fvy = (torch.sin(fang) * spd).unsqueeze(2)
    pxr = p_x.unsqueeze(1); pyr = p_y.unsqueeze(1)               # (B,1,Ec) planet position
    pvx = (p_x - p_x_prev).unsqueeze(1); pvy = (p_y - p_y_prev).unsqueeze(1)   # (B,1,Ec) planet velocity (finite diff)
    rr = (p_radius * p_radius).unsqueeze(1)
    alive = (p_alive > 0.5).unsqueeze(1)
    rx = fx.unsqueeze(2) - pxr                  # (B,M,Ec) relative position (fleet - planet)
    ry = fy.unsqueeze(2) - pyr
    wx = fvx - pvx; wy = fvy - pvy              # (B,M,Ec) relative velocity (fleet - planet)
    a = wx * wx + wy * wy
    b = 2.0 * (rx * wx + ry * wy)
    c = rx * rx + ry * ry - rr
    disc = b * b - 4.0 * a * c
    sq = torch.sqrt(disc.clamp_min(0.0))
    t_hit = (-b - sq) / (2.0 * a).clamp_min(1e-9)               # first contact = smaller root (TICKS)
    t_hit = torch.where(c <= 0.0, torch.zeros_like(t_hit), t_hit)   # fleet already inside the disk -> now
    hit = (disc >= 0.0) & (t_hit >= 0.0) & alive & (a > 1e-9)
    tvals = torch.where(hit, t_hit, torch.full_like(t_hit, BIG))
    best_t, best_e = tvals.min(2)               # (B,M) earliest contact TIME + planet
    any_hit = best_t < BIG
    # sun occlusion: is the (static) sun reached BEFORE the target? compare in TICKS
    sx = fx - CENTER; sy = fy - CENTER
    dxx = torch.cos(fang); dyy = torch.sin(fang)
    tcs = -(sx * dxx + sy * dyy)
    sperp2 = sx * sx + sy * sy - tcs * tcs
    sr = SUN_RADIUS * SUN_RADIUS
    t_sun = (tcs - torch.sqrt((sr - sperp2).clamp_min(0.0))) / spd.clamp_min(1e-6)   # ticks to sun entry
    sun_block = (tcs >= 0.0) & (sperp2 <= sr) & (t_sun >= 0.0) & (t_sun < best_t)
    valid = any_hit & (~sun_block)
    tgt = torch.where(valid, best_e, torch.full_like(best_e, -1))             # SUN-AWARE (sun-blocked -> -1)
    eta = torch.where(valid, best_t, torch.zeros_like(best_t))                # best_t already in TICKS
    app_tgt = torch.where(any_hit, best_e, torch.full_like(best_e, -1))       # APPARENT target (ignores the sun)
    app_eta = torch.where(any_hit, best_t, torch.zeros_like(best_t))          # -> sun-bound fleets still attributed here
    return tgt, eta, app_tgt, app_eta


def fleet_target_batch(env, fx, fy, fang, fships, vmax):
    '''fx,fy,fang,fships: (B,M). Returns (tgt, eta, app_tgt, app_eta), each (B,M): tgt/eta = the
    SUN-AWARE target (a fleet whose path is absorbed by the sun -> -1); app_tgt/app_eta = the APPARENT
    target ignoring the sun (the planet the ray points at) -- the threat encoder uses the apparent
    target so a sun-bound fleet still SHOWS UP in the threat list of the planet it is aimed at.
    Thin env wrapper around _fleet_target_core (resolves the previous-position finite-diff fallback).'''
    p_x_prev = getattr(env, "p_x_prev", env.p_x); p_y_prev = getattr(env, "p_y_prev", env.p_y)
    return _fleet_target_core(fx, fy, fang, fships, vmax, env.p_x, env.p_y, env.p_radius, env.p_alive,
                              p_x_prev, p_y_prev)


def _encode_core(ego, na, T, vmax, p_alive, p_owner, p_x, p_y, p_radius, p_is_comet, ang_vel,
                 p_prod, p_ships, f_x, f_y, f_angle, f_ships, f_alive, f_owner, f_seq, step_ct,
                 p_x_prev, p_y_prev, p_x_prev2, p_y_prev2):
    """Pure-tensor core of env_encode (NO env object) -> torch.compile-friendly. ``ego``/``na``/``T``/
    ``vmax`` are python scalars (compile specializes one graph per distinct value -- a tiny, fixed set:
    ego in 0..3, na in {2,4}, T in {200,500}); everything else is a tensor. Identical math to the
    wrapper below; shapes + device are read off the tensors instead of the env object."""
    B, Ec = p_owner.shape
    Fc = f_owner.shape[1]
    dev = p_owner.device
    alive = p_alive > 0.5
    av = p_alive
    owner = p_owner
    dxc = p_x - CENTER; dyc = p_y - CENTER
    dist = torch.sqrt(dxc * dxc + dyc * dyc)
    comet = p_is_comet > 0.5
    rotating = (~comet) & ((dist + p_radius) < ROTATION_RADIUS_LIMIT)
    angv = ang_vel.unsqueeze(1)
    vmag = torch.where(rotating, angv.abs() * dist, torch.zeros_like(dist))
    cw = torch.where(rotating,
                     torch.where(angv >= 0.0, torch.ones_like(dist), -torch.ones_like(dist)),
                     torch.zeros_like(dist))
    # ego-relative seat ONE-HOT: rel = (owner - ego) mod n_players -> channel rel fires;
    # neutral = all-zero. Channel 0 = self; channels 1..3 = the other seats in seat order
    # (rotation-equivariant: the same relative seat always lands in the same channel).
    rel = owner - float(ego)
    rel = rel - na * torch.floor(rel / na)
    _owned = owner >= 0.0
    own_oh = [(_owned & (rel == float(k))).to(DTYPE) for k in range(4)]
    b0 = p_x / BOARD_SIZE
    b1 = p_y / BOARD_SIZE
    b2 = vmag / THREAT_MAX_SPEED
    b3 = cw
    b4 = p_radius / 3.0
    b5 = p_prod / 5.0
    b6 = p_ships  # RAW garrison (per-planet ship-log compression removed; un-normalized integer count)
    b7a, b7b, b7c, b7d = own_oh   # [self, enemy+1, enemy+2, enemy+3]; neutral all-zero
    b8 = dist / DIAG_HALF
    b9 = comet.to(DTYPE)
    actable = (owner == float(ego)) & alive & (p_ships > 0.0)
    b10 = actable.to(DTYPE)
    # v13 MOTION: per-planet velocity + curvature via BACKWARD finite-diff (the only definition
    # reproducible at serve time -- prod cannot see the next tick). Uniform over body types: static->0,
    # orbital->tangent/centripetal, comet->drift/bend (the comet case is the new signal; orbital was
    # already covered by b2/b3). Absolute board coords, so the seat-swap rot-aug rotates v/a with the
    # positions automatically (prev/prev2 are set post-rotation at reset). See docs/v13_motion.md.
    b14 = (p_x - p_x_prev) / V_SCALE                                  # vx
    b15 = (p_y - p_y_prev) / V_SCALE                                  # vy
    b16 = (p_x - 2.0 * p_x_prev + p_x_prev2) / A_SCALE                # ax (curvature)
    b17 = (p_y - 2.0 * p_y_prev + p_y_prev2) / A_SCALE                # ay (curvature)
    body = torch.stack([b0, b1, b2, b3, b4, b5, b6, b7a, b7b, b7c, b7d, b8, b9, b10,
                        b14, b15, b16, b17], 2)  # (B,Ec,18)

    # threat: per planet, the TOP N_THREAT_FLEETS inbound fleets BY SIZE. Attribution uses the APPARENT
    # target (the planet the ray points at, IGNORING the sun), so a fleet that will be absorbed by the sun
    # still SHOWS UP in its target's threat list (it is genuinely aimed here).
    _t, _e, app_tgt, app_eta = _fleet_target_core(f_x, f_y, f_angle, f_ships, vmax, p_x, p_y, p_radius,
                                                  p_alive, p_x_prev, p_y_prev)
    falive = f_alive > 0.5
    # fleet ownership = 4-channel EGO-RELATIVE seat ONE-HOT [self, enemy+1, +2, +3] (rotation-equivariant;
    # 2p only lights channels 0..1; empty/dead slots all-zero, masked by `valid` below).
    frel = f_owner - float(ego)
    frel = frel - na * torch.floor(frel / na)               # (B,Fc) ego-relative seat
    fraw = f_ships.abs()                                    # RAW inbound-fleet size (log compression is legacy)
    slot = torch.arange(Ec, dtype=DTYPE, device=dev).view(1, Ec, 1)
    tgtf = app_tgt.to(DTYPE).unsqueeze(1)                    # (B,1,Fc) APPARENT target (sun-bound fleets included)
    targeting = (tgtf == slot) & (app_tgt >= 0).unsqueeze(1) & falive.unsqueeze(1)  # (B,Ec,Fc)
    W = float(1 << 20)
    absf = f_ships.abs().unsqueeze(1)
    seqf = f_seq.unsqueeze(1)
    key_big = torch.where(targeting, absf * W - seqf, torch.full((1,), -BIG, device=dev))
    big_top_v, idx = torch.topk(key_big, N_THREAT_FLEETS, 2, largest=True)   # TOP 30 BY SIZE (seq tie-break)
    valid = big_top_v > -BIG * 0.5
    def pick(src):
        return src.unsqueeze(1).expand(B, Ec, Fc).gather(2, idx)
    zt = torch.zeros(B, Ec, N_THREAT_FLEETS, device=dev)
    own_ch = [torch.where(valid, pick((frel == float(k)).to(DTYPE)), zt) for k in range(4)]  # [self,e+1,e+2,e+3]
    etf = torch.where(valid, pick(app_eta) / THREAT_ETA_SCALE, zt)
    slg = torch.where(valid, pick(fraw), zt)
    threat = torch.stack(own_ch + [etf, slg], 3).reshape(B, Ec, N_THREAT_FEATS * N_THREAT_FLEETS)  # (B,Ec,180)

    entities = torch.cat([body, threat], 2) * av.unsqueeze(2)   # zero dead slots
    entity_mask = av
    action_mask = b10

    # globals (B,10)
    def psum(mask):
        return (p_ships * mask.to(DTYPE)).sum(1)
    mine = (owner == float(ego)) & alive
    en = (owner >= 0.0) & (owner != float(ego)) & alive    # ALL opponents pooled
    neu = (owner < 0.0) & alive
    my_ships = psum(mine); en_ships = psum(en)
    my_pl = mine.to(DTYPE).sum(1); en_pl = en.to(DTYPE).sum(1); neu_pl = neu.to(DTYPE).sum(1)
    npl = av.sum(1); total = npl.clamp_min(1.0)
    my_fleet = (f_ships * (f_owner == float(ego)).to(DTYPE) * falive.to(DTYPE)).sum(1)
    en_fleet = (f_ships * (f_owner != float(ego)).to(DTYPE) * falive.to(DTYPE)).sum(1)
    ME = 40.0
    g0 = step_ct / max(1, T)
    g1 = ang_vel * 10.0
    g2 = ship_log_t(my_ships); g3 = ship_log_t(en_ships)
    g4 = my_pl / total; g5 = en_pl / total; g6 = neu_pl / total
    g7 = ship_log_t(my_fleet); g8 = ship_log_t(en_fleet)
    g9 = torch.minimum(npl, torch.full_like(npl, ME)) / ME
    globals_ = torch.stack([g0, g1, g2, g3, g4, g5, g6, g7, g8, g9], 1)
    return entities, entity_mask, action_mask, globals_


# The compiled handle is installed by maybe_compile_encode(cfg) at train start (USE_COMPILE); until
# then env_encode runs _encode_core EAGERLY -- so eval/league/MCTS paths are unaffected unless compiled.
_ENCODE_CORE = _encode_core


def env_encode(env, ego=0):
    """v9: ownership is a 4-channel EGO-RELATIVE seat ONE-HOT [self, enemy seat+1, seat+2,
    seat+3] (neutral = all-zero), so the policy can tell WHICH opponent owns a body with no
    ordinal fake-magnitude. In 2p only the first two channels ever fire, so 4p generalizes the
    same input space; a dead seat's channel simply stops firing. The GLOBAL enemy aggregates
    still pool ALL opponents (everyone-vs-me totals).

    Thin env wrapper: unpacks the env tensors + scalars and calls _encode_core (compiled in place by
    maybe_compile_encode, else eager). Keeping the heavy math in a pure-tensor function is what lets
    torch.compile trace it -- dynamo cannot follow the env object's getattr-with-default accesses."""
    na = float(getattr(env, "n_players", 2))
    p_x_prev = getattr(env, "p_x_prev", env.p_x); p_y_prev = getattr(env, "p_y_prev", env.p_y)
    p_x_prev2 = getattr(env, "p_x_prev2", p_x_prev); p_y_prev2 = getattr(env, "p_y_prev2", p_y_prev)
    # The encoder never needs grad (its output is stored into the rollout buffers as constants; the
    # update re-runs net(...) on those, not on this output). Forcing no_grad at the dispatch boundary
    # keeps grad_mode CONSTANT across every caller (rollout ego/opp, league recal, eval, MCTS) so the
    # torch.compile'd _encode_core never recompiles on a grad_mode guard flip (was thrashing the
    # recompile_limit when the ungated ego encode ran grad-on between no_grad opponent encodes).
    with torch.no_grad():
        return _ENCODE_CORE(int(ego), na, int(env.T), env.vmax, env.p_alive, env.p_owner, env.p_x, env.p_y,
                            env.p_radius, env.p_is_comet, env.ang_vel, env.p_prod, env.p_ships, env.f_x,
                            env.f_y, env.f_angle, env.f_ships, env.f_alive, env.f_owner, env.f_seq,
                            env.step_ct, p_x_prev, p_y_prev, p_x_prev2, p_y_prev2)


def maybe_compile_encode(cfg):
    """Install a torch.compile'd _encode_core as the handle env_encode dispatches to (idempotent,
    called once at train start when cfg.USE_COMPILE). The pure core has only tensor args + a few
    python scalars (ego/na/T/vmax), so it traces with NO graph breaks -- fusing the ~50-op encoder the
    launch-bound rollout calls 2-4x/step. Robust: torch.compile is LAZY, so a backend failure surfaces
    on the FIRST call, not at compile() time -- a one-shot guard catches it and permanently reverts to
    eager, so an absent/broken compiler can never kill a training run."""
    global _ENCODE_CORE
    if not getattr(cfg, "USE_COMPILE", False) or getattr(_ENCODE_CORE, "_ow_compiled", False):
        return _ENCODE_CORE
    try:
        compiled = torch.compile(_encode_core, mode=getattr(cfg, "COMPILE_MODE", "default"))
    except Exception as e:
        print("  [compile] env_encode not compiled (%r) -> eager" % e)
        return _ENCODE_CORE

    def _guard(*a, **k):                  # first call triggers the (lazy) backend; fall back on failure
        global _ENCODE_CORE
        try:
            out = compiled(*a, **k)
            _ENCODE_CORE = compiled       # success -> dispatch straight to the bare compiled core thereafter
            return out
        except Exception as e:
            print("  [compile] env_encode failed at runtime (%r) -> reverting to eager" % e)
            _ENCODE_CORE = _encode_core
            return _encode_core(*a, **k)
    _guard._ow_compiled = True
    _ENCODE_CORE = _guard
    return _ENCODE_CORE


def launch_fleets(env, owner, from_slot, angle, ships, commit, seq):
    '''Append committed launches (all (B,L)) into the fleet pool, distinct free slots.'''
    B, Fc = env.B, env.Fc
    dev = env.dev
    L = from_slot.shape[1]
    free = env.f_alive < 0.5
    freef = free.to(DTYPE)
    fr = torch.cumsum(freef, 1) - 1.0
    n_free = freef.sum(1)
    idx = torch.where(free, fr.long(), torch.full_like(fr.long(), Fc))
    rank_to_slot = torch.full((B, Fc + 1), Fc, dtype=torch.long, device=dev)
    slot_src = torch.arange(Fc, dtype=torch.long, device=dev).unsqueeze(0).expand(B, Fc)
    rank_to_slot.scatter_(1, idx, slot_src)
    rank_to_slot = rank_to_slot[:, :Fc]
    commitb = commit > 0.5
    crank = (torch.cumsum(commit.to(DTYPE), 1) - 1.0).long()
    place = commitb & (crank < n_free.unsqueeze(1).long())
    gslot = rank_to_slot.gather(1, crank.clamp(0, Fc - 1))
    dump = torch.full_like(gslot, Fc)
    wslot = torch.where(place, gslot, dump)
    def scatter_into(field, vals):
        aug = torch.cat([field, torch.zeros(B, 1, dtype=DTYPE, device=dev)], 1)
        aug.scatter_(1, wslot, vals)
        return aug[:, :Fc]
    fs = from_slot.clamp(0, env.p_x.shape[1] - 1)
    opx = env.p_x.gather(1, fs); opy = env.p_y.gather(1, fs); orad = env.p_radius.gather(1, fs)
    sx = opx + torch.cos(angle) * (orad + 0.1)
    sy = opy + torch.sin(angle) * (orad + 0.1)
    onesL = torch.ones(B, L, dtype=DTYPE, device=dev)
    env.f_alive = scatter_into(env.f_alive, onesL)
    env.f_owner = scatter_into(env.f_owner, owner)
    env.f_x = scatter_into(env.f_x, sx)
    env.f_y = scatter_into(env.f_y, sy)
    env.f_angle = scatter_into(env.f_angle, angle)
    env.f_ships = scatter_into(env.f_ships, ships)
    env.f_seq = scatter_into(env.f_seq, seq)


def opponent_action(cfg, env, opponent, pid=1):
    '''Scripted launches for seat `pid` (v8: any seat in a 2p/4p game): one per owned planet, half garrison (>=20).
    0=random heading, 1=starter (nearest static non-owned), 2=noop, 3=medium (starter++: nearest incl. rotating, lead-aim),
    4=greedy (nearest beatable non-owned planet within CAPTURE_RADIUS, just-enough ships),
    5=intermediate (in-flight-aware capture from nearest capable source + counter + rebalance + comet-escape).
    -> angle,ships,commit (B,Ec).'''
    B, Ec = env.B, env.Ec
    dev = env.dev
    min_ships = 20.0
    angle = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    ships = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    commit = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    if opponent == 2:
        return angle, ships, commit
    half = torch.floor(env.p_ships / 2.0)
    base = (env.p_owner == float(pid)) & (env.p_alive > 0.5) & (half >= min_ships)
    if opponent == 0:  # random heading
        angle = torch.rand(B, Ec, dtype=DTYPE, device=dev) * (2.0 * PI)
        ships = torch.where(base, half, torch.zeros_like(half))
        commit = base.to(DTYPE)
        return angle, ships, commit
    if opponent == 3:  # medium (starter++): nearest non-owned planet INCL. rotating, lead-aimed at its intercept
        sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)             # (B,Ec,1) source
        tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)            # (B,1,Ec) dest
        vt = (env.p_alive > 0.5) & (env.p_owner != float(pid))                 # (B,Ec) valid targets (rotating allowed)
        d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)
        dmask = torch.where(vt.unsqueeze(1), d, torch.full_like(d, BIG))
        bestd, tgt = dmask.min(2)                                       # (B,Ec) nearest valid target per source
        has_tgt = bestd < BIG * 0.5
        dpx = env.p_x.gather(1, tgt); dpy = env.p_y.gather(1, tgt)      # (B,Ec) target position
        if cfg.LEAD_TARGET:                                            # lead the (possibly orbiting) target
            rdx = dpx - CENTER; rdy = dpy - CENTER
            r_d = torch.sqrt(rdx * rdx + rdy * rdy)
            phi0 = torch.atan2(rdy, rdx)
            drot = env.p_rotates.gather(1, tgt) > 0.5
            w = torch.where(drot, env.ang_vel.view(B, 1), torch.zeros(B, 1, device=dev, dtype=DTYPE))
            off = env.p_radius + 0.1
            v = fleet_speed_t(half.clamp_min(1.0), env.vmax)
            t = (torch.sqrt((dpx - env.p_x) ** 2 + (dpy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            for _ in range(16):
                phi = phi0 + w * t
                ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
                t = (torch.sqrt((ix - env.p_x) ** 2 + (iy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            angle = torch.atan2(iy - env.p_y, ix - env.p_x)
        else:
            angle = torch.atan2(dpy - env.p_y, dpx - env.p_x)
        go = base & has_tgt
        ships = torch.where(go, half, torch.zeros_like(half))
        commit = go.to(DTYPE)
        return angle, ships, commit
    if opponent == 4:  # greedy: capture the NEAREST non-owned planet within CAPTURE_RADIUS we can already beat
        sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)            # (B,Ec,1) source
        tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)           # (B,1,Ec) dest
        d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)                # (B,Ec,Ec)
        vt = (env.p_alive > 0.5) & (env.p_owner != float(pid))                # (B,Ec) non-owned alive
        can_beat = env.p_ships.unsqueeze(2) > env.p_ships.unsqueeze(1) # (B,src,dst) src ships > dst ships
        cand = vt.unsqueeze(1) & can_beat & (d <= cfg.CAPTURE_RADIUS)  # capturable & in range
        dmask = torch.where(cand, d, torch.full_like(d, BIG))
        bestd, tgt = dmask.min(2)                                       # (B,Ec) nearest capturable target
        has_tgt = bestd < BIG * 0.5
        tgt_ships = env.p_ships.gather(1, tgt)                          # (B,Ec)
        n = torch.minimum(env.p_ships, torch.floor(tgt_ships) + cfg.CAPTURE_MARGIN)  # just enough to capture (+ buffer)
        dpx = env.p_x.gather(1, tgt); dpy = env.p_y.gather(1, tgt)      # (B,Ec) target position
        if cfg.LEAD_TARGET:                                            # lead the (possibly orbiting) target
            rdx = dpx - CENTER; rdy = dpy - CENTER
            r_d = torch.sqrt(rdx * rdx + rdy * rdy)
            phi0 = torch.atan2(rdy, rdx)
            drot = env.p_rotates.gather(1, tgt) > 0.5
            w = torch.where(drot, env.ang_vel.view(B, 1), torch.zeros(B, 1, device=dev, dtype=DTYPE))
            off = env.p_radius + 0.1
            v = fleet_speed_t(n.clamp_min(1.0), env.vmax)
            t = (torch.sqrt((dpx - env.p_x) ** 2 + (dpy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            for _ in range(16):
                phi = phi0 + w * t
                ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
                t = (torch.sqrt((ix - env.p_x) ** 2 + (iy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            angle = torch.atan2(iy - env.p_y, ix - env.p_x)
        else:
            angle = torch.atan2(dpy - env.p_y, dpx - env.p_x)
        own_ok = (env.p_owner == float(pid)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
        go = own_ok & has_tgt & (n >= 1.0)
        ships = torch.where(go, n, torch.zeros_like(n))
        commit = go.to(DTYPE)
        return angle, ships, commit
    if opponent == 5:  # intermediate: in-flight-aware capture from nearest capable source (cascade) + rebalance
        owned = (env.p_owner == float(pid)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)   # (B,Ec) sources that can act
        sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)
        tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)
        d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)                # (B,src,dst)
        srcrange = torch.arange(Ec, device=dev).view(1, Ec, 1)
        # capture (in-flight aware): per target, needed = garrison + enemy_inbound - friendly_inbound + margin
        ftgt, _feta, *_ = fleet_target_batch(env, env.f_x, env.f_y, env.f_angle, env.f_ships, env.vmax)
        f_ok = (env.f_alive > 0.5) & (ftgt >= 0)            # SUN-AWARE target: sun-bound fleets don't arrive
        tslot_f = ftgt.clamp(0, Ec - 1)
        in_fr = torch.zeros(B, Ec, device=dev, dtype=DTYPE)            # friendly ships already inbound per planet
        in_en = torch.zeros(B, Ec, device=dev, dtype=DTYPE)            # enemy ships inbound per planet
        in_fr.scatter_add_(1, tslot_f, env.f_ships * ((env.f_owner == float(pid)) & f_ok).to(DTYPE))
        in_en.scatter_add_(1, tslot_f, env.f_ships * ((env.f_owner != float(pid)) & f_ok).to(DTYPE))
        vt = (env.p_alive > 0.5) & (env.p_owner != float(pid))
        needed_t = torch.floor(env.p_ships) + torch.ceil(in_en) - torch.floor(in_fr) + cfg.CAPTURE_MARGIN   # (B,Ec)
        open_t = vt & (needed_t > 0.0)                                 # not already covered by friendly inbound
        can_supply = env.p_ships.unsqueeze(2) >= needed_t.unsqueeze(1) # (B,src,dst) source can cover the requirement
        capable = owned.unsqueeze(2) & open_t.unsqueeze(1) & can_supply
        d_cap = torch.where(capable, d, torch.full_like(d, BIG))
        nearest_cap = d_cap.argmin(1)                                  # (B,dst) nearest CAPABLE source (cascade)
        target_has = d_cap.min(1).values < BIG * 0.5
        assigned = (nearest_cap.unsqueeze(1) == srcrange) & target_has.unsqueeze(1)
        d_asg = torch.where(assigned, d, torch.full_like(d, BIG))
        cap_d, cap_tgt = d_asg.min(2)                                  # (B,Ec) source's nearest assigned target
        cap_has = cap_d < BIG * 0.5
        cap_n = torch.minimum(env.p_ships, needed_t.gather(1, cap_tgt))   # send exactly the requirement
        # rebalance: feed the poorest owned planet WITHIN REBALANCE_RADIUS (local) when surplus exceeds a percentage
        eyeE = (torch.eye(Ec, device=dev, dtype=DTYPE).unsqueeze(0) > 0.5)
        reb_cand = owned.unsqueeze(1) & (d <= cfg.REBALANCE_RADIUS) & (~eyeE)   # (B,src,dst) owned, in-radius, not self
        reb_dst_ships = torch.where(reb_cand, env.p_ships.unsqueeze(1), torch.full_like(d, BIG))
        poor_ships, poor_idx = reb_dst_ships.min(2)                    # (B,Ec) poorest owned-in-radius per source
        has_poor = poor_ships < BIG * 0.5
        gap = env.p_ships - poor_ships
        reb_ok = owned & (~cap_has) & has_poor & (gap > cfg.REBALANCE_PCT * env.p_ships.clamp_min(1.0))
        reb_n = torch.floor(gap * 0.5)
        reb_tgt = poor_idx
        # comet escape (top priority): owned comets evacuate ALL ships to the nearest owned non-comet planet
        safe_dst = (owned & (env.p_is_comet < 0.5)).unsqueeze(1) & (~eyeE)   # (B,src,dst) dest owned non-comet, not self
        d_evac = torch.where(safe_dst, d, torch.full_like(d, BIG))
        evac_d, evac_tgt = d_evac.min(2)
        evac_ok = owned & (env.p_is_comet > 0.5) & (evac_d < BIG * 0.5) & (env.p_ships >= 1.0)
        # priority: comet-escape > capture > rebalance
        use_cap = (~evac_ok) & cap_has
        use_reb = (~evac_ok) & (~cap_has) & reb_ok
        tgt = torch.where(evac_ok, evac_tgt, torch.where(use_cap, cap_tgt, reb_tgt))
        nn = torch.where(evac_ok, env.p_ships, torch.where(use_cap, cap_n, torch.where(use_reb, reb_n, torch.zeros_like(reb_n))))
        go = evac_ok | use_cap | use_reb
        dpx = env.p_x.gather(1, tgt); dpy = env.p_y.gather(1, tgt)
        if cfg.LEAD_TARGET:
            rdx = dpx - CENTER; rdy = dpy - CENTER
            r_d = torch.sqrt(rdx * rdx + rdy * rdy)
            phi0 = torch.atan2(rdy, rdx)
            drot = env.p_rotates.gather(1, tgt) > 0.5
            w = torch.where(drot, env.ang_vel.view(B, 1), torch.zeros(B, 1, device=dev, dtype=DTYPE))
            off = env.p_radius + 0.1
            v = fleet_speed_t(nn.clamp_min(1.0), env.vmax)
            t = (torch.sqrt((dpx - env.p_x) ** 2 + (dpy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            for _ in range(16):
                phi = phi0 + w * t
                ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
                t = (torch.sqrt((ix - env.p_x) ** 2 + (iy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            angle = torch.atan2(iy - env.p_y, ix - env.p_x)
        else:
            angle = torch.atan2(dpy - env.p_y, dpx - env.p_x)
        ships = torch.where(go, nn, torch.zeros_like(nn))
        commit = go.to(DTYPE)
        return angle, ships, commit
    # opponent == 1: starter -- fire at nearest static non-owned planet
    distc = torch.sqrt((env.p_x - CENTER) ** 2 + (env.p_y - CENTER) ** 2)
    is_static = (distc + env.p_radius) >= ROTATION_RADIUS_LIMIT
    valid_tgt = is_static & (env.p_alive > 0.5) & (env.p_owner != float(pid))
    sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)
    tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)
    d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)
    dmask = torch.where(valid_tgt.unsqueeze(1), d, torch.full_like(d, BIG))
    bestd, bestidx = dmask.min(2)
    has_tgt = bestd < BIG * 0.5
    btx = env.p_x.gather(1, bestidx); bty = env.p_y.gather(1, bestidx)
    angle = torch.atan2(bty - env.p_y, btx - env.p_x)
    go = base & has_tgt
    ships = torch.where(go, half, torch.zeros_like(half))
    commit = go.to(DTYPE)
    return angle, ships, commit


class StepOut:
    pass


def _decode_target(cfg, env, action, legal):
    '''Gated-allocation decode. action (B,Ec,Ec+1): [...,:Ec]=WHERE allocation rows, [...,Ec]=fire{0,1}.
    A planet FIRES (full garrison, routed by its renormalised off-diagonal WHERE row) or HOLDS. ships[s,d]
    = floor(Ahat[s,d]*S_s) for d!=s, to ALIVE dests, >= MIN_LAUNCH_SHIPS, only if fire[s]=1; the rest stay
    home. Each launch aims at the orbiting intercept. Returns angle/ships/can (B,Ec,Ec), valid/invalid/launches (B,).'''
    B, Ec, dev = env.B, env.Ec, env.dev
    A = action[..., :Ec]                                        # (B,Ec,Ec) WHERE allocation
    fire = action[..., Ec]                                      # (B,Ec) launch gate in {0,1}
    S = env.p_ships                                             # (B,Ec)
    eye = torch.eye(Ec, dtype=DTYPE, device=dev).unsqueeze(0)   # (1,Ec,Ec)
    A_send = A * (1.0 - eye)                                    # drop self-diagonal (s->s meaningless)
    A_send = A_send / A_send.sum(2, keepdim=True).clamp_min(1e-6)   # renormalise -> fire = FULL garrison out
    raw = A_send * (S * fire).unsqueeze(2)                      # (B,Ec,Ec) ships src->dst, zero if held
    dest_alive = (env.p_alive > 0.5).to(DTYPE).unsqueeze(1)     # (B,1,Ec)
    gate = legal.to(DTYPE).unsqueeze(2) * dest_alive            # (B,Ec,Ec) legal source & alive dest
    n = torch.floor(raw) * gate
    n = torch.where(n >= float(cfg.MIN_LAUNCH_SHIPS), n, torch.zeros_like(n))   # drop dribbles -> stay home
    can = (n >= 1.0).to(DTYPE)
    # intercept heading for every (src,dst): src = row planet, dst = col planet
    spx = env.p_x.unsqueeze(2); spy = env.p_y.unsqueeze(2)      # (B,Ec,1) source
    dpx = env.p_x.unsqueeze(1); dpy = env.p_y.unsqueeze(1)      # (B,1,Ec) dest
    off = env.p_radius.unsqueeze(2) + 0.1                       # (B,Ec,1) source surface
    if cfg.LEAD_TARGET:
        rdx = dpx - CENTER; rdy = dpy - CENTER
        r_d = torch.sqrt(rdx * rdx + rdy * rdy)                 # (B,1,Ec)
        phi0 = torch.atan2(rdy, rdx)
        drot = (env.p_rotates > 0.5).unsqueeze(1)               # (B,1,Ec)
        w = torch.where(drot, env.ang_vel.view(B, 1, 1), torch.zeros(B, 1, 1, device=dev, dtype=DTYPE))
        v = fleet_speed_t(n.clamp_min(1.0), env.vmax)           # (B,Ec,Ec)
        t = (torch.sqrt((dpx - spx) ** 2 + (dpy - spy) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
        for _ in range(16):
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            t = (torch.sqrt((ix - spx) ** 2 + (iy - spy) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
        phi = phi0 + w * t
        ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
        angle = torch.atan2(iy - spy, ix - spx)                 # (B,Ec,Ec)
    else:
        angle = torch.atan2(dpy - spy, dpx - spx).expand(B, Ec, Ec).contiguous()
    valid = can.sum((1, 2))                                     # launches all go to alive dests
    invalid = torch.zeros(B, dtype=DTYPE, device=dev)           # below-min launches just stay home (no penalty); budget is structural
    launches = can.sum((1, 2))
    return angle, n, can, valid, invalid, launches


def spawn_comets(cfg, env):
    '''Spawn COMET_PAIRS point-symmetric comet pairs into free planet slots: neutral objects that
    sweep the inner region at COMET_SPEED and expire past COMET_EXPIRE_RADIUS. Capturable; produce
    COMET_PRODUCTION once owned. Velocity aims past the center at a random perihelion (avoids the sun).'''
    B, dev = env.B, env.dev
    for _pair in range(cfg.COMET_PAIRS):
        theta = torch.rand(B, device=dev) * (2.0 * PI)
        d_peri = cfg.COMET_PERI_MIN + torch.rand(B, device=dev) * (cfg.COMET_PERI_MAX - cfg.COMET_PERI_MIN)
        sgn = torch.where(torch.rand(B, device=dev) < 0.5, torch.ones(B, device=dev), -torch.ones(B, device=dev))
        ships = torch.randint(1, 100, (B, 4), device=dev).min(1).values.to(DTYPE)   # min-of-4 -> skewed low
        for _c in range(2):
            th = theta + (0.0 if _c == 0 else PI)                       # point-symmetric pair
            r0 = float(cfg.COMET_SPAWN_RADIUS)
            px = CENTER + r0 * torch.cos(th); py = CENTER + r0 * torch.sin(th)
            toward = torch.atan2(CENTER - py, CENTER - px)
            alpha = torch.asin((d_peri / r0).clamp(max=0.99)) * (sgn if _c == 0 else -sgn)
            vdir = toward + alpha
            vx = cfg.COMET_SPEED * torch.cos(vdir); vy = cfg.COMET_SPEED * torch.sin(vdir)
            free = env.p_alive < 0.5
            slot = torch.argmax(free.to(torch.int8), 1, keepdim=True)    # (B,1) first free slot
            has = free.any(1, keepdim=True)
            def _set(field, val):
                v = val.unsqueeze(1) if val.dim() == 1 else val
                field.scatter_(1, slot, torch.where(has, v, field.gather(1, slot)))
            _set(env.p_alive, torch.ones(B, device=dev))
            _set(env.p_owner, torch.full((B,), -1.0, device=dev))
            _set(env.p_x, px); _set(env.p_y, py); _set(env.p_init_x, px); _set(env.p_init_y, py)
            # v13: seed the position history at the spawn point so the first finite-diff velocity/curvature
            # is computed off the comet itself, not the recycled slot's stale (dead-planet) prev position.
            _set(env.p_x_prev, px); _set(env.p_y_prev, py); _set(env.p_x_prev2, px); _set(env.p_y_prev2, py)
            _set(env.p_comet_vx, vx); _set(env.p_comet_vy, vy)
            _set(env.p_ships, ships)
            _set(env.p_prod, torch.full((B,), float(COMET_PRODUCTION), device=dev))
            _set(env.p_radius, torch.full((B,), float(COMET_RADIUS), device=dev))
            _set(env.p_is_comet, torch.ones(B, device=dev))
            _set(env.p_rotates, torch.zeros(B, device=dev))


def spawn_comets_official(env, e):
    """Place the 4 precomputed symmetric comets of spawn event e at waypoint 0 (free planet slots)."""
    B, dev = env.B, env.dev
    has = env.c_len[:, e] > 0                                    # (B,) event exists for this world
    one = torch.ones(B, 1, dtype=DTYPE, device=dev)
    for m in range(4):
        free = env.p_alive < 0.5
        slot = torch.argmax(free.to(torch.int8), 1, keepdim=True)            # (B,1) first free slot
        ok = (has & free.any(1)).unsqueeze(1)                                # (B,1)
        px = env.c_paths[:, e, m, 0, 0:1]; py = env.c_paths[:, e, m, 0, 1:2]
        def _set(field, val):
            field.scatter_(1, slot, torch.where(ok, val, field.gather(1, slot)))
        _set(env.p_alive, one)
        _set(env.p_owner, -one)
        _set(env.p_x, px); _set(env.p_y, py)
        _set(env.p_init_x, px); _set(env.p_init_y, py)
        # v13: seed position history at the spawn waypoint (see spawn_comets) so the first velocity/
        # curvature is off the comet, not the recycled slot's stale prev.
        _set(env.p_x_prev, px); _set(env.p_y_prev, py); _set(env.p_x_prev2, px); _set(env.p_y_prev2, py)
        _set(env.p_ships, env.c_ships[:, e].unsqueeze(1))
        _set(env.p_prod, one * float(COMET_PRODUCTION))
        _set(env.p_radius, one * float(COMET_RADIUS))
        _set(env.p_is_comet, one)
        _set(env.p_rotates, torch.zeros_like(one))
        env.c_slot[:, e, m] = torch.where(ok.squeeze(1), slot.squeeze(1),
                                          torch.full_like(slot.squeeze(1), -1))


def env_step(cfg, env, ego_action, opponent=None, opp_action=None, step_idx=None, seats=None):
    """One tick, N-player (v8). ego_action (B,Ec,Ec+1)=[alloc | fire] for player 0.
    seats = list of opponent seat dicts, one per pid 1..n_players-1:
        {"pid": p, "script": code}    scripted bot (opponent_action codes), or
        {"pid": p, "action": tensor}  neural (B,Ec,Ec+1) gated-alloc action.
    v7 back-compat: seats=None -> a single pid-1 seat built from (opponent, opp_action)."""
    B, Ec, Fc = env.B, env.Ec, env.Fc
    N = int(getattr(env, "n_players", 2))
    if seats is None:
        seats = [{"pid": 1, "action": opp_action}] if opp_action is not None else [{"pid": 1, "script": opponent}]
    _sc = int(env.step_ct[0].item()) if step_idx is None else int(step_idx)   # avoid a per-step CUDA sync
    if cfg.COMETS_ENABLED and _sc in cfg.COMET_SPAWN_STEPS:   # hidden-schedule comet spawn
        if cfg.COMET_OFFICIAL and getattr(env, 'c_paths', None) is not None:
            spawn_comets_official(env, cfg.COMET_SPAWN_STEPS.index(_sc))
        else:
            spawn_comets(cfg, env)
    vmax = env.vmax
    dev = env.dev
    ego = 0

    ego_owned0 = (env.p_owner == float(ego)) & (env.p_alive > 0.5)
    comet0 = env.p_is_comet > 0.5   # comet status at step start (mask comet expiry out of capture/lost)

    # --- decode ego action (gated allocation: fire -> full garrison, routed) ---
    legal = (env.p_owner == float(ego)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
    e_ang, e_shp, e_can, valid, invalid, launches = _decode_target(cfg, env, ego_action, legal)

    # --- opponent seat launches (scripted: one launch/planet; neural: full alloc decode) ---
    slot_idx = torch.arange(Ec, dtype=torch.long, device=dev).unsqueeze(0).expand(B, Ec)
    src_mat = torch.arange(Ec, dtype=torch.long, device=dev).view(1, Ec, 1).expand(B, Ec, Ec).reshape(B, Ec * Ec)
    flatM = lambda x: x.reshape(B, Ec * Ec)                                   # (B,Ec,Ec) -> (B,Ec*Ec)
    own_blk = [torch.full((B, Ec * Ec), float(ego), dtype=DTYPE, device=dev)]
    slot_blk = [src_mat]
    ang_blk = [flatM(e_ang)]; shp_blk = [flatM(e_shp)]; can_blk = [flatM(e_can)]
    ded = e_shp.sum(2)                                                        # (B,Ec) ships deducted/source
    for st in seats:
        pid = int(st["pid"])
        if st.get("action") is not None:          # neural seat: decode exactly like the ego
            legal_p = (env.p_owner == float(pid)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
            o_ang, o_shp, o_can, _, _, _ = _decode_target(cfg, env, st["action"], legal_p)
            ded = ded + o_shp.sum(2)
            own_blk.append(torch.full((B, Ec * Ec), float(pid), dtype=DTYPE, device=dev))
            slot_blk.append(src_mat)
            ang_blk.append(flatM(o_ang)); shp_blk.append(flatM(o_shp)); can_blk.append(flatM(o_can))
        else:                                     # scripted seat: one launch per owned planet
            o_ang, o_shp, o_can = opponent_action(cfg, env, int(st["script"]), pid)
            ded = ded + o_shp
            own_blk.append(torch.full((B, Ec), float(pid), dtype=DTYPE, device=dev))
            slot_blk.append(slot_idx)
            ang_blk.append(o_ang); shp_blk.append(o_shp); can_blk.append(o_can)

    # --- deduct ships from origin planets ---
    env.p_ships = env.p_ships - ded

    # --- place fleets (ego first, then seats in pid order) ---
    owner = torch.cat(own_blk, 1)
    from_slot = torch.cat(slot_blk, 1)
    angle = torch.cat(ang_blk, 1)
    ships = torch.cat(shp_blk, 1)
    commit = torch.cat(can_blk, 1)
    Ltot = owner.shape[1]
    seq = (env.step_ct * float(Ltot + 1)).unsqueeze(1) + \
          torch.arange(Ltot, dtype=DTYPE, device=dev).unsqueeze(0)
    launch_fleets(env, owner, from_slot, angle, ships, commit, seq)

    # --- production ---
    env.p_ships = env.p_ships + env.p_prod * (env.p_owner != -1.0).to(DTYPE) * (env.p_alive > 0.5).to(DTYPE)

    # --- planet new positions (orbit) ---
    stepf = env.step_ct
    dxc = env.p_init_x - CENTER; dyc = env.p_init_y - CENTER
    r = torch.sqrt(dxc * dxc + dyc * dyc)
    ia = torch.atan2(dyc, dxc)
    ca = ia + env.ang_vel.unsqueeze(1) * stepf.unsqueeze(1)
    rot = env.p_rotates > 0.5
    cmt = env.p_is_comet > 0.5
    nx = torch.where(rot, CENTER + r * torch.cos(ca), torch.where(cmt, env.p_x + env.p_comet_vx, env.p_x))
    ny = torch.where(rot, CENTER + r * torch.sin(ca), torch.where(cmt, env.p_y + env.p_comet_vy, env.p_y))
    old_px, old_py = env.p_x, env.p_y
    _comet_expired = None
    if cfg.COMETS_ENABLED and cfg.COMET_OFFICIAL and getattr(env, 'c_paths', None) is not None:
        # waypoint playback: comet of event e at tick u sweeps path[u-s] -> path[u-s+1]
        _comet_expired = torch.zeros(B, Ec, dtype=torch.bool, device=dev)
        _ar = torch.arange(B, device=dev)
        for _e, _s_e in enumerate(cfg.COMET_SPAWN_STEPS):
            if not (_s_e <= _sc <= _s_e + cfg.COMET_MAX_LEN):   # at most one event is ever live
                continue
            k = int(_sc - _s_e + 1)                          # waypoint index this tick (same for all envs)
            for _m in range(4):
                sl = env.c_slot[:, _e, _m]                   # (B,) planet slot, -1 = none
                live = sl >= 0
                if not bool(live.any()):
                    continue
                slc = sl.clamp_min(0).unsqueeze(1)
                adv = live & (k < env.c_len[:, _e])          # still has waypoints -> advance
                exp = live & (k >= env.c_len[:, _e])         # path ended -> expire after combat
                wp = env.c_paths[_ar, _e, _m, min(k, cfg.COMET_MAX_LEN - 1)]   # (B,2)
                nx.scatter_(1, slc, torch.where(adv.unsqueeze(1), wp[:, 0:1], nx.gather(1, slc)))
                ny.scatter_(1, slc, torch.where(adv.unsqueeze(1), wp[:, 1:2], ny.gather(1, slc)))
                _comet_expired.scatter_(1, slc, exp.unsqueeze(1) | _comet_expired.gather(1, slc))
                env.c_slot[:, _e, _m] = torch.where(exp, torch.full_like(sl, -1), sl)

    # --- fleet movement + swept collision against planet paths ---
    falive = env.f_alive > 0.5
    speed = fleet_speed_t(env.f_ships, vmax)
    fox, foy = env.f_x, env.f_y
    fnx = fox + torch.cos(env.f_angle) * speed
    fny = foy + torch.sin(env.f_angle) * speed
    Ax = fox.unsqueeze(2); Ay = foy.unsqueeze(2)
    Bx = fnx.unsqueeze(2); By = fny.unsqueeze(2)
    P0x = old_px.unsqueeze(1); P0y = old_py.unsqueeze(1)
    P1x = nx.unsqueeze(1); P1y = ny.unsqueeze(1)
    rad = env.p_radius.unsqueeze(1)
    palive = (env.p_alive.unsqueeze(1) > 0.5)
    d0x = Ax - P0x; d0y = Ay - P0y
    dvx = (Bx - Ax) - (P1x - P0x); dvy = (By - Ay) - (P1y - P0y)
    a = dvx * dvx + dvy * dvy
    b = 2.0 * (d0x * dvx + d0y * dvy)
    c = d0x * d0x + d0y * d0y - rad * rad
    disc = b * b - 4.0 * a * c
    sq = torch.sqrt(disc.clamp_min(0.0))
    t1 = (-b - sq) / (2.0 * a)
    t2 = (-b + sq) / (2.0 * a)
    hit_quad = (disc >= 0.0) & (t2 >= 0.0) & (t1 <= 1.0)
    hit_lin = (a < 1e-12) & (c <= 0.0)
    hit = torch.where(a < 1e-12, hit_lin, hit_quad)
    hit = hit & palive & falive.unsqueeze(2)
    slotf = torch.arange(Ec, dtype=DTYPE, device=dev).view(1, 1, Ec)
    order = torch.where(hit, slotf, torch.full_like(slotf, float(Ec)))
    fh_v, tgt_slot = order.min(2)
    has_hit = fh_v < float(Ec)

    # OOB / sun removal (point-to-segment to sun center)
    oob = (fnx < 0.0) | (fnx > BOARD_SIZE) | (fny < 0.0) | (fny > BOARD_SIZE)
    vx, vy, wx, wy = fox, foy, fnx, fny
    l2 = (vx - wx) ** 2 + (vy - wy) ** 2
    tt = ((CENTER - vx) * (wx - vx) + (CENTER - vy) * (wy - vy)) / l2.clamp_min(1e-12)
    tt = tt.clamp(0.0, 1.0)
    prx = vx + tt * (wx - vx); pry = vy + tt * (wy - vy)
    sundist = torch.sqrt((CENTER - prx) ** 2 + (CENTER - pry) ** 2)
    sun_hit = (l2 > 0.0) & (sundist < SUN_RADIUS)
    sun_pt = torch.sqrt((CENTER - vx) ** 2 + (CENTER - vy) ** 2) < SUN_RADIUS
    sun_hit = torch.where(l2 > 0.0, sun_hit, sun_pt)

    remove_fleet = falive & (has_hit | oob | sun_hit)
    contributes = falive & has_hit

    # --- combat: arrivals per PLAYER, official top-vs-second resolve (N-player) ---
    cf = contributes.to(DTYPE)
    tslot = tgt_slot.clamp(0, Ec - 1)
    arr_p = []
    for p in range(N):
        sp = env.f_ships * cf * (env.f_owner == float(p)).to(DTYPE)
        ap = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
        ap.scatter_add_(1, tslot, sp)
        arr_p.append(ap)
    arr = torch.stack(arr_p, 2)                                  # (B,Ec,N)
    top2v, top2i = arr.topk(2, dim=2)                            # N >= 2 always
    surv_ships = top2v[..., 0] - top2v[..., 1]                   # exact tie -> 0 survivors
    surv_owner = torch.where(surv_ships > 0.0, top2i[..., 0].to(DTYPE),
                             torch.full((B, Ec), -1.0, dtype=DTYPE, device=dev))
    any_arr = top2v[..., 0] > 0.0
    apply = any_arr & (surv_ships > 0.0) & (env.p_alive > 0.5)
    same = (env.p_owner == surv_owner)
    reinforce = apply & same
    attack = apply & (~same)
    env.p_ships = torch.where(reinforce, env.p_ships + surv_ships, env.p_ships)
    after = env.p_ships - surv_ships
    flips = attack & (after < 0.0)
    env.p_ships = torch.where(attack, torch.where(after < 0.0, -after, after), env.p_ships)
    env.p_owner = torch.where(flips, surv_owner, env.p_owner)

    # --- apply planet positions; clear removed fleets ---
    # shift the position history: prev2 <- prev (pre-update) THEN prev <- this step's start pos.
    # encode at the next step start then reads p_x(=new), p_x_prev(=this start), p_x_prev2(=last start)
    # -> velocity = p_x-prev, curvature = p_x-2*prev+prev2 (threat lead still uses prev only).
    env.p_x_prev2 = env.p_x_prev; env.p_y_prev2 = env.p_y_prev
    env.p_x_prev = old_px; env.p_y_prev = old_py
    env.p_x = nx; env.p_y = ny
    if cfg.COMETS_ENABLED:                               # comet expiry: official = path end; legacy = swept past the inner region
        if _comet_expired is not None:
            gone = _comet_expired & (env.p_alive > 0.5)
        else:
            cdist = torch.sqrt((env.p_x - CENTER) ** 2 + (env.p_y - CENTER) ** 2)
            gone = (env.p_is_comet > 0.5) & (env.p_alive > 0.5) & (cdist > cfg.COMET_EXPIRE_RADIUS)
        keepc = (~gone).to(DTYPE)
        env.p_alive = env.p_alive * keepc; env.p_is_comet = env.p_is_comet * keepc
        env.p_comet_vx = env.p_comet_vx * keepc; env.p_comet_vy = env.p_comet_vy * keepc
        env.p_ships = env.p_ships * keepc
        env.p_owner = torch.where(gone, torch.full_like(env.p_owner, -1.0), env.p_owner)
    keep = falive & (~remove_fleet)
    keepf = keep.to(DTYPE)
    env.f_alive = keepf
    env.f_owner = env.f_owner * keepf
    env.f_x = fnx * keepf; env.f_y = fny * keepf
    env.f_angle = env.f_angle * keepf
    env.f_ships = env.f_ships * keepf

    env.step_ct = env.step_ct + 1.0

    ego_owned1 = (env.p_owner == float(ego)) & (env.p_alive > 0.5)

    out = StepOut()
    out.invalid = invalid
    out.valid = valid
    out.launches = launches
    out.launched_ships = e_shp.sum((1, 2))   # total ships launched this step (ship-weighted launch reward)
    out.owned_launch_ships = (e_shp * ego_owned0.unsqueeze(1).to(DTYPE)).sum((1, 2))   # ships launched to ALREADY-OWNED planets
    noncomet1 = env.p_is_comet < 0.5
    out.captured = (ego_owned1 & (~ego_owned0) & noncomet1).to(DTYPE).sum(1)
    out.lost = (ego_owned0 & (~ego_owned1) & (~comet0)).to(DTYPE).sum(1)
    out.captured_prod = (env.p_prod * (ego_owned1 & (~ego_owned0) & noncomet1).to(DTYPE)).sum(1)
    out.lost_prod = (env.p_prod * (ego_owned0 & (~ego_owned1) & (~comet0)).to(DTYPE)).sum(1)
    return out


def settle_n(env):
    """Per-player settle: ships (B,N) = planets+fleets per player; alive (B,N) bool.

    Vectorized over players (one-hot owner broadcast on a trailing player axis) -- a few large kernels
    instead of the per-player Python loop's ~6N small launches, which matters because this runs every
    rollout step. Bit-identical to the loop (same masks, same per-player sums)."""
    N = int(getattr(env, "n_players", 2))
    pid = torch.arange(N, device=env.p_owner.device, dtype=env.p_owner.dtype)    # (N,)
    op = (env.p_owner.unsqueeze(-1) == pid) & (env.p_alive > 0.5).unsqueeze(-1)  # (B,E,N)
    gp = (env.f_owner.unsqueeze(-1) == pid) & (env.f_alive > 0.5).unsqueeze(-1)  # (B,Fc,N)
    s = (env.p_ships.unsqueeze(-1) * op.to(DTYPE)).sum(1) + (env.f_ships.unsqueeze(-1) * gp.to(DTYPE)).sum(1)  # (B,N)
    alive = op.any(1) | gp.any(1)                                               # (B,N)
    return s, alive


def settle(env):
    """v7-compat 2p view: (s0, s1, side0_alive, side1_alive). BC/eval/probe cells use this."""
    s, alive = settle_n(env)
    return s[:, 0], s[:, 1], alive[:, 0], alive[:, 1]

## `orbit_wars_v12.checkpoint`

Model (de)serialization, invited-member discovery, and resumable train state.

Ported from notebook cells 28 (snapshot helpers + invited discovery) and 30 (save_ckpt /
save_train_state / load_train_state). The checkpoint ``config`` blob keeps UPPER keys so existing
``.pt`` files round-trip; per-member dims are read back via :func:`build_from_cfg` so league
snapshots and (smaller) invited members rebuild at their own sizes. ``ACT_FN`` defaults to
``relu`` for pre-tanh checkpoints.

In [ ]:
def _freeze_snapshot(net):
    """Move to CPU, eval mode, no grad -> a fixed sparring partner."""
    net = net.cpu().eval()
    for p in net.parameters():
        p.requires_grad_(False)
    return net


class LegacyObsAdapter(torch.nn.Module):
    """Wraps a league member saved with an OLDER observation layout so it still plays inside the
    current build: each forward DOWN-CONVERTS the current obs to the member's layout.

    Current build (F_DIM=198): 18 body = 14 v12 channels (4-ch seat one-hot ownership, ...) + 4 v13
    MOTION channels (vx,vy,ax,ay), then 30 inbound fleets x 6 (4-ch fleet seat one-hot + eta + raw
    ships). EVERY supported older member first DROPS the 4 motion channels (they never had them):
      * F_DIM_NO_MOTION (194): pre-v13 v12 -> drop motion only; 6-feat threat passes through unchanged.
      * F_DIM_BINARY_THREAT (104): member used a single self/enemy SIGN per fleet -> collapse each
        fleet's 4-ch owner one-hot to that sign (+1 self / -1 enemy / 0 empty) -> 30 x 3.
      * F_DIM_SCALAR_OWNER (101): member also used SCALAR body ownership -> additionally collapse the
        body 4-ch one-hot to the scalar code (1 self, 2 ANY enemy, 0 neutral).
    DORMANT (the net is returned unwrapped) whenever the member's F_DIM matches the current build."""
    def __init__(self, net, member_fdim):
        super().__init__()
        self.net = net
        self.member_fdim = int(member_fdim)

    def forward(self, ent, em, am, gl):
        pre = ent.shape[:-1]
        body = ent[..., :N_BODY_FEATURES_V12]                                        # drop v13 motion -> (...,14)
        thr = ent[..., N_BODY_FEATURES:].reshape(*pre, N_THREAT_FLEETS, N_THREAT_FEATS)   # 6-feat threat (...,30,6)
        if self.member_fdim == F_DIM_NO_MOTION:                                      # pre-v13 v12: threat unchanged
            return self.net(torch.cat([body, thr.reshape(*pre, N_THREAT_FEATS * N_THREAT_FLEETS)], -1), em, am, gl)
        sign = thr[..., 0:1] - thr[..., 1:4].sum(-1, keepdim=True)                   # +1 self / -1 enemy / 0 empty
        thr_old = torch.cat([sign, thr[..., 4:6]], -1).reshape(*pre, 3 * N_THREAT_FLEETS)   # (...,90)
        if self.member_fdim == F_DIM_SCALAR_OWNER:                                   # collapse body ownership too
            b7 = body[..., 7:8] + 2.0 * body[..., 8:11].sum(-1, keepdim=True).clamp(max=1.0)
            body = torch.cat([body[..., :7], b7, body[..., 11:]], -1)               # (...,11)
        return self.net(torch.cat([body, thr_old], -1), em, am, gl)


def _unwrap(net):
    return net.net if isinstance(net, LegacyObsAdapter) else net


def _wrap_if_legacy(net, member_cfg):
    """Current-format members pass through; supported older obs layouts get the down-converting
    adapter; anything else is incompatible."""
    fd = int((member_cfg or {}).get("F_DIM", F_DIM))
    if fd == F_DIM:
        return net
    if fd in (F_DIM_NO_MOTION, F_DIM_BINARY_THREAT, F_DIM_SCALAR_OWNER):
        return LegacyObsAdapter(net, fd)
    raise ValueError("incompatible member F_DIM %d (current %d)" % (fd, F_DIM))


_POLICY_KW = ("HIDDEN", "D_G", "N_RES_BLOCKS", "USE_GLU", "USE_ATTENTION", "VALUE_RES_BLOCKS",
              "ARCH", "N_HEADS", "N_TX_LAYERS", "TX_MLP_RATIO", "N_STEM_RES", "N_HEAD_RES",
              "TRUNK_SPEC", "F_DIM")


def _live(cfg, k):
    """The current build's value for a _POLICY_KW key (F_DIM is a constant; the rest are on cfg)."""
    return F_DIM if k == "F_DIM" else getattr(cfg, k)


def _cfg_from(cfg, blob_cfg, fallback=None):
    """Per-member PolicyNet dims: ckpt config > fallback (e.g. INVITE_CFG) > current build (cfg)."""
    out = {}
    for k in _POLICY_KW:
        if blob_cfg and k in blob_cfg:
            out[k] = blob_cfg[k]
        elif fallback and k in fallback:
            out[k] = fallback[k]
        else:
            out[k] = _live(cfg, k)
    # activation: missing in OLD checkpoints -> they were trained with relu, so rebuild as relu
    out["ACT_FN"] = (blob_cfg or {}).get("ACT_FN", (fallback or {}).get("ACT_FN", "relu"))
    # aux reward head: an ARCHITECTURE-changing flag absent from pre-aux blobs. Unlike the dims above it
    # must NOT fall back to the live cfg -- that would build a head the member's .pt has no weights for and
    # break the strict load. Default OFF so pre-aux snapshots (and invited members) rebuild unchanged.
    out["AUX_REWARD_PRED"] = bool((blob_cfg or {}).get("AUX_REWARD_PRED",
                                                       (fallback or {}).get("AUX_REWARD_PRED", False)))
    out["AUX_WIN_BET"] = bool((blob_cfg or {}).get("AUX_WIN_BET",
                                                   (fallback or {}).get("AUX_WIN_BET", False)))
    return out


def build_from_cfg(cfg, member_cfg):
    return PolicyNet(cfg, int(member_cfg.get("F_DIM", F_DIM)), G_DIM, member_cfg["HIDDEN"],
                     member_cfg["D_G"], PLANET_CAP, member_cfg["N_RES_BLOCKS"], member_cfg["USE_GLU"],
                     cfg.INIT_GATE_BIAS, use_attention=member_cfg["USE_ATTENTION"],
                     value_res_blocks=member_cfg["VALUE_RES_BLOCKS"], arch=member_cfg["ARCH"],
                     n_heads=member_cfg["N_HEADS"], n_tx_layers=member_cfg["N_TX_LAYERS"],
                     tx_mlp_ratio=member_cfg["TX_MLP_RATIO"], n_stem_res=member_cfg["N_STEM_RES"],
                     n_head_res=member_cfg["N_HEAD_RES"], act=member_cfg.get("ACT_FN", "relu"),
                     aux_reward_pred=member_cfg.get("AUX_REWARD_PRED", False),
                     aux_win_bet=member_cfg.get("AUX_WIN_BET", False),
                     trunk_spec=member_cfg.get("TRUNK_SPEC", ""))


def load_snapshot(cfg, path, fallback_cfg=None):
    """Load a save_ckpt() .pt into a frozen PolicyNet on CPU. Returns (net, member_cfg) -- the
    member_cfg is stored on the league member so resumable states can rebuild differently-sized
    (invited) nets."""
    blob = torch.load(path, map_location="cpu", weights_only=False)
    member_cfg = _cfg_from(cfg, blob.get("config", {}), fallback_cfg)
    net = build_from_cfg(cfg, member_cfg)
    net.load_state_dict(blob["model"])
    return _freeze_snapshot(_wrap_if_legacy(net, member_cfg)), member_cfg


_ELO_RE = re.compile(r"_elo(-?\d+)", re.IGNORECASE)


def _invited_elo(cfg, path):
    """Manually-anchored Elo of an invited ckpt: filename `_elo<NNNN>` > ckpt meta > default."""
    m = _ELO_RE.search(os.path.basename(path))
    if m:
        return float(m.group(1))
    try:
        blob = torch.load(path, map_location="cpu", weights_only=False)
        if blob.get("elo") is not None:
            return float(blob["elo"])
    except Exception:
        pass
    return float(cfg.INVITE_DEFAULT_ELO)


def discover_invited(cfg):
    """Scan the FIRST existing INVITE_DIRS folder for *.pt members; return the top INVITE_MAX
    [(path, anchored_elo)] sorted by Elo (desc)."""
    for d in cfg.INVITE_DIRS:
        if d and os.path.isdir(d):
            cands = sorted(glob.glob(os.path.join(d, "*.pt")))
            if cands:
                ranked = sorted(((p, _invited_elo(cfg, p)) for p in cands), key=lambda t: -t[1])
                print("[league] invite dir %s: %d candidate(s) -> inviting top %d"
                      % (d, len(ranked), min(cfg.INVITE_MAX, len(ranked))))
                return ranked[:cfg.INVITE_MAX]
    print("[league] no invite dir found (searched: %s)" % ", ".join(x for x in cfg.INVITE_DIRS if x))
    return []


def save_ckpt(cfg, net, path, meta=None):
    blob = {"model": net.state_dict(),
            "config": {"HIDDEN": cfg.HIDDEN, "N_RES_BLOCKS": cfg.N_RES_BLOCKS, "D_G": cfg.D_G,
                       "PLANET_CAP": PLANET_CAP, "F_DIM": F_DIM, "G_DIM": G_DIM,
                       "ARCH": cfg.ARCH, "N_TX_LAYERS": cfg.N_TX_LAYERS, "N_HEADS": cfg.N_HEADS,
                       "TX_MLP_RATIO": cfg.TX_MLP_RATIO, "N_STEM_RES": cfg.N_STEM_RES, "N_HEAD_RES": cfg.N_HEAD_RES,
                       "TRUNK_SPEC": getattr(cfg, "TRUNK_SPEC", ""),   # [blockseq] rebuild the res/attn sequence
                       "USE_GLU": cfg.USE_GLU, "USE_ATTENTION": cfg.USE_ATTENTION, "VALUE_RES_BLOCKS": cfg.VALUE_RES_BLOCKS,
                       "ACT_FN": cfg.ACT_FN, "target_actor": True,
                       # persist the ACTUAL net's head state (not cfg) -- members built without the head
                       # (invited / legacy-wrapped) must round-trip as AUX_REWARD_PRED=False.
                       "AUX_REWARD_PRED": bool(getattr(net, "aux_reward_pred", False)),
                       "AUX_WIN_BET": bool(getattr(net, "aux_win_bet", False))}}
    if meta:
        blob.update(meta)
    # league snapshots can be SMALLER than the live config (invited dims): persist the true dims
    if hasattr(net, "trunk") and hasattr(net.trunk, "layers"):
        blob["config"]["HIDDEN"] = net.h
        blob["config"]["N_TX_LAYERS"] = len(net.trunk.layers)
        blob["config"]["N_HEADS"] = net.trunk.layers[0].nh if net.trunk.layers else cfg.N_HEADS
        blob["config"]["N_STEM_RES"] = len(net.trunk.stem_a)
        blob["config"]["N_HEAD_RES"] = len(net.trunk.head_a)
    torch.save(blob, path)


def save_train_state(net, opt, league, global_iter, best_wr, path):
    """Full resumable state: weights + Adam moments + global iter + best score + dual-Elo league."""
    pa = getattr(net, "popart", None)
    torch.save({"model": net.state_dict(), "opt": opt.state_dict(), "league": league.state_dict(),
                "global_iter": int(global_iter), "best_wr": float(best_wr),
                "popart": ({"mu": pa.mu, "nu": pa.nu, "sigma": pa.sigma, "init": pa.init} if pa else None)}, path)


def load_train_state(cfg, net, opt, league, path):
    """Restore net/opt/league IN PLACE. Returns (start_iter, best_wr). Accepts a full train-state
    OR a plain save_ckpt weight (warm-start: weights only, fresh Adam/league). v7 states load too
    (missing elo4 fields default to the 2p values); invited members are re-discovered if absent."""
    blob = torch.load(path, map_location=cfg.device, weights_only=False)
    net.load_state_dict(blob["model"])
    if blob.get("popart") and getattr(net, "popart", None) is not None:
        d = blob["popart"]; net.popart.mu = d["mu"]; net.popart.nu = d["nu"]; net.popart.sigma = d["sigma"]; net.popart.init = d["init"]
    if "opt" in blob:
        opt.load_state_dict(blob["opt"])
    else:
        print("  [resume] %s has no optimizer state -> WARM-START (fresh Adam moments)" % os.path.basename(path))
    start_it = int(blob.get("global_iter", blob.get("iter", 0)))
    if "league" in blob:
        league.load_state_dict(blob["league"])
        if not any(m["kind"] == "invited" for m in league.members):   # v7 state: re-invite
            for p, aelo in discover_invited(cfg):
                try:
                    league.add_invited(p, aelo)
                except Exception as e:
                    print("  !! invite failed %s -> %r" % (p, e))
    else:  # warm-start: calibrate learner Elo vs anchors so the first snapshot is not stamped Elo 0
        rnd = (start_it // cfg.WORLD_RESAMPLE_EVERY) if cfg.WORLD_RESAMPLE_EVERY > 0 else 0
        print("  [resume] no league state -> calibrating learner Elo vs anchors")
        league.recalibrate_elo(net, make_world_pool(cfg, cfg.ELO_RECAL_ENVS, base_seed=cfg.SEED + rnd * cfg.N_WORLDS))
        league.learner_elo4 = league.learner_elo
    return start_it, float(blob.get("best_wr", -1.0))

## `orbit_wars_v12.ppo`

PPO clipped-surrogate update (notebook cell 25).

``ppo_update`` runs ``cfg.UPDATE_EPOCHS`` minibatch passes with KL early-stop, PopArt value
targets, grad clipping, and GPU-side stat accumulators flushed once at the end. The two values the
notebook mutated as globals are now explicit args: ``ent_coef`` (the annealed entropy coefficient,
was ``CUR_ENT_COEF``) and ``policy_coef`` (0 during critic warmup, was ``PPO_POLICY_COEF``).

In [ ]:
def _surrogate_logratio(cfg, logp, oldlp, n_owned=None):
    """log(pi/pi_old) for the clipped surrogate. [D1/GSPO] optionally length-normalized: divide by
    #owned planets so the ratio measures policy change PER DECISION (the joint ratio is a product
    over a variable planet-count, whose variance grows with #planets and unevenly trips the clip/KL)."""
    lr = logp - oldlp
    if cfg.RATIO_LENGTHNORM and n_owned is not None:
        lr = lr / n_owned.clamp_min(1.0)
    return lr


def policy_surrogate(cfg, logp, oldlp, adv, clip, n_owned=None):
    lr = _surrogate_logratio(cfg, logp, oldlp, n_owned)
    ratio = torch.exp(lr.clamp(-cfg.LOGRATIO_CLAMP, cfg.LOGRATIO_CLAMP))
    clip_hi = cfg.CLIP_HI if cfg.CLIP_HI > 0.0 else clip          # [D3/DAPO] asymmetric upper clip
    unclipped = ratio * adv
    clipped = torch.clamp(ratio, 1.0 - clip, 1.0 + clip_hi) * adv
    return -torch.minimum(unclipped, clipped).mean()


def ppo_update(cfg, net, opt, tb, ent_coef, policy_coef):
    N = tb["n"]; mb = cfg.MINIBATCHES; mbsize = N // mb
    s = {"total": 0.0, "policy": 0.0, "vf": 0.0, "entropy": 0.0, "sigma": 0.0,
         "approx_kl": 0.0, "clipfrac": 0.0, "grad_norm": 0.0, "auxr": 0.0}
    if mbsize == 0:
        return s
    nsteps = 0
    dev = tb["entities"].device
    _acc = {k: torch.zeros((), device=dev) for k in            # GPU-side stat accumulators: flush ONCE at
            ("total", "policy", "vf", "auxr", "entropy", "sigma", "clipfrac", "grad_norm")}   # the end (was ~6 .item()/mb)
    for epoch in range(cfg.UPDATE_EPOCHS):
        perm = torch.randperm(N, device=dev)
        ep_kl = []
        for b in range(mb):
            mi = perm[b * mbsize:(b + 1) * mbsize]
            ent = tb["entities"].index_select(0, mi)          # already on GPU (no .to(DEVICE))
            em = tb["entity_mask"].index_select(0, mi)
            am = tb["action_mask"].index_select(0, mi)
            gl = tb["globals"].index_select(0, mi)
            act_mb = tb["action"].index_select(0, mi)
            oldlp = tb["old_logp"].index_select(0, mi)
            adv = tb["advantage"].index_select(0, mi)
            ret = tb["returns"].index_select(0, mi)

            logp, entropy, value, rpred = evaluate(cfg, net, ent, em, am, gl, act_mb)
            n_owned = am.sum(1).clamp_min(1.0)                # per-owned-source mean entropy (scale-correct)
            pol = policy_surrogate(cfg, logp, oldlp, adv, cfg.CLIP, n_owned)
            vtarget = net.popart.normalize(ret) if (cfg.USE_POPART and getattr(net, "popart", None) is not None) else ret
            vloss = F.mse_loss(value, vtarget)
            ent_b = (entropy / n_owned).mean()
            pc = policy_coef   # 0 during value warmup
            loss = pc * pol + cfg.VF_COEF * vloss - (pc * ent_coef) * ent_b
            rloss = torch.zeros((), device=dev); auxr_mb = torch.zeros((), device=dev)
            if cfg.AUX_WIN_BET:                               # win-bet aux: bet b_t = tanh(rpred) in [-1,1] on the eventual
                z = tb["outcome"].index_select(0, mi)         # outcome z; payoff = b_t*z, so MINIMIZE -(b_t*z). Pressures
                bet_payoff = (torch.tanh(rpred) * z).mean()   # the trunk to discriminate winning/losing states (NOT in the
                rloss = -bet_payoff                           # advantage). payoff>0 = betting well. Trains only AFTER warmup.
                loss = loss + cfg.AUX_WIN_BET_COEF * rloss
                auxr_mb = bet_payoff.detach()                 # logged as 'bet' in the heartbeat (>0 = betting right)
            elif cfg.AUX_REWARD_PRED:                         # UNREAL aux: regress the immediate reward r_t off the
                rtgt = tb["reward"].index_select(0, mi)       # shared trunk (representation aux; NOT in the advantage).
                rloss = F.mse_loss(rpred, rtgt)               # During value warmup pc=0 nulls the non-val_ grads below,
                loss = loss + cfg.AUX_REWARD_COEF * rloss     # so (like the policy) the head trains only AFTER warmup.
                auxr_mb = rloss.detach()                      # logged as 'ar' (MSE) in the heartbeat

            opt.zero_grad()
            loss.backward()
            if pc == 0.0:                                  # critic warmup: val-only step --
                for pn, pp in net.named_parameters():      # shared-trunk vf grads otherwise
                    if not pn.startswith('val_'):          # drag the sharp BC policy around
                        pp.grad = None
            gnorm = torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.MAX_GRAD_NORM)
            if torch.isfinite(gnorm):
                opt.step()
            if cfg.USE_POPART and getattr(net, "popart", None) is not None:
                net.popart.update(ret, net.val_out)           # output-preserving stats update

            with torch.no_grad():
                lr = _surrogate_logratio(cfg, logp, oldlp, n_owned)   # match the surrogate (length-norm aware)
                lrc = lr.clamp(-cfg.LOGRATIO_CLAMP, cfg.LOGRATIO_CLAMP)
                ratio = torch.exp(lrc)
                mb_kl = ((ratio - 1.0) - lrc).mean().item()   # k3 KL estimator (>=0)
                _acc["total"] += loss.detach(); _acc["policy"] += pol.detach(); _acc["vf"] += vloss.detach()
                _acc["auxr"] += auxr_mb
                _acc["entropy"] += ent_b.detach(); _acc["sigma"] += ent_b.detach()
                s["approx_kl"] += mb_kl   # mb_kl already on CPU (the KL early-stop branch needs it)
                _acc["clipfrac"] += (torch.abs(ratio - 1.0) > cfg.CLIP).to(DTYPE).mean()
                _acc["grad_norm"] += gnorm.detach()
                ep_kl.append(mb_kl)
            nsteps += 1
            if cfg.KL_STOP_MINIBATCH and mb_kl > cfg.KL_HARD_MULT * cfg.KL_TARGET:
                break
        if cfg.KL_STOP_MINIBATCH and ep_kl and ep_kl[-1] > cfg.KL_HARD_MULT * cfg.KL_TARGET:
            break
        if cfg.KL_EARLYSTOP and ep_kl and (sum(ep_kl) / len(ep_kl)) > cfg.KL_TARGET:   # end-of-epoch (mean), not mid-minibatch
            break
    for k in _acc:                       # ONE GPU->CPU sync for all logging stats (was ~6 per minibatch)
        s[k] = _acc[k].item()
    if nsteps:
        for k in s:
            s[k] /= nsteps
    return s

## `orbit_wars_v12.grpo`

Group-Relative Policy Optimization update -- a critic-free PPO variant.

GRPO replaces PPO's *learned* value baseline with a GROUP baseline: several rollouts share one
initial world (assigned in :func:`orbit_wars_v12.rollout.collect_ppo` when ``cfg.ALGO == "grpo"``),
and each rollout's advantage is its return STANDARDIZED within its group. The advantage is therefore
precomputed in the rollout and stored in ``tb["advantage"]`` -- exactly like PPO -- so this update
just consumes it. Versus :func:`orbit_wars_v12.ppo.ppo_update` the only differences are:

  * **no value-function loss** -- the policy net's critic head is left untrained and unused;
  * an **optional KL-to-reference penalty** (DeepSeekMath ``beta`` = ``cfg.GRPO_KL_COEF``), estimated
    with Schulman's unbiased k3 estimator against a frozen reference net.

Everything else (clipped surrogate, entropy bonus, KL early-stop, grad clipping, GPU stat
accumulators) is shared with PPO. ``policy_coef`` is accepted for call-site parity but unused: there
is no critic to warm up, so the policy is always trained at full weight.

In [ ]:
def grpo_update(cfg, net, opt, tb, ent_coef, policy_coef, ref_net=None):
    """``cfg.UPDATE_EPOCHS`` minibatch passes of the clipped surrogate on the group-relative
    advantages already in ``tb`` -- NO value loss, optional KL-to-``ref_net``. The KL-to-ref value is
    logged in the ``vf`` slot (there is no critic) so the training heartbeat prints unchanged."""
    N = tb["n"]; mb = cfg.MINIBATCHES; mbsize = N // mb
    s = {"total": 0.0, "policy": 0.0, "vf": 0.0, "entropy": 0.0, "sigma": 0.0,
         "approx_kl": 0.0, "clipfrac": 0.0, "grad_norm": 0.0, "auxr": 0.0}
    if mbsize == 0:
        return s
    nsteps = 0
    dev = tb["entities"].device
    _acc = {k: torch.zeros((), device=dev) for k in            # GPU-side stat accumulators: flush ONCE
            ("total", "policy", "vf", "entropy", "sigma", "clipfrac", "grad_norm", "auxr")}   # at the end
    beta = cfg.GRPO_KL_COEF if ref_net is not None else 0.0
    for epoch in range(cfg.UPDATE_EPOCHS):
        perm = torch.randperm(N, device=dev)
        ep_kl = []
        for b in range(mb):
            mi = perm[b * mbsize:(b + 1) * mbsize]
            ent = tb["entities"].index_select(0, mi)          # already on GPU (no .to(DEVICE))
            em = tb["entity_mask"].index_select(0, mi)
            am = tb["action_mask"].index_select(0, mi)
            gl = tb["globals"].index_select(0, mi)
            act_mb = tb["action"].index_select(0, mi)
            oldlp = tb["old_logp"].index_select(0, mi)
            adv = tb["advantage"].index_select(0, mi)

            logp, entropy, value, rpred = evaluate(cfg, net, ent, em, am, gl, act_mb)   # value: GRPO_AUX_VALUE; rpred: aux head
            n_owned = am.sum(1).clamp_min(1.0)                # per-owned-source mean entropy (scale-correct)
            pol = policy_surrogate(cfg, logp, oldlp, adv, cfg.CLIP, n_owned)
            ent_b = (entropy / n_owned).mean()
            loss = pol - ent_coef * ent_b
            vloss = torch.zeros((), device=dev)
            ret_mb = None
            if cfg.GRPO_AUX_VALUE:                            # [B2/C5] aux: value head -> MC return (NOT in the advantage)
                ret_mb = tb["returns"].index_select(0, mi)
                vtgt = net.popart.normalize(ret_mb) if (cfg.USE_POPART and getattr(net, "popart", None) is not None) else ret_mb
                vloss = F.mse_loss(value, vtgt)
                loss = loss + cfg.GRPO_AUX_COEF * vloss
            auxr_mb = torch.zeros((), device=dev)
            if cfg.AUX_WIN_BET:                               # win-bet aux head (same as PPO): maximize bet*outcome
                z = tb["outcome"].index_select(0, mi)         #   bet = tanh(rpred) in [-1,1]; representation aux only
                bet_payoff = (torch.tanh(rpred) * z).mean()   #   >0 = betting in the right direction on the outcome
                loss = loss + cfg.AUX_WIN_BET_COEF * (-bet_payoff)
                auxr_mb = bet_payoff.detach()                 #   logged as 'ar' in the heartbeat
            elif cfg.AUX_REWARD_PRED:                         # UNREAL reward-prediction aux head (MSE to r_t)
                aux_mse = F.mse_loss(rpred, tb["reward"].index_select(0, mi))
                loss = loss + cfg.AUX_REWARD_COEF * aux_mse
                auxr_mb = aux_mse.detach()
            kl_ref = torch.zeros((), device=dev)
            if beta > 0.0:                                    # KL(pi || pi_ref), unbiased k3 estimator
                with torch.no_grad():
                    rlp, _, _, _ = evaluate(cfg, ref_net, ent, em, am, gl, act_mb)
                logr = (rlp - logp).clamp(-cfg.LOGRATIO_CLAMP, cfg.LOGRATIO_CLAMP)   # log(pi_ref/pi)
                kl_ref = (torch.exp(logr) - logr - 1.0).mean()                       # >= 0, grad flows via logp
                loss = loss + beta * kl_ref

            opt.zero_grad()
            loss.backward()
            gnorm = torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.MAX_GRAD_NORM)
            if torch.isfinite(gnorm):
                opt.step()
            if cfg.GRPO_AUX_VALUE and cfg.USE_POPART and getattr(net, "popart", None) is not None:
                net.popart.update(ret_mb, net.val_out)        # output-preserving stats update (aux value head)

            with torch.no_grad():
                lr = _surrogate_logratio(cfg, logp, oldlp, n_owned)   # match the surrogate (length-norm aware)
                lrc = lr.clamp(-cfg.LOGRATIO_CLAMP, cfg.LOGRATIO_CLAMP)
                ratio = torch.exp(lrc)
                mb_kl = ((ratio - 1.0) - lrc).mean().item()   # k3 KL(old||new) for the early-stop branch
                _acc["total"] += loss.detach(); _acc["policy"] += pol.detach()
                _acc["vf"] += (kl_ref + vloss).detach()       # no critic: surface KL-to-ref (+ aux vloss) in the vf slot
                _acc["entropy"] += ent_b.detach(); _acc["sigma"] += ent_b.detach()
                _acc["auxr"] += auxr_mb
                s["approx_kl"] += mb_kl
                _acc["clipfrac"] += (torch.abs(ratio - 1.0) > cfg.CLIP).to(DTYPE).mean()
                _acc["grad_norm"] += gnorm.detach()
                ep_kl.append(mb_kl)
            nsteps += 1
            if cfg.KL_STOP_MINIBATCH and mb_kl > cfg.KL_HARD_MULT * cfg.KL_TARGET:
                break
        if cfg.KL_STOP_MINIBATCH and ep_kl and ep_kl[-1] > cfg.KL_HARD_MULT * cfg.KL_TARGET:
            break
        if cfg.KL_EARLYSTOP and ep_kl and (sum(ep_kl) / len(ep_kl)) > cfg.KL_TARGET:
            break
    for k in _acc:                       # ONE GPU->CPU sync for all logging stats
        s[k] = _acc[k].item()
    if nsteps:
        for k in s:
            s[k] /= nsteps
    return s

## `orbit_wars_v12.rollout`

One PPO iteration: on-GPU rollout + v5 reward assembly + GAE (notebook cell 23).

``collect_ppo`` plays the learner (player 0) against ``seats`` (league members filling pids
1..n_players-1) across ``cfg.B`` parallel envs, with seat-swap world tiling, potential-based
reward shaping (or the legacy dense channels), terminal outcome payment, and an advantage estimate
-- GPU GAE for PPO, or a per-group standardized return for GRPO (``cfg.ALGO == "grpo"``) -- returning
a flattened transition buffer + heartbeat stats. n_players=2 is the v7 path.

In [ ]:
def _potentials(env):
    """Policy-invariant shaping potentials (Ng et al.), N-player: ship-margin and production
    share vs the STRONGEST opponent (reduces exactly to the v7 pair when n_players == 2)."""
    s, _ = settle_n(env)
    phi_ship = ship_log_t(s[:, 0]) - ship_log_t(s[:, 1:].max(1).values)
    pa = env.p_alive > 0.5
    N = int(getattr(env, "n_players", 2))
    prods = [(env.p_prod * ((env.p_owner == float(p)) & pa).to(DTYPE)).sum(1) for p in range(N)]
    pe = prods[0]
    pen = torch.stack(prods[1:], 1).max(1).values
    tot = (env.p_prod * pa.to(DTYPE)).sum(1).clamp_min(1.0)
    return phi_ship, (pe - pen) / tot


def one_sided_bet_reward(coef, bet_buf, outcome, alive):
    """ONE-SIDED confident-win bet reward (raw units, pre PPO_REWARD_SCALE). ``bet_buf`` (Tu,Bn) already holds
    relu(tanh(bet_t)) >= 0 (the confident-POSITIVE part of the bet). Pays ``coef * bet * max(z,0)`` so it
    rewards confident WINS and is EXACTLY ZERO on a draw/loss (z<=0) -- the symmetric ``bet*z`` reward is a
    passivity trap (it would pay coef/step for confidently LOSING). ``outcome`` (Bn,) z in [-1,1]; ``alive``
    (Tu,Bn) the live-step mask. Returns (Tu,Bn). Factored out so the one-sidedness is unit-testable."""
    return coef * bet_buf * outcome.clamp_min(0.0).unsqueeze(0) * alive


def _opp_baseline(cfg, Re, opp_exp):
    """[E1] subtract the league's Elo-expected outcome vs THIS opponent (a constant per iteration).
    Under one-opponent-per-iter + per-group centering this is a no-op for the centered advantage; it
    only bites once groups span opponents ([E2] GRPO_OPP_STRATA>1), which is exactly when it's needed."""
    if cfg.GRPO_OPP_BASELINE_W > 0.0 and opp_exp is not None:
        return Re - cfg.GRPO_OPP_BASELINE_W * (2.0 * float(opp_exp) - 1.0) * (cfg.WIN_BONUS / cfg.PPO_REWARD_SCALE)
    return Re


def grpo_flat_advantage(cfg, Re, outcome, G, nW, n_players=2, opp_exp=None):
    """One scalar advantage per rollout (Bn,) + ``unit_scaled`` (already ~unit-variance, so the
    buffer-level whitening should leave it alone). Selects the active GRPO advantage estimator:
      [A]  analytic binary (2p) -- closed form at a Beta-shrunk win-rate;
      [C1] rank -- within-group average rank, bounded in [-sqrt3, sqrt3], scale-INVARIANT (anti-collapse);
      [C2] CVaR -- centre on the worst-alpha tail (risk-sensitive);
      [B]  center-only / RLOO / per-group-std (the legacy ladder).
    Shared by :func:`collect_ppo` and :func:`orbit_wars_v12.grpo_diag` so the run and the proof agree."""
    Bn = Re.shape[0]
    Re = _opp_baseline(cfg, Re, opp_exp)
    Rg = Re.view(nW, G)
    if cfg.GRPO_ANALYTIC_ADV and n_players == 2 and outcome is not None:
        og = outcome.view(nW, G)
        win = (og > 0.5).to(DTYPE); loss = (og < -0.5).to(DTYPE)
        a0 = cfg.GRPO_ANALYTIC_ALPHA
        p_hat = (win.sum(1, keepdim=True) + a0) / (G + 2.0 * a0)
        Ae = (win * torch.sqrt((1.0 - p_hat) / p_hat) - loss * torch.sqrt(p_hat / (1.0 - p_hat))).reshape(Bn)
        return Ae, True
    if cfg.GRPO_RANK_ADV:                                            # [C1] van der Waerden average-rank score
        gt = (Rg.unsqueeze(2) > Rg.unsqueeze(1)).to(DTYPE).sum(2)    # peers strictly below
        eq = (Rg.unsqueeze(2) == Rg.unsqueeze(1)).to(DTYPE).sum(2)   # ties (incl. self) -> average rank
        rank = gt + 0.5 * (eq - 1.0)                                 # in [0, G-1]
        cG = ((G * G - 1.0) / 12.0) ** 0.5 + 1e-8                    # std of uniform ranks -> unit variance
        Ae = ((rank - (G - 1) / 2.0) / cG).reshape(Bn)
        return Ae, True
    if cfg.GRPO_CVAR_ALPHA > 0.0:                                    # [C2] risk-sensitive worst-alpha baseline
        k = max(1, int(round(cfg.GRPO_CVAR_ALPHA * G)))
        qa = Rg.kthvalue(k, dim=1, keepdim=True).values
        tail = (Rg <= qa).to(DTYPE)
        cvar = (Rg * tail).sum(1, keepdim=True) / tail.sum(1, keepdim=True).clamp_min(1.0)
        Ae = ((Rg - cvar) * (1.0 + cfg.GRPO_CVAR_LAMBDA * tail)).reshape(Bn)
        return Ae, False
    sum_g = Rg.sum(1, keepdim=True)                                  # [B] center (optionally RLOO) +/- per-group std
    mu = (sum_g - Rg) / (G - 1) if (cfg.GRPO_LOO and G > 1) else sum_g / G
    cen = Rg - mu
    if cfg.GRPO_STD_NORM:
        return (cen / (Rg.std(1, keepdim=True) + cfg.GRPO_ADV_EPS)).reshape(Bn), True
    return cen.reshape(Bn), False


def _grpo_sibling_advantage(cfg, Re, phiship, rew, alive, done, Tu, Bn, grp, opp_exp=None):
    """[B1] Per-step credit from shared-root siblings (VinePPO-style). The G rollouts of a world-group
    share s_0 and branch on the ego's actions; bucket them by quantized phi_ship at each step and use
    the matched-bucket mean terminal return as a per-step value V_hat(s_t) (whole-group mean when a
    bucket is thinner than GRPO_SIBLING_MIN_PEERS), then GAE(gamma,lambda) on the per-step reward."""
    dev = Re.device
    G, nW = grp
    Re = _opp_baseline(cfg, Re, opp_exp)
    Reg = Re.view(nW, G).unsqueeze(1)                              # (nW,1,G) peer (j) returns, broadcast over i
    whole = Re.view(nW, G).mean(1, keepdim=True)                   # (nW,1) group-mean fallback
    q = torch.round(phiship / cfg.GRPO_SIBLING_BUCKET_DELTA).view(Tu, nW, G)
    alg = alive.view(Tu, nW, G)
    Vhat = torch.zeros(Tu, Bn, device=dev)
    for t in range(Tu):
        same = (q[t].unsqueeze(2) == q[t].unsqueeze(1)).to(DTYPE) * alg[t].unsqueeze(1)   # (nW,G_i,G_j) alive same-bucket peers
        cnt = same.sum(2)                                          # (nW,G) bucket occupancy
        mean_bucket = (same * Reg).sum(2) / cnt.clamp_min(1.0)     # (nW,G) matched-bucket mean peer return
        use = (cnt >= cfg.GRPO_SIBLING_MIN_PEERS).to(DTYPE)
        Vhat[t] = (use * mean_bucket + (1.0 - use) * whole).reshape(Bn)
    Vhat = Vhat * alive
    adv = torch.zeros(Tu, Bn, device=dev)
    A = torch.zeros(Bn, device=dev); nextval = torch.zeros(Bn, device=dev)
    for t in range(Tu - 1, -1, -1):
        nd = 1.0 - done[t]
        delta = rew[t] + cfg.GAMMA * nextval * nd - Vhat[t]
        A = delta + cfg.GAMMA * cfg.GAE_LAMBDA * nd * A
        adv[t] = A * alive[t]
        nextval = Vhat[t]; A = A * alive[t]
    return adv.reshape(-1), (adv + Vhat).reshape(-1)


def grpo_adv_mode(cfg, n_players=2):
    """Resolve which (MUTUALLY EXCLUSIVE) advantage ESTIMATOR collect_ppo will actually use given the
    flags + their precedence, and which other estimators are enabled-but-IGNORED. The estimator flags do
    NOT stack -- turning several on silently selects ONE. Precedence (matches collect_ppo):
        phi-value[C] > sibling[B1] > analytic[A] (2p only) > rank[C1] > cvar[C2] > center/std[B].
    Orthogonal levers (clip-hi[D3], length-norm[D1], rb-gate[A1], aux-value[B2], opp-baseline[E1],
    target-rms[D4], std/LOO) COMPOSE and are not part of this resolution. Returns (active, [ignored...])."""
    if cfg.ALGO != "grpo":
        return "ppo+gae", []
    cands = [
        ("phi-value[C]", bool(cfg.GRPO_PHI_VALUE and cfg.USE_POTENTIAL_SHAPING)),
        ("sibling[B1]",  bool(cfg.GRPO_SIBLING_BASELINE)),
        ("analytic[A]",  bool(cfg.GRPO_ANALYTIC_ADV and n_players == 2)),
        ("rank[C1]",     bool(cfg.GRPO_RANK_ADV)),
        ("cvar[C2]",     bool(cfg.GRPO_CVAR_ALPHA > 0.0)),
    ]
    enabled = [name for name, on in cands if on]
    active = enabled[0] if enabled else ("drgrpo-center[B]" if not cfg.GRPO_STD_NORM else "legacy-std[B]")
    return active, enabled[1:]


def collect_ppo(cfg, env, net, world_pool, cursor, rng, seats, n_players=2, n_envs=None, opp_exp=None):
    """One PPO iteration (learner = player 0) vs `seats` -- league members filling pids
    1..n_players-1. Scripted anchors run fully on-GPU; neural members act per step (no-grad).
    On-GPU buffers + GPU GAE; reward = potential shaping + outcome. n_players=2 is the v7 path.
    stats["seat_scores"][k] = the learner's mean pairwise score vs seat k (dual-Elo signal)."""
    Bn = int(n_envs or cfg.B)
    Ec, T = env.Ec, env.T
    dev = env.dev
    specs = []
    for i, m in enumerate(seats):
        pid = i + 1
        if m.get("net") is None:
            specs.append({"pid": pid, "script": _KIND2OPP[m["kind"]]})
        else:
            specs.append({"pid": pid, "net": m["net"]})

    # ---- seat-swap render: tile W=Bn//P distinct worlds P-fold and stamp a per-seat board
    # rotation (env._rot_k) so the ego (pid 0, fixed) plays every seat's geometry in this batch.
    # The 4-fold-symmetric board makes k*90deg an EXACT seat remap; sharing each world across the
    # P seat-blocks lets the normalized advantages cancel that world's positional bias.
    P = int(n_players)
    grpo = (cfg.ALGO == "grpo")
    phi_value = grpo and cfg.GRPO_PHI_VALUE and cfg.USE_POTENTIAL_SHAPING   # [C] Phi-as-value GAE (needs the potentials)
    sib = grpo and cfg.GRPO_SIBLING_BASELINE and not phi_value             # [B1] per-step shared-root sibling baseline
    crn = grpo and cfg.GRPO_CRN                                             # [D] common-random-numbers opponent coupling
    if grpo:
        # GRPO render: tile nW distinct worlds into contiguous groups of G envs (one group = one
        # world rolled out G times) so the per-group baseline cancels that world's difficulty. Each
        # group gets a SINGLE seat geometry, cycling _SEAT_ROT across groups so the ego still trains
        # on every seat without breaking within-group homogeneity.
        G = cfg.GRPO_GROUP or cfg.GROUP_SIZE
        if G <= 0 or Bn % G != 0:
            G = Bn                                         # fallback: the whole batch is one group
        nW = Bn // G
        base = [world_pool[(cursor + i) % len(world_pool)] for i in range(nW)]
        cursor += nW
        worlds = [w for w in base for _ in range(G)]       # base[0]xG, base[1]xG, ... (contiguous groups)
        if cfg.SEAT_SWAP and P in _SEAT_ROT:
            rots = _SEAT_ROT[P]
            rk = torch.empty(Bn, 1, dtype=torch.long, device=dev)
            for g in range(nW):
                rk[g * G:(g + 1) * G, 0] = rots[g % len(rots)]   # one seat geometry per group
            env._rot_k = rk; env._rot_aug = True
        else:
            env._rot_k = None; env._rot_aug = False
        _grp = (G, nW)
    elif cfg.SEAT_SWAP and P in _SEAT_ROT and Bn % P == 0:
        W = Bn // P
        base = [world_pool[(cursor + i) % len(world_pool)] for i in range(W)]
        cursor += W
        worlds = base * P                                  # P seat-blocks of the same W worlds
        rk = torch.empty(Bn, 1, dtype=torch.long, device=dev)
        for b, kk in enumerate(_SEAT_ROT[P]):
            rk[b * W:(b + 1) * W, 0] = kk                  # block b -> seat rotation kk
        env._rot_k = rk; env._rot_aug = True
        _grp = None
    else:
        worlds = [world_pool[(cursor + i) % len(world_pool)] for i in range(Bn)]
        cursor += Bn
        env._rot_k = None; env._rot_aug = False
        _grp = None
    env.reset(worlds, n_players=n_players)

    # on-GPU rollout buffers (compute-bound: no host copies, no per-step D2H sync)
    ent_buf = torch.zeros(T, Bn, Ec, F_DIM, device=dev)
    em_buf = torch.zeros(T, Bn, Ec, device=dev); am_buf = torch.zeros(T, Bn, Ec, device=dev)
    gl_buf = torch.zeros(T, Bn, G_DIM, device=dev); act_buf = torch.zeros(T, Bn, Ec, Ec + 1, device=dev)
    oldlp_buf = torch.zeros(T, Bn, device=dev); valid_buf = torch.zeros(T, Bn, device=dev)
    rew_buf = torch.zeros(T, Bn, device=dev); val_buf = torch.zeros(T, Bn, device=dev); done_buf = torch.zeros(T, Bn, device=dev)
    phi_buf = torch.zeros(T, Bn, device=dev) if phi_value else None        # [C] Phi(s_t) per step, fed to the V_hat baseline
    phiship_buf = torch.zeros(T, Bn, device=dev) if sib else None          # [B1] phi_ship(s_t) per step, for sibling buckets
    want_bet = bool(cfg.AUX_WIN_BET and cfg.AUX_WIN_BET_REWARD)            # one-sided confident-win bet reward enabled?
    bet_buf = torch.zeros(T, Bn, device=dev) if want_bet else None         # relu(tanh(bet_t)) per step (-> reward once z known)

    active = torch.ones(Bn, device=dev); outcome = torch.zeros(Bn, device=dev)
    seat_sc = torch.zeros(len(specs), Bn, device=dev)   # pairwise score vs each seat, frozen at term
    inv_sum = torch.zeros(Bn, device=dev); lnch_sum = torch.zeros(Bn, device=dev)
    valid_sum = torch.zeros(Bn, device=dev); step_sum = torch.zeros(Bn, device=dev)
    launch_paid = torch.zeros(Bn, device=dev); milestone_prev = torch.zeros(Bn, device=dev)
    dense_paid = torch.zeros(Bn, device=dev)   # running sum of the CAPPED dense channels (USE_DENSE_GAME_CAP)
    captureR = torch.zeros(Bn, device=dev); launchR = torch.zeros(Bn, device=dev); prodMR = torch.zeros(Bn, device=dev)
    shapeR = torch.zeros(Bn, device=dev)   # integrated potential-shaping reward (heartbeat 'sh'; 0 unless shaping on)
    aliveR = torch.zeros(Bn, device=dev)   # v13: integrated per-step survival reward (heartbeat 'al')

    phi_ship_prev, phi_prod_prev = _potentials(env)   # shaping potentials at s_0

    def _scores_outcome(s_all):
        """Pairwise scores vs every seat + the scalar outcome in [-1, 1]."""
        s0 = s_all[:, 0]
        scs = []
        for sp in specs:
            sk = s_all[:, sp["pid"]]
            scs.append(torch.where(s0 > sk, torch.ones_like(s0),
                                   torch.where(s0 < sk, torch.zeros_like(s0), torch.full_like(s0, 0.5))))
        if n_players == 2 or cfg.OUTCOME_4P == "winner":
            oc = torch.sign(s0 - s_all[:, 1:].max(1).values)    # official: top score only
        else:
            oc = 2.0 * torch.stack(scs, 0).mean(0) - 1.0        # placement-linear (beat-all = +1)
        return scs, oc

    last_t = 0
    for t in range(T):
        last_t = t
        if phi_value:                       # Phi(s_t): phi_*_prev still holds the current state (s_0, or s_t from the prior step)
            phi_buf[t] = cfg.SHAPE_SHIP * phi_ship_prev + cfg.SHAPE_PROD * phi_prod_prev
        ent, em, am, gl = env_encode(env, 0)
        if want_bet:
            a_t, logp, value, rpred_t = act(cfg, net, ent, em, am, gl, greedy=False, want_rpred=True)
            bet_buf[t] = torch.relu(torch.tanh(rpred_t)) * active   # confident-POSITIVE bet, masked to live envs
        else:
            a_t, logp, value = act(cfg, net, ent, em, am, gl, greedy=False)
        ent_buf[t] = ent; em_buf[t] = em; am_buf[t] = am; gl_buf[t] = gl
        act_buf[t] = a_t; oldlp_buf[t] = logp; valid_buf[t] = active; val_buf[t] = value

        step_seats = []
        for sp in specs:
            if "net" in sp:
                with torch.no_grad():
                    oe, om, oa, og = env_encode(env, sp["pid"])
                    # [D] CRN: share this opponent's sampling noise across the G rollouts of each world-group
                    # (the ego, sampled above, keeps independent noise -> the group baseline isolates ego variance).
                    o_act, _, _ = act(cfg, sp["net"], oe, om, oa, og, greedy=False,
                                      crn_group=(_grp[0] if crn else 0))
                step_seats.append({"pid": sp["pid"], "action": o_act})
            else:
                step_seats.append({"pid": sp["pid"], "script": sp["script"]})
        out = env_step(cfg, env, a_t, seats=step_seats, step_idx=t)
        a = active
        s_all, alive_all = settle_n(env)    # ONE settle after the step -> reused by the milestone reward, the
        #                                     sibling buckets, and the terminal/outcome check below (was 2x/step)

        # --- dense per-step channels: legacy capture/prod-milestone/launch and/or Ng potential shaping.
        # Normally mutually exclusive (shaping REPLACES the legacy channels). DENSE_WITH_SHAPING lets them
        # COEXIST -- the legacy channels run AND the telescoping shaping deltas are summed on top.
        shaping_on = cfg.USE_POTENTIAL_SHAPING
        dense_on = (not shaping_on) or cfg.DENSE_WITH_SHAPING        # are the legacy capped channels live?
        if dense_on:
            s0n = s_all[:, 0]
            cap_t = a * (cfg.CAPTURE_REWARD * ((out.captured + cfg.CAPTURE_PROD_SCALE * out.captured_prod)
                                               - cfg.CAPTURE_LOSS_FRAC * (out.lost + cfg.CAPTURE_PROD_SCALE * out.lost_prod)))
            m_now = torch.where(s0n >= cfg.PROD_MILESTONE_BASE,
                                torch.floor(torch.log2(s0n.clamp_min(cfg.PROD_MILESTONE_BASE) / cfg.PROD_MILESTONE_BASE)) + 1.0,
                                torch.zeros_like(s0n))
            mr_t = a * (cfg.PROD_MILESTONE_REWARD * (m_now - milestone_prev).clamp_min(0.0))
            milestone_prev = torch.maximum(milestone_prev, m_now)
            if t < cfg.LAUNCH_WINDOW:
                _enemy_ships = out.launched_ships - out.owned_launch_ships
                _self_ships = torch.clamp(out.owned_launch_ships, max=cfg.SELF_LAUNCH_CAP)
                _disp_raw = cfg.LAUNCH_REWARD * _enemy_ships + cfg.SELF_LAUNCH_REWARD * _self_ships
                launch_t = a * torch.clamp(_disp_raw, max=cfg.LAUNCH_STEP_CAP)
            else:
                launch_t = torch.zeros_like(cap_t)
            launch_room = (cfg.LAUNCH_GAME_CAP - launch_paid).clamp_min(0.0)
            launch_t = torch.minimum(launch_t, launch_room)
            launch_paid = launch_paid + launch_t
        else:
            cap_t = torch.zeros(Bn, device=dev); mr_t = torch.zeros(Bn, device=dev); launch_t = torch.zeros(Bn, device=dev)
        if shaping_on:
            phi_ship_now, phi_prod_now = _potentials(env)
            shape_t = a * (cfg.SHAPE_SHIP * (cfg.GAMMA * phi_ship_now - phi_ship_prev)    # ship-margin shaping
                           + cfg.SHAPE_PROD * (cfg.GAMMA * phi_prod_now - phi_prod_prev))  # production-share shaping
            phi_ship_prev, phi_prod_prev = phi_ship_now, phi_prod_now
        else:
            shape_t = torch.zeros(Bn, device=dev)
        # global dense-reward attenuator: shrink capture + prod-milestone vs the terminal +-WIN/LOSS so the W/L
        # outcome dominates the return. LAUNCH is deliberately EXEMPT: it never dominates and it is the per-step
        # activity incentive that guards the passivity ratchet (scaling it down under critic-free mc/grpo
        # collapses the policy to "stop launching"). Shaping is policy-invariant -> also exempt.
        if cfg.DENSE_REWARD_SCALE != 1.0:
            cap_t = cap_t * cfg.DENSE_REWARD_SCALE; mr_t = mr_t * cfg.DENSE_REWARD_SCALE
        # shared hard GAME cap on the legacy dense sum (capture + prod-milestone + launch). When the step
        # would push the running total over DENSE_GAME_CAP, the three channels are scaled DOWN TOGETHER so
        # they share the remaining room (their per-step ratio is preserved). Net negative steps free room
        # back. Shaping (shape_t) is NOT inside the cap.
        if cfg.USE_DENSE_GAME_CAP and dense_on:
            dense_step = cap_t + mr_t + launch_t
            room = (cfg.DENSE_GAME_CAP - dense_paid).clamp_min(0.0)
            scale = torch.where(dense_step > room, room / dense_step.clamp_min(1e-8), torch.ones_like(dense_step))
            cap_t = cap_t * scale; mr_t = mr_t * scale; launch_t = launch_t * scale
            dense_paid = dense_paid + (cap_t + mr_t + launch_t)
        captureR = captureR + cap_t; prodMR = prodMR + mr_t; launchR = launchR + launch_t; shapeR = shapeR + shape_t
        dense_t = torch.zeros_like(cap_t) if phi_value else (cap_t + mr_t + launch_t + shape_t)   # [C] Phi -> value, not reward
        # v13 survival reward: a flat ALIVE_REWARD for every active (ego-alive, game-undecided) step. A
        # genuine reward (NOT shaping), so it is EXEMPT from DENSE_REWARD_SCALE and present even in phi_value
        # mode -- it densifies the halved (+-500) terminal outcome across the trajectory.
        alive_t = a * cfg.ALIVE_REWARD
        aliveR = aliveR + alive_t
        rew_buf[t] = (dense_t + alive_t) / cfg.PPO_REWARD_SCALE

        inv_sum = inv_sum + a * out.invalid
        lnch_sum = lnch_sum + a * out.launches
        valid_sum = valid_sum + a * out.valid
        step_sum = step_sum + a

        if sib:                             # [B1] ship-margin position s_{t} for matched-prefix sibling buckets
            phiship_buf[t] = ship_log_t(s_all[:, 0]) - ship_log_t(s_all[:, 1:].max(1).values)
        step_now = env.step_ct
        n_alive = alive_all.to(DTYPE).sum(1)
        term = (step_now >= float(T - 2)) | (n_alive <= 1.0) | (~alive_all[:, 0])   # ego dead -> decided
        done_buf[t] = ((active > 0.5) & term).to(DTYPE)        # terminal on cap OR death (no spurious bootstrap)
        newly = (active > 0.5) & term
        scs, oc = _scores_outcome(s_all)
        for k in range(len(specs)):
            seat_sc[k] = torch.where(newly, scs[k], seat_sc[k])
        outcome = torch.where(newly, oc, outcome)
        active = torch.where(term, torch.zeros_like(active), active)
        # NOTE: no per-step active.sum().item() early-break -> that is a CUDA sync. Finished envs are masked.

    s_all, alive_all = settle_n(env)
    scs, oc = _scores_outcome(s_all)
    still = active > 0.5
    for k in range(len(specs)):
        seat_sc[k] = torch.where(still, scs[k], seat_sc[k])
    outcome = torch.where(still, oc, outcome)

    bootstrap = torch.zeros(Bn, device=dev)
    if cfg.ALGO == "ppo":                                  # only PPO+GAE bootstraps from the critic (grpo/mc are critic-free)
        with torch.no_grad():
            fe, fm, fa_, fg = env_encode(env, 0)
            _, _, vT = act(cfg, net, fe, fm, fa_, fg, greedy=False)
            bootstrap = vT * active

    Tu = last_t + 1
    rew = rew_buf[:Tu].clone(); val = val_buf[:Tu]; done = done_buf[:Tu]; alive = valid_buf[:Tu]
    length = alive.sum(0)
    # ONE-SIDED confident-win bet reward: pay coef * relu(tanh(bet_t)) * max(z,0), rectified to the WINNING
    # side so it can never pay for confidently losing (the two-sided bet*z reward is a passivity trap). The
    # bet is DETACHED (a reward, not a loss); it flows through GAE / the per-step GRPO return like any reward.
    r_win_bet_log = 0.0
    if want_bet:
        bet_r = one_sided_bet_reward(cfg.AUX_WIN_BET_REWARD_COEF, bet_buf[:Tu], outcome, alive)
        rew = rew + bet_r / cfg.PPO_REWARD_SCALE                                   # raw units / scale, like the other channels
        r_win_bet_log = bet_r.sum(0).mean().item()                                # heartbeat 'wb' (raw units, like R[o c p ln])
    len_eff = (length - cfg.DECAY_START_STEP).clamp_min(0.0)
    if cfg.USE_WIN_POOL:
        # WIN POOL: a win pays the REST of the WIN_POOL after the per-step ALIVE_REWARD drip already paid
        # (-> a won game totals exactly WIN_POOL, undecayed; discounting still rewards winning fast). The
        # loss side keeps the decayed LOSS_PENALTY. Clawback below cancels the drip on non-wins.
        win_val = (cfg.WIN_POOL - cfg.ALIVE_REWARD * length).clamp_min(0.0)
    else:
        win_val = torch.pow(torch.full_like(outcome, cfg.WIN_DECAY), len_eff) * cfg.WIN_BONUS
    loss_val = torch.pow(torch.full_like(outcome, cfg.LOSS_DECAY), len_eff) * cfg.LOSS_PENALTY
    # fractional outcomes (4p placement mode) scale the outcome magnitudes linearly;
    # in 2p outcome is exactly {-1, 0, +1} -> identical to the v7 payment
    Ot = torch.where(outcome > 0.0, outcome * win_val, outcome * loss_val)
    if cfg.USE_WIN_POOL:
        # claw back the survival drip (ALIVE_REWARD*len) on NON-winning outcomes so a long-surviving loss
        # can never net positive (the passivity ratchet): a loser then nets exactly -decayed(LOSS_PENALTY).
        Ot = Ot - (outcome < 0.0).to(Ot.dtype) * cfg.ALIVE_REWARD * length
    r_outcome_log = Ot.mean().item()
    if dense_on:
        disp_cashout = torch.clamp(cfg.LAUNCH_WINDOW - length, min=0.0) * cfg.LAUNCH_STEP_CAP * (outcome > 0.0).to(outcome.dtype)
        cash_room = ((cfg.DENSE_GAME_CAP - dense_paid) if cfg.USE_DENSE_GAME_CAP
                     else (cfg.LAUNCH_GAME_CAP - launch_paid)).clamp_min(0.0)
        disp_cashout = torch.minimum(disp_cashout, cash_room)
        launchR = launchR + disp_cashout
        Ot = (Ot + disp_cashout) / cfg.PPO_REWARD_SCALE
    else:
        Ot = Ot / cfg.PPO_REWARD_SCALE
    last_idx = (length - 1.0).clamp_min(0.0).long().unsqueeze(0)
    rew.scatter_add_(0, last_idx, Ot.unsqueeze(0))

    Re = (Ot if cfg.GRPO_OUTCOME_ONLY else (rew * alive).sum(0)) if grpo else None   # (Bn,) per-env return
    adv = None; unit_scaled = False
    if grpo and sib:
        # ---- [B1] per-step credit from matched-prefix shared-root siblings (critic-free GAE) ----
        adv_full, ret_full = _grpo_sibling_advantage(cfg, Re, phiship_buf[:Tu], rew, alive, done, Tu, Bn, _grp, opp_exp)
    elif grpo and not phi_value:
        # ---- flat group-relative advantage: one scalar per rollout, broadcast over its steps ----
        G, nW = _grp
        Ae, unit_scaled = grpo_flat_advantage(cfg, Re, outcome, G, nW, n_players, opp_exp)   # [A]/[B]/[C1]/[C2]/[E1]
        adv = Ae.unsqueeze(0).expand(Tu, Bn) * alive
        ret_full = Re.unsqueeze(0).expand(Tu, Bn).reshape(-1)        # placeholder (no critic target)
        adv_full = adv.reshape(-1)
    elif grpo and phi_value:
        # ---- [C] per-step GAE with a hand-built value V_hat(s_t) = mu_g + a*(Phi(s_t)-Phi(s_0)).
        # The shared root mu_g (= group baseline) cancels world difficulty; a*dPhi propagates the
        # sparse outcome to the moves that improved position. a is the OLS slope of outcome on terminal dPhi.
        G, nW = _grp
        Re = _opp_baseline(cfg, Re, opp_exp)
        Rg = Re.view(nW, G); sum_g = Rg.sum(1, keepdim=True)
        mu_g = (sum_g - Rg) / (G - 1) if (cfg.GRPO_LOO and G > 1) else (sum_g / G).expand(nW, G)
        mu = mu_g.reshape(Bn)                                       # V_hat(s_0), one baseline per rollout
        phi = phi_buf[:Tu]; phi0 = phi[0]
        dphi = phi - phi0.unsqueeze(0)                              # (Tu,Bn) positional change vs the start state
        li = (length - 1.0).clamp_min(0.0).long().unsqueeze(0)
        dphi_T = dphi.gather(0, li).squeeze(0)                       # (Bn,) terminal positional change
        a_slope = ((((Re - mu) * dphi_T).sum()) / (dphi_T * dphi_T).sum().clamp_min(1e-6)).clamp(0.0, cfg.GRPO_PHI_A_MAX)
        Vhat = (mu.unsqueeze(0) + a_slope * dphi) * alive           # (Tu,Bn)
        adv = torch.zeros(Tu, Bn, device=dev)
        A = torch.zeros(Bn, device=dev)
        nextval = (mu + a_slope * (phi[Tu - 1] - phi0)) * active     # bootstrap from V_hat (active envs only)
        for t in range(Tu - 1, -1, -1):
            al = alive[t]; notdone = 1.0 - done[t]
            delta = rew[t] + cfg.GAMMA * nextval * notdone - Vhat[t]
            A = delta + cfg.GAMMA * cfg.GAE_LAMBDA * notdone * A
            adv[t] = A * al
            nextval = Vhat[t]; A = A * al
        ret_full = (adv + Vhat).reshape(-1); adv_full = adv.reshape(-1)
    elif cfg.ALGO == "mc":
        # ---- vanilla Monte-Carlo: advantage = discounted return-to-go G_t (no GAE, no critic, no value
        # loss). Whitened below (center + global scale); consumed by the critic-free clipped-surrogate update. ----
        ret_buf = torch.zeros(Tu, Bn, device=dev)
        G = torch.zeros(Bn, device=dev)
        for t in range(Tu - 1, -1, -1):
            G = rew[t] + cfg.GAMMA * G * (1.0 - done[t])   # reset at episode end (done)
            ret_buf[t] = G * alive[t]
        adv_full = ret_buf.reshape(-1); ret_full = ret_buf.reshape(-1)
    else:
        # GAE on GPU (no .cpu(), no synchronize)
        adv = torch.zeros(Tu, Bn, device=dev)
        A = torch.zeros(Bn, device=dev); nextval = bootstrap.clone()
        for t in range(Tu - 1, -1, -1):
            al = alive[t]; notdone = 1.0 - done[t]
            delta = rew[t] + cfg.GAMMA * nextval * notdone - val[t]
            A = delta + cfg.GAMMA * cfg.GAE_LAMBDA * notdone * A
            adv[t] = A * al
            nextval = val[t]
            A = A * al
        ret_full = (adv + val).reshape(-1); adv_full = adv.reshape(-1)
    mean_return_log = (rew.sum(0).mean().item()) * cfg.PPO_REWARD_SCALE

    keep = valid_buf[:Tu].reshape(-1).nonzero().squeeze(-1)
    def sel(x): return x.index_select(0, keep)
    tb = {}
    tb["entities"] = sel(ent_buf[:Tu].reshape(Tu * Bn, Ec, F_DIM))
    tb["entity_mask"] = sel(em_buf[:Tu].reshape(Tu * Bn, Ec))
    tb["action_mask"] = sel(am_buf[:Tu].reshape(Tu * Bn, Ec))
    tb["globals"] = sel(gl_buf[:Tu].reshape(Tu * Bn, G_DIM))
    tb["action"] = sel(act_buf[:Tu].reshape(Tu * Bn, Ec, Ec + 1))
    tb["old_logp"] = sel(oldlp_buf[:Tu].reshape(Tu * Bn))
    adv_s = sel(adv_full)
    # An already ~unit-variance path ([B] per-group std, [C1] rank, [A] analytic) is left as-is. Every
    # other path -- center-only [B], CVaR [C2], phi-value [C], sibling [B1], explicit GRPO_WHITEN, or PPO --
    # gets ONE GLOBAL scale, preserving the cross-group p(1-p) weighting (docs/grpo_v12.tex [B]). [Tier0/D4]
    # ADV_TARGET_RMS pins that scale to a constant so league difficulty can't drift the effective LR via Adam.
    if grpo and unit_scaled and not cfg.GRPO_WHITEN:
        tb["advantage"] = adv_s
    elif grpo and cfg.ADV_TARGET_RMS > 0.0:
        rms = adv_s.pow(2).mean().sqrt().clamp_min(1e-8)
        tb["advantage"] = (adv_s - adv_s.mean()) * (cfg.ADV_TARGET_RMS / rms)
    else:
        tb["advantage"] = (adv_s - adv_s.mean()) / (adv_s.std() + 1e-8)
    tb["returns"] = sel(ret_full)
    if cfg.AUX_REWARD_PRED:                                 # aux reward-prediction target: the per-step reward r_t
        tb["reward"] = sel(rew.reshape(-1))                # (the same signal GAE consumes; includes the terminal payment)
    if cfg.AUX_WIN_BET:                                     # win-bet target: the episode outcome z in [-1,1], broadcast to
        tb["outcome"] = sel(outcome.unsqueeze(0).expand(Tu, Bn).reshape(-1))   # every step (the bet b_t is scored b_t*z)
    tb["n"] = keep.shape[0]
    del ent_buf, em_buf, am_buf, gl_buf, act_buf, oldlp_buf, val_buf, rew_buf, done_buf, valid_buf, adv_full, ret_full

    step_total = step_sum.sum().item()
    stats = {
        "mean_return": mean_return_log,
        "mean_len": step_total / Bn,
        "win_rate": (seat_sc.min(0).values >= 0.999).to(DTYPE).mean().item(),   # beat EVERY opponent
        "score": seat_sc.mean().item(),
        "seat_scores": [float(seat_sc[k].mean().item()) for k in range(len(specs))],
        "fmt": n_players,
        "inv_per_step": (inv_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "lnch_per_step": (lnch_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "valid_per_step": (valid_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "transitions": tb["n"],
        "r_outcome": r_outcome_log,
        "r_capture": captureR.mean().item(),      # legacy capture channel (0 under pure shaping)
        "r_launch": launchR.mean().item(),
        "r_milestone": prodMR.mean().item(),      # legacy prod-milestone channel (0 under pure shaping)
        "r_shape": shapeR.mean().item(),          # integrated potential shaping (ship-margin + prod-share)
        "r_alive": aliveR.mean().item(),          # v13: integrated per-step survival reward
        "r_win_bet": r_win_bet_log,               # one-sided confident-win bet reward (0 unless AUX_WIN_BET_REWARD)
        # advantage-health diagnostics (anti-collapse proof): the FINAL advantage fed to the surrogate.
        # The legacy per-group-std path can explode on near-degenerate groups; rank/Dr.GRPO stay bounded.
        "adv_absmax": float(tb["advantage"].abs().max().item()) if tb["n"] > 0 else 0.0,
        "adv_std": float(tb["advantage"].std().item()) if tb["n"] > 1 else 0.0,
    }
    return tb, stats, cursor

## `orbit_wars_v12.bc`

Behaviour-cloning warm-start (notebook cell 32).

Clones the MEDIUM scripted rule directly into the gated-alloc heads so PPO starts from a
committing, aiming policy (the from-scratch bootstrap is the proven passivity trap). Expert =
medium (nearest non-owned planet, lead-aim) expressed as one-hot WHERE rows + fire; states are
visited by the clone itself vs a mixed scripted-opponent schedule; loss = BCE(gate) + CE(masked
WHERE) on firing sources. Saves ``cfg.BC_CKPT_PATH``.

In [ ]:
def medium_targets_ego(env):
    """The medium rule for EGO (owner 0): (fire*, tgt) both (B,Ec)."""
    half = torch.floor(env.p_ships / 2.0)
    base = (env.p_owner == 0.0) & (env.p_alive > 0.5) & (half >= 20.0)
    sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)
    tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)
    vt = (env.p_alive > 0.5) & (env.p_owner != 0.0)
    d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)
    dmask = torch.where(vt.unsqueeze(1), d, torch.full_like(d, BIG))
    bestd, tgt = dmask.min(2)
    fire = base & (bestd < BIG * 0.5)
    return fire, tgt


def bc_expert_action(env):
    """(B,Ec,Ec+1) gated-alloc action bundle of the expert: one-hot WHERE rows + fire."""
    fire, tgt = medium_targets_ego(env)
    B_, Ec = fire.shape
    alloc = torch.zeros(B_, Ec, Ec, dtype=DTYPE, device=env.dev)
    alloc.scatter_(2, tgt.unsqueeze(-1), 1.0)
    return torch.cat([alloc, fire.to(DTYPE).unsqueeze(-1)], -1), fire, tgt


def bc_pretrain(cfg, net=None, rounds=None, epochs=None, lr=None, log=True):
    """Supervised clone of the medium rule. Returns the net; saves cfg.BC_CKPT_PATH."""
    rounds = cfg.BC_ROUNDS if rounds is None else rounds
    epochs = cfg.BC_EPOCHS if epochs is None else epochs
    lr = cfg.BC_LR if lr is None else lr
    net = net or build_policy(cfg)
    try:
        opt = torch.optim.Adam(net.parameters(), lr=lr, eps=cfg.ADAM_EPS, fused=(cfg.device.type == 'cuda'))
    except Exception:
        opt = torch.optim.Adam(net.parameters(), lr=lr, eps=cfg.ADAM_EPS)
    env = GpuEnv(cfg)
    pool = make_world_pool(cfg, max(cfg.B, 256), base_seed=cfg.SEED + 900001)
    opps = [1, 0, 3, 1, 4, 1, 3, 0, 1, 3, 4, 1]          # starter-heavy mix, incl. medium/greedy
    rng = random.Random(cfg.SEED + 31337)
    for r in range(rounds):
        worlds = [pool[rng.randrange(len(pool))] for _ in range(cfg.B)]
        env.reset(worlds)
        opp = opps[r % len(opps)]
        ents, ems, ams, gls, fires, tgts, actives = [], [], [], [], [], [], []
        active = torch.ones(cfg.B, device=cfg.device)
        with torch.no_grad():
            for t in range(env.T):
                ent, em, am, gl = env_encode(env, 0)
                a_t, fire, tgt = bc_expert_action(env)
                # stash on CPU: a full on-GPU rollout of encoded states is ~0.6 GB
                ents.append(ent.cpu()); ems.append(em.cpu()); ams.append(am.cpu()); gls.append(gl.cpu())
                fires.append(fire.cpu()); tgts.append(tgt.cpu()); actives.append(active.cpu().clone())
                env_step(cfg, env, a_t, opp, None, step_idx=t)
                s0, s1, side0, side1 = settle(env)
                active = active * (side0 & side1).to(DTYPE)
        ent = torch.cat(ents); em = torch.cat(ems); am = torch.cat(ams); gl = torch.cat(gls)
        fire = torch.cat(fires); tgt = torch.cat(tgts); act_m = torch.cat(actives)
        keep = act_m > 0.5                                    # drop post-terminal states
        ent, em, am, gl, fire, tgt = ent[keep], em[keep], am[keep], gl[keep], fire[keep], tgt[keep]
        N = ent.shape[0]
        mbs = max(8, cfg.BC_MINIBATCH // PLANET_CAP)          # boards per minibatch (~85 @ 4096/48)
        stats = [0.0, 0.0, 0.0, 0]
        for ep in range(epochs):
            perm = torch.randperm(N)
            for b in range(0, N, mbs):
                mi = perm[b:b + mbs]
                ent_d, em_d, am_d, gl_d = (x[mi].to(cfg.device) for x in (ent, em, am, gl))
                fire_d, tgt_d = fire[mi].to(cfg.device), tgt[mi].to(cfg.device)
                dist, _, _ = _make_dist(cfg, net, ent_d, em_d, am_d, gl_d)
                own = am_d > 0.5
                f = fire_d.to(DTYPE)
                p = dist.p                                     # smooth fire-prob (post-fix)
                bce = -(f * torch.log(p.clamp_min(1e-8)) + (1 - f) * torch.log((1 - p).clamp_min(1e-8)))
                bce = (bce * own.to(DTYPE)).sum() / own.sum().clamp_min(1)
                lsm = torch.log_softmax(dist.glogits, -1)      # masked WHERE logits (alive+reach+no-self)
                ce_all = -lsm.gather(2, tgt_d.unsqueeze(-1)).squeeze(-1)
                ok = own & fire_d & (dist.glogits.gather(2, tgt_d.unsqueeze(-1)).squeeze(-1) > -1e8)
                ce = (ce_all * ok.to(DTYPE)).sum() / ok.sum().clamp_min(1)
                loss = bce + ce
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), cfg.MAX_GRAD_NORM)
                opt.step()
                with torch.no_grad():
                    hit = ((lsm.argmax(-1) == tgt_d) & ok).sum() / ok.sum().clamp_min(1)
                    gate_acc = (((p > 0.5) == fire_d) & own).sum() / own.sum().clamp_min(1)
                stats[0] += loss.item(); stats[1] += hit.item(); stats[2] += gate_acc.item(); stats[3] += 1
        if log:
            print("BC round %2d/%d opp=%d N=%6d | loss %.4f | dest-acc %.3f gate-acc %.3f"
                  % (r + 1, rounds, opp, N, stats[0] / stats[3], stats[1] / stats[3], stats[2] / stats[3]), flush=True)
    save_ckpt(cfg, net, cfg.BC_CKPT_PATH, meta={"iter": 0, "bc": True})
    print("saved BC init ->", cfg.BC_CKPT_PATH)
    return net

## `orbit_wars_v12.league`

Dual-Elo self-play league + scripted/neural evaluation helpers (notebook cells 28 + 29).

Every member carries a 2p (``elo``) and a 4p (``elo4``) rating; the learner is grounded on a
ROLLING self-anchor (its own frozen high-water snapshot) so the rating never clamps on saturated
scripts. Scripted/invited members are rating-anchored; learner snapshots are PFSP-sampled. Recals
BLEND into the live ratings (first=raw) and run a MEASURED eviction of mastered scripts at start
and at every recalibration (v12). All hyperparameters come from ``self.cfg``.

In [ ]:
# ---- fixed FFA trio / calibration kinds ----------------------------------------------------
GAUNTLET4_SEATS = ("starter", "greedy", "intermediate")   # fixed FFA trio for the 4p gauntlet
_CAL4_KINDS = ("random", "starter", "intermediate", "greedy", "medium")

# ---- rolling self-anchor recal constants + primitives --------------------------------------
SELF_KIND = "selfanchor"
PROMOTE_S2 = 0.80    # 2p seat-avg score vs the top anchor required to promote
PROMOTE_WR4 = 0.40   # 4p 1st-place win-rate vs the top anchor required to promote
ANCHOR_KEEP = 8      # max self-anchors kept resident (top rungs)
GREEDY_WATCHDOG = 0.85   # alarm if learner 2p score vs greedy falls below this
RECAL_CLAMP = (0.02, 0.98)


def eval_gauntlet(cfg, net, worlds, anchors=("starter", "intermediate", "greedy"), n_envs=None):
    """Fixed-opponent 2p score for best-ckpt selection (greedy, deployment-faithful op set)."""
    n = n_envs or cfg.ELO_RECAL_ENVS
    return sum(_eval_score(cfg, net, a, worlds, n) for a in anchors) / len(anchors)


def _eval_score(cfg, net, anchor_kind, worlds, n_envs):
    """No-grad mean learner-score (win + 0.5*draw) of `net` vs a scripted anchor on `worlds` (greedy, 2p)."""
    dev = cfg.device
    env = GpuEnv(cfg)
    env.reset([worlds[i % len(worlds)] for i in range(n_envs)])
    active = torch.ones(n_envs, device=dev); score = torch.zeros(n_envs, device=dev)
    def _sc(s0, s1):
        return torch.where(s0 > s1, torch.ones_like(s0), torch.where(s0 < s1, torch.zeros_like(s0), torch.full_like(s0, 0.5)))
    with torch.no_grad():
        for _t in range(env.T):
            ent, em, am, gl = env_encode(env, 0)
            a_t, _, _ = act(cfg, net, ent, em, am, gl, greedy=True)
            env_step(cfg, env, a_t, _OPP_BY_KIND[anchor_kind], None, step_idx=_t)
            s0, s1, a0, a1 = settle(env)
            term = (env.step_ct >= float(env.T - 2)) | (~(a0 & a1))
            newly = (active > 0.5) & term
            score = torch.where(newly, _sc(s0, s1), score)
            active = torch.where(term, torch.zeros_like(active), active)
            if active.sum().item() == 0:
                break
        s0, s1, _, _ = settle(env)
        score = torch.where(active > 0.5, _sc(s0, s1), score)
    return float(score.mean().item())


def _eval_score4(cfg, net, trio, worlds, n_envs):
    """No-grad mean PAIRWISE score of greedy `net` (seat 0) vs 3 scripted anchors in a 4p FFA."""
    dev = cfg.device
    env = GpuEnv(cfg)
    env.reset([worlds[i % len(worlds)] for i in range(n_envs)], n_players=4)
    seats = [{"pid": p + 1, "script": _OPP_BY_KIND[k]} for p, k in enumerate(trio)]
    active = torch.ones(n_envs, device=dev); score = torch.zeros(n_envs, device=dev)
    def _pw(s_all):
        s0 = s_all[:, 0:1].expand_as(s_all[:, 1:]); rest = s_all[:, 1:]
        one = torch.ones_like(rest)
        return torch.where(s0 > rest, one, torch.where(s0 < rest, 0.0 * one, 0.5 * one)).mean(1)
    with torch.no_grad():
        for _t in range(env.T):
            ent, em, am, gl = env_encode(env, 0)
            a_t, _, _ = act(cfg, net, ent, em, am, gl, greedy=True)
            env_step(cfg, env, a_t, seats=seats, step_idx=_t)
            s_all, alive_all = settle_n(env)
            n_alive = alive_all.to(DTYPE).sum(1)
            term = (env.step_ct >= float(env.T - 2)) | (n_alive <= 1.0) | (~alive_all[:, 0])
            newly = (active > 0.5) & term
            score = torch.where(newly, _pw(s_all), score)
            active = torch.where(term, torch.zeros_like(active), active)
            if active.sum().item() == 0:
                break
        s_all, _ = settle_n(env)
        score = torch.where(active > 0.5, _pw(s_all), score)
    return float(score.mean().item())


def _eval_winrate4(cfg, net, kind, worlds, n_envs):
    """4p FFA 1st-place (outright-win) rate of greedy `net` (seat 0) vs THREE copies of
    scripted `kind` (seats 1-3). Baseline 0.25 when all four are equal."""
    dev = cfg.device
    env = GpuEnv(cfg)
    env.reset([worlds[i % len(worlds)] for i in range(n_envs)], n_players=4)
    seats = [{"pid": p + 1, "script": _OPP_BY_KIND[kind]} for p in range(3)]
    active = torch.ones(n_envs, device=dev); win = torch.zeros(n_envs, device=dev)
    def _win(s_all):
        s0 = s_all[:, 0:1]; rest = s_all[:, 1:]
        return (s0 > rest).all(1).to(s_all.dtype)
    with torch.no_grad():
        for _t in range(env.T):
            ent, em, am, gl = env_encode(env, 0)
            a_t, _, _ = act(cfg, net, ent, em, am, gl, greedy=True)
            env_step(cfg, env, a_t, seats=seats, step_idx=_t)
            s_all, alive_all = settle_n(env)
            n_alive = alive_all.to(DTYPE).sum(1)
            term = (env.step_ct >= float(env.T - 2)) | (n_alive <= 1.0) | (~alive_all[:, 0])
            newly = (active > 0.5) & term
            win = torch.where(newly, _win(s_all), win)
            active = torch.where(term, torch.zeros_like(active), active)
            if active.sum().item() == 0:
                break
        s_all, _ = settle_n(env)
        win = torch.where(active > 0.5, _win(s_all), win)
    return float(win.mean().item())


def _eval_winrate4_net(cfg, net, opp_net, worlds, n_envs):
    """4p FFA 1st-place (outright-win) rate of greedy `net` (seat 0) vs THREE copies of the NEURAL
    `opp_net` (seats 1-3) -- the neural twin of `_eval_winrate4`, so league players are eviction-tested
    on the identical standard as scripted anchors. Baseline 0.25 when all four are equal."""
    dev = cfg.device
    env = GpuEnv(cfg)
    env.reset([worlds[i % len(worlds)] for i in range(n_envs)], n_players=4)
    active = torch.ones(n_envs, device=dev); win = torch.zeros(n_envs, device=dev)
    def _win(s_all):
        s0 = s_all[:, 0:1]; rest = s_all[:, 1:]
        return (s0 > rest).all(1).to(s_all.dtype)
    with torch.no_grad():
        for _t in range(env.T):
            ent, em, am, gl = env_encode(env, 0)
            a_t, _, _ = act(cfg, net, ent, em, am, gl, greedy=True)
            step_seats = []
            for pid in (1, 2, 3):
                oe, om, oa, og = env_encode(env, pid)
                o_act, _, _ = act(cfg, opp_net, oe, om, oa, og, greedy=True)
                step_seats.append({"pid": pid, "action": o_act})
            env_step(cfg, env, a_t, seats=step_seats, step_idx=_t)
            s_all, alive_all = settle_n(env)
            n_alive = alive_all.to(DTYPE).sum(1)
            term = (env.step_ct >= float(env.T - 2)) | (n_alive <= 1.0) | (~alive_all[:, 0])
            newly = (active > 0.5) & term
            win = torch.where(newly, _win(s_all), win)
            active = torch.where(term, torch.zeros_like(active), active)
            if active.sum().item() == 0:
                break
        s_all, _ = settle_n(env)
        win = torch.where(active > 0.5, _win(s_all), win)
    return float(win.mean().item())


def _eval_pair_scores4(cfg, net, trio, worlds, n_envs):
    """4p FFA with greedy `net` at seat 0 and the scripted `trio` at seats 1..3. Returns
    {(i, j): mean pairwise score of seat i vs seat j} for the pairs among seats 1..3 only
    (the ego is just part of the arena), frozen per env at termination."""
    dev = cfg.device
    env = GpuEnv(cfg)
    env.reset([worlds[i % len(worlds)] for i in range(n_envs)], n_players=4)
    seats = [{"pid": p + 1, "script": _OPP_BY_KIND[k]} for p, k in enumerate(trio)]
    pairs = ((1, 2), (1, 3), (2, 3))
    active = torch.ones(n_envs, device=dev)
    psc = torch.zeros(len(pairs), n_envs, device=dev)
    def _ps(s_all):
        out = []
        for (i, j) in pairs:
            si, sj = s_all[:, i], s_all[:, j]
            out.append(torch.where(si > sj, torch.ones_like(si),
                                   torch.where(si < sj, torch.zeros_like(si),
                                               torch.full_like(si, 0.5))))
        return out
    with torch.no_grad():
        for _t in range(env.T):
            ent, em, am, gl = env_encode(env, 0)
            a_t, _, _ = act(cfg, net, ent, em, am, gl, greedy=True)
            env_step(cfg, env, a_t, seats=seats, step_idx=_t)
            s_all, alive_all = settle_n(env)
            n_alive = alive_all.to(DTYPE).sum(1)
            term = (env.step_ct >= float(env.T - 2)) | (n_alive <= 1.0) | (~alive_all[:, 0])
            newly = (active > 0.5) & term
            for k, p in enumerate(_ps(s_all)):
                psc[k] = torch.where(newly, p, psc[k])
            active = torch.where(term, torch.zeros_like(active), active)
            if active.sum().item() == 0:
                break
        s_all, _ = settle_n(env)
        still = active > 0.5
        for k, p in enumerate(_ps(s_all)):
            psc[k] = torch.where(still, p, psc[k])
    return {pairs[k]: float(psc[k].mean().item()) for k in range(len(pairs))}


def _js_div(p, q):
    m = 0.5 * (p + q); eps = 1e-9
    kl = lambda a, b: (a * (torch.log(a + eps) - torch.log(b + eps))).sum()
    return float(0.5 * kl(p, m) + 0.5 * kl(q, m))


def _recal_link(cfg, base, s):
    """Elo of an agent scoring `s` (in [0,1], pairwise-style 0.5=even) vs a reference at `base`."""
    s = min(max(s, RECAL_CLAMP[0]), RECAL_CLAMP[1])
    return base + cfg.ELO_SCALE * math.log10(s / (1.0 - s))


def _score2_vs(cfg, net, opp_net, worlds, n):
    """Seat-AVERAGED greedy 2p score of `net` vs frozen `opp_net` (kills the seat-0 bias)."""
    dev = cfg.device
    def _one(a_net, b_net):
        env = GpuEnv(cfg)
        env.reset([worlds[i % len(worlds)] for i in range(n)], n_players=2)
        active = torch.ones(n, device=dev); score = torch.zeros(n, device=dev)
        def _sc(s0, s1):
            return torch.where(s0 > s1, torch.ones_like(s0),
                               torch.where(s0 < s1, torch.zeros_like(s0), 0.5 * torch.ones_like(s0)))
        with torch.no_grad():
            for t in range(env.T):
                e, m, a, g = env_encode(env, 0); a0, _, _ = act(cfg, a_net, e, m, a, g, greedy=True)
                oe, om, oa, og = env_encode(env, 1); a1, _, _ = act(cfg, b_net, oe, om, oa, og, greedy=True)
                env_step(cfg, env, a0, seats=[{"pid": 1, "action": a1}], step_idx=t)
                s0, s1, al0, al1 = settle(env)
                term = (env.step_ct >= float(env.T - 2)) | (~(al0 & al1))
                newly = (active > 0.5) & term
                score = torch.where(newly, _sc(s0, s1), score)
                active = torch.where(term, torch.zeros_like(active), active)
                if active.sum().item() == 0:
                    break
            s0, s1, _, _ = settle(env)
            score = torch.where(active > 0.5, _sc(s0, s1), score)
        return float(score.mean().item())
    ab = _one(net, opp_net)
    ba = _one(opp_net, net)
    return 0.5 * (ab + (1.0 - ba))


def _eval4_vs(cfg, net, anc_net, worlds, n):
    """4p FFA: seat0=net, seat1=anchor, seats 2,3 = greedy/intermediate scripts.
    Returns (winrate_1st, pairwise_vs_anchor) -- gate on the win-rate, rate the chain on pairwise."""
    dev = cfg.device
    env = GpuEnv(cfg)
    env.reset([worlds[i % len(worlds)] for i in range(n)], n_players=4)
    fill = (_OPP_BY_KIND[GAUNTLET4_SEATS[1]], _OPP_BY_KIND[GAUNTLET4_SEATS[2]])   # greedy, intermediate
    active = torch.ones(n, device=dev)
    wr = torch.zeros(n, device=dev); pw = torch.zeros(n, device=dev)
    def _eval(s_all):
        s0 = s_all[:, 0]; a1 = s_all[:, 1]
        first = (s0 > s_all[:, 1:].max(1).values).to(DTYPE)                       # strict 1st place
        pwa = torch.where(s0 > a1, torch.ones_like(s0),
                          torch.where(s0 < a1, torch.zeros_like(s0), 0.5 * torch.ones_like(s0)))
        return first, pwa
    with torch.no_grad():
        for t in range(env.T):
            e, m, a, g = env_encode(env, 0); a0, _, _ = act(cfg, net, e, m, a, g, greedy=True)
            oe, om, oa, og = env_encode(env, 1); a1, _, _ = act(cfg, anc_net, oe, om, oa, og, greedy=True)
            seats = [{"pid": 1, "action": a1}, {"pid": 2, "script": fill[0]}, {"pid": 3, "script": fill[1]}]
            env_step(cfg, env, a0, seats=seats, step_idx=t)
            s_all, alive_all = settle_n(env)
            n_alive = alive_all.to(DTYPE).sum(1)
            term = (env.step_ct >= float(env.T - 2)) | (n_alive <= 1.0) | (~alive_all[:, 0])
            newly = (active > 0.5) & term
            f_, p_ = _eval(s_all)
            wr = torch.where(newly, f_, wr); pw = torch.where(newly, p_, pw)
            active = torch.where(term, torch.zeros_like(active), active)
            if active.sum().item() == 0:
                break
        s_all, _ = settle_n(env); f_, p_ = _eval(s_all)
        wr = torch.where(active > 0.5, f_, wr); pw = torch.where(active > 0.5, p_, pw)
    return float(wr.mean().item()), float(pw.mean().item())


_NEURAL_G4 = {"nets": None}


def _neural_gauntlet_nets(cfg):
    """Top invited ckpts as a FIXED neural opponent trio (lazy-loaded, cached for the run)."""
    if _NEURAL_G4["nets"] is None:
        nets = []
        for p, _aelo in discover_invited(cfg)[:3]:
            try:
                n_, _cfg = load_snapshot(cfg, p, fallback_cfg=cfg.INVITE_CFG)
                nets.append(n_)
            except Exception as e:
                print("    [gauntlet4n] skip %s -> %r" % (os.path.basename(p), e))
        _NEURAL_G4["nets"] = nets
    return _NEURAL_G4["nets"]


def _eval_score4_vs(cfg, net, opp_nets, worlds, n_envs):
    """Mean pairwise score of greedy `net` (seat 0) in a 4p FFA where seats 1..3 are the
    given neural nets, padded with the GAUNTLET4_SEATS scripts when fewer than 3."""
    dev = cfg.device
    env = GpuEnv(cfg)
    env.reset([worlds[i % len(worlds)] for i in range(n_envs)], n_players=4)
    seats = []
    for i in range(3):
        if i < len(opp_nets):
            seats.append({"pid": i + 1, "net": opp_nets[i]})
        else:
            seats.append({"pid": i + 1, "script": _OPP_BY_KIND[GAUNTLET4_SEATS[i]]})
    active = torch.ones(n_envs, device=dev); score = torch.zeros(n_envs, device=dev)
    def _pw(s_all):
        s0 = s_all[:, 0:1].expand_as(s_all[:, 1:]); rest = s_all[:, 1:]
        one = torch.ones_like(rest)
        return torch.where(s0 > rest, one, torch.where(s0 < rest, 0.0 * one, 0.5 * one)).mean(1)
    with torch.no_grad():
        for _t in range(env.T):
            ent, em, am, gl = env_encode(env, 0)
            a_t, _, _ = act(cfg, net, ent, em, am, gl, greedy=True)
            step_seats = []
            for sp in seats:
                if "net" in sp:
                    oe, om, oa, og = env_encode(env, sp["pid"])
                    o_act, _, _ = act(cfg, sp["net"], oe, om, oa, og, greedy=True)
                    step_seats.append({"pid": sp["pid"], "action": o_act})
                else:
                    step_seats.append(sp)
            env_step(cfg, env, a_t, seats=step_seats, step_idx=_t)
            s_all, alive_all = settle_n(env)
            n_alive = alive_all.to(DTYPE).sum(1)
            term = (env.step_ct >= float(env.T - 2)) | (n_alive <= 1.0) | (~alive_all[:, 0])
            newly = (active > 0.5) & term
            score = torch.where(newly, _pw(s_all), score)
            active = torch.where(term, torch.zeros_like(active), active)
            if active.sum().item() == 0:
                break
        s_all, _ = settle_n(env)
        score = torch.where(active > 0.5, _pw(s_all), score)
    return float(score.mean().item())


def eval_gauntlet4(cfg, net, worlds, n_envs=None):
    """Mixed 4p best-ckpt gauntlet: scripted trio FFA blended with an invited-neural FFA
    (falls back to scripted-only when no invited members are present)."""
    n = n_envs or cfg.ELO_RECAL_ENVS
    g_s = _eval_score4(cfg, net, GAUNTLET4_SEATS, worlds, n)
    nets = _neural_gauntlet_nets(cfg)
    if not nets:
        return g_s
    for n_ in nets:
        n_.to(cfg.device)
    g_n = _eval_score4_vs(cfg, net, nets, worlds, n)
    for n_ in nets:
        n_.to('cpu')
    return (1.0 - cfg.GAUNTLET_NEURAL_W) * g_s + cfg.GAUNTLET_NEURAL_W * g_n


class League:
    """Dual-Elo self-play league: every member carries a 2p (elo) AND a 4p (elo4)
    rating; the learner is grounded on a ROLLING self-anchor (its own frozen high-water
    snapshot) so the rating never clamps on saturated scripts. Scripted/invited members
    are rating-anchored; snapshots are PFSP-sampled. Recals blend (first=raw) and run a
    MEASURED eviction of mastered scripts at start and every recalibration."""

    def __init__(self, cfg, rng, invite=True):
        self.cfg = cfg
        self.rng = rng
        self.learner_elo = cfg.ELO_INIT
        self.learner_elo4 = cfg.ELO_INIT
        self.advanced = False                                # curriculum: greedy/medium locked until the advance gate
        def anc(label, kind, elo):
            return {"label": label, "net": None, "kind": kind, "elo": elo, "elo4": elo,
                    "anchor": True, "pinned": True, "n": 0, "n4": 0}
        self.members = [
            anc("random", "random", cfg.ELO_RANDOM),
            anc("starter", "starter", cfg.ELO_STARTER),
            anc("greedy", "greedy", cfg.ELO_GREEDY),
            anc("medium", "medium", cfg.ELO_MEDIUM),
            anc("intermediate", "intermediate", cfg.ELO_INTERMEDIATE),
        ]
        if invite:
            for path, aelo in discover_invited(cfg):
                try:
                    self.add_invited(path, aelo)
                except Exception as e:
                    print("  !! invite failed %s -> %r" % (path, e))

    def add_invited(self, path, aelo):
        net, member_cfg = load_snapshot(self.cfg, path, fallback_cfg=self.cfg.INVITE_CFG)
        lab = os.path.splitext(os.path.basename(path))[0]
        m = {"label": "inv:" + lab, "net": net, "kind": "invited", "cfg": member_cfg,
             "elo": float(aelo), "elo4": float(aelo) + self.cfg.INVITE_ELO4_OFFSET,
             "anchor": True, "pinned": True, "n": 0, "n4": 0}
        self.members.append(m)
        print("    invited %-26s provisional elo2 %6.0f / elo4 %6.0f (measured at train start)  (%s)"
              % (m["label"], m["elo"], m["elo4"], os.path.basename(path)))
        return m

    def snapshots(self):
        return [m for m in self.members if m["kind"] == "snapshot"]

    def anchor(self, kind):
        return next(m for m in self.members if m["kind"] == kind)

    def add_checkpoint(self, path, label=None, elo=None):
        net, member_cfg = load_snapshot(self.cfg, path)
        m = {"label": label or os.path.basename(path), "net": net, "kind": "snapshot", "cfg": member_cfg,
             "elo": self.cfg.ELO_INIT if elo is None else elo, "elo4": self.cfg.ELO_INIT if elo is None else elo,
             "anchor": False, "pinned": True, "n": 0, "n4": 0}
        self.members.append(m)
        return m

    def add_learner_snapshot(self, net, it):
        snap = _freeze_snapshot(copy.deepcopy(net))
        m = {"label": "it%d" % it, "net": snap, "kind": "snapshot",
             "elo": self.learner_elo, "elo4": self.learner_elo4,
             "anchor": False, "pinned": False, "n": 0, "n4": 0}
        self.members.append(m)
        self._prune()
        try:  # persist the weights as soon as the snapshot is generated (survives pruning)
            d = os.path.join(self.cfg.CKPT_DIR, "league_agents"); os.makedirs(d, exist_ok=True)
            save_ckpt(self.cfg, snap, os.path.join(d, m["label"] + ".pt"),
                      meta={"iter": it, "elo": float(m["elo"]), "elo4": float(m["elo4"])})
        except Exception as e:
            print("  !! league snapshot save failed: %r" % e)

    def _prune(self):
        """Keep LEAGUE_MAX_SNAPSHOTS most distinct auto-snapshots: repeatedly drop the one whose
        NEAREST neighbor (dest+gate Jensen-Shannon divergence) is closest = most redundant.
        Anchors / pinned ckpts and the NEWEST snapshot are always kept. FIFO fallback on error."""
        autos = [m for m in self.members if m['kind'] == 'snapshot' and not m['pinned']]
        if len(autos) <= self.cfg.LEAGUE_MAX_SNAPSHOTS:
            return
        try:
            fps = [self._fingerprint(m['net']) for m in autos]
        except Exception as e:
            print('  !! league diversity-prune -> FIFO fallback: %r' % e)
            for m in autos[:len(autos) - self.cfg.LEAGUE_MAX_SNAPSHOTS]:
                self.members.remove(m)
            return
        def d(i, j):
            return _js_div(fps[i][0], fps[j][0]) + _js_div(fps[i][1], fps[j][1])
        newest = len(autos) - 1                      # most recently appended -> always keep
        keep = list(range(len(autos)))
        while len(keep) > self.cfg.LEAGUE_MAX_SNAPSHOTS:
            drop, drop_nn = None, float('inf')
            for i in keep:
                if i == newest:
                    continue
                nn = min(d(i, j) for j in keep if j != i)
                if nn < drop_nn:
                    drop_nn, drop = nn, i
            if drop is None:
                break
            keep.remove(drop)
        kept = set(keep)
        for idx, m in enumerate(autos):
            if idx not in kept:
                self.members.remove(m)
        print('  league diversity-prune: kept %d/%d snapshots (max behavioral distinctness)'
              % (len(kept), len(autos)))

    def _p_beat(self, m, fmt):
        me = self.learner_elo if fmt == 2 else self.learner_elo4
        oe = m["elo"] if fmt == 2 else m["elo4"]
        return 1.0 / (1.0 + 10.0 ** ((oe - me) / self.cfg.ELO_SCALE))

    def _weights(self, fmt, net=None):
        """Per-member sampling weight for one format: PFSP(expected score) x footprint mix
        (2p only) x invited/scripted boosts x advance lock, then the scripted-share cap and the
        watchdog / 2p-mastered gates. (Weak-point boost + dominated-bench removed in v12.)"""
        cfg = self.cfg
        members = self.members
        elo_w = [self._pfsp_weight(self._p_beat(m, fmt)) for m in members]
        ws = list(elo_w)
        if fmt == 2 and net is not None and cfg.MATCH_FP_W > 0.0:
            try:
                lfp = self._fingerprint(net)
                fd = []
                for m in members:
                    if m["net"] is None:
                        fd.append(1.0)                                  # scripted anchor: maximally distinct
                    else:
                        if m.get("fp") is None:
                            m["fp"] = self._fingerprint(m["net"])
                        fd.append(_js_div(lfp[0], m["fp"][0]) + _js_div(lfp[1], m["fp"][1]))
                mxe = max(elo_w) or 1.0; mxf = max(fd) or 1.0
                ws = [cfg.MATCH_ELO_W * (elo_w[i] / mxe) + cfg.MATCH_FP_W * (fd[i] / mxf) + cfg.PFSP_FLOOR
                      for i in range(len(members))]
            except Exception:
                ws = list(elo_w)
        for i, m in enumerate(members):
            if m["kind"] == "invited":
                ws[i] *= cfg.INVITE_MATCH_BOOST if not self._mastered(m, fmt, cfg.INVITE_MASTER_WR) else cfg.INVITE_MASTERED_W
            elif m["net"] is None and cfg.SCRIPT_MATCH_BOOST != 1.0:
                if not self._mastered(m, fmt, cfg.SCRIPT_BOOST_DROP_WR):
                    ws[i] *= cfg.SCRIPT_MATCH_BOOST              # spend compute on UNMASTERED scripted anchors
            if (not self.advanced) and m["kind"] in ("greedy", "medium"):
                ws[i] = 0.0                                  # curriculum lock until the advance gate
        # scripted-share cap: scripts collectively <= SCRIPT_SHARE_CAP while any neural carries weight
        scr = sum(w for w, m in zip(ws, members) if m["net"] is None)
        neu = sum(w for w, m in zip(ws, members) if m["net"] is not None)
        if neu > 0.0 and scr > cfg.SCRIPT_SHARE_CAP * (scr + neu):
            f = (cfg.SCRIPT_SHARE_CAP / (1.0 - cfg.SCRIPT_SHARE_CAP)) * (neu / scr)
            ws = [w * f if m["net"] is None else w for w, m in zip(ws, members)]
        # watchdog / mastery gates: dormant watchdogs out (calibration-only); a 2p-mastered
        # script leaves 2p matchmaking (still eligible in 4p until removed by eviction)
        for i, m in enumerate(members):
            if m["net"] is not None:
                continue
            if m["kind"] in cfg.WATCHDOG_KINDS and not m.get("wd_active", False):
                ws[i] = 0.0
            elif fmt == 2 and self._mastered(m, 2, cfg.SCRIPT_BOOST_DROP_WR):
                ws[i] = 0.0
        return ws

    def sample_seats(self, net=None, k=3, fmt=4):
        """k seats for an FFA: weighted draws WITHOUT replacement; at most MAX_SCRIPT_SEATS_4P
        scripted seats (the rest go to neural members), and a neural member fills any seat the
        pool is too thin to fill distinctly."""
        ws = self._weights(fmt, net)
        members = self.members
        neural = [m for m in members if m["net"] is not None]
        out = []; n_script = 0
        for _ in range(k):
            if n_script >= self.cfg.MAX_SCRIPT_SEATS_4P and any(w > 0.0 for w, m in zip(ws, members) if m["net"] is not None):
                ws = [0.0 if m["net"] is None else w for w, m in zip(ws, members)]
            tot = sum(ws)
            if tot <= 0.0:                                   # not enough distinct players -> a network takes the seat
                out.append(self.rng.choice(neural) if neural else self.rng.choice(members))
                continue
            r = self.rng.random() * tot; c = 0.0; pick = len(members) - 1
            for i, w in enumerate(ws):
                c += w
                if r <= c:
                    pick = i; break
            m = members[pick]; out.append(m)
            if m["net"] is None:
                n_script += 1
            if sum(1 for w in ws if w > 0.0) > 1:
                ws[pick] = 0.0                                # without replacement while distinct players remain
        return out

    def share_4p(self):
        """Share of 4p iterations from an EMA of the (elo2-elo4) gap, clipped to [S4_MIN, S4_MAX]."""
        cfg = self.cfg
        gap = (self.learner_elo - self.learner_elo4) / 400.0
        prev = getattr(self, "_gap_ema", None)
        self._gap_ema = gap if prev is None else (1.0 - cfg.S4_EMA_BETA) * prev + cfg.S4_EMA_BETA * gap
        return min(cfg.S4_MAX, max(cfg.S4_MIN, cfg.S4_BASE + cfg.S4_GAIN * self._gap_ema))

    # ---------------- rolling self-anchor recal (the learner is its own moving reference) ----------------
    def _self_anchors(self):
        return [m for m in self.members if m.get("kind") == SELF_KIND]

    def _top2(self):
        a = self._self_anchors()
        return max(a, key=lambda m: m["elo"]) if a else None

    def _top4(self):
        a = self._self_anchors()
        return max(a, key=lambda m: m["elo4"]) if a else None

    def _freeze_anchor(self, net, R2, R4):
        snap = _freeze_snapshot(copy.deepcopy(_unwrap(net)))
        k = 1 + max([int(x["label"][5:]) for x in self._self_anchors()
                     if x["label"].startswith("selfA") and x["label"][5:].isdigit()] + [-1])
        m = {"label": "selfA%d" % k, "net": snap, "kind": SELF_KIND,
             "elo": float(R2), "elo4": float(R4), "anchor": True, "pinned": True, "n": 0, "n4": 0}
        self.members.append(m)
        top4 = self._top4()                          # never trim the current 4p reference
        while len(self._self_anchors()) > ANCHOR_KEEP:
            victim = next((x for x in sorted(self._self_anchors(), key=lambda z: z["elo"]) if x is not top4), None)
            if victim is None:
                break
            self.members.remove(victim)
        return m

    def _ground_ratings(self, net, worlds, n, ground_members, first):
        """Ground the learner (and, on a full recal, every snapshot) on the top self-anchor.
        Blends into the live ratings (RECAL_BLEND) unless `first` (raw). NO promotion / eviction
        here -- those live in _reground so a promotion re-ground can reuse this safely.
        Returns (s2, wr4, pw4, top2, top4, g2)."""
        cfg = self.cfg
        def blend(old, meas):
            return meas if first else (1.0 - cfg.RECAL_BLEND) * old + cfg.RECAL_BLEND * meas
        # lazy chain init: freeze the current learner as the origin anchor selfA0
        if not self._self_anchors():
            R2 = self.learner_elo
            if (not math.isfinite(R2)) or (R2 < cfg.ELO_GREEDY):    # uncalibrated -> tie origin to the script floor
                gm = [self.anchor(k) for k in ("greedy", "medium")]
                R2 = sum(_recal_link(cfg, a["elo"], _eval_score(cfg, net, a["kind"], worlds, n)) for a in gm) / len(gm)
            R4 = self.learner_elo4 if (math.isfinite(self.learner_elo4) and self.learner_elo4 >= cfg.ELO_GREEDY) else R2
            m0 = self._freeze_anchor(net, R2, R4)
            print("    [chain init] froze learner as %s (R2 %.0f / R4 %.0f)" % (m0["label"], R2, R4))
        top2 = self._top2(); top2["net"].to(cfg.device)
        s2 = _score2_vs(cfg, net, top2["net"], worlds, n)
        self.learner_elo = blend(self.learner_elo, _recal_link(cfg, top2["elo"], s2))
        g2 = _eval_score(cfg, net, "greedy", worlds, n)             # collapse watchdog (NOT in the estimate)
        wr4, pw4, top4 = 1.0, 1.0, None
        if cfg.FOURP_ENABLED:
            top4 = self._top4(); top4["net"].to(cfg.device)
            wr4, pw4 = _eval4_vs(cfg, net, top4["net"], worlds, n)
            self.learner_elo4 = blend(self.learner_elo4, _recal_link(cfg, top4["elo4"], pw4))
        if ground_members:                                          # de-compress the snapshot ladder (full recals)
            for mm in self.members:
                if mm["anchor"] or mm["net"] is None or mm.get("kind") == SELF_KIND:
                    continue
                mm["net"].to(cfg.device)
                mm["elo"] = blend(mm["elo"], _recal_link(cfg, top2["elo"], _score2_vs(cfg, mm["net"], top2["net"], worlds, n)))
                if cfg.FOURP_ENABLED:
                    _, mpw = _eval4_vs(cfg, mm["net"], top4["net"], worlds, n)
                    mm["elo4"] = blend(mm["elo4"], _recal_link(cfg, top4["elo4"], mpw))
                mm["net"].to("cpu"); mm["n"] = 0; mm["n4"] = 0
        for a in self._self_anchors():
            if a["net"] is not None:
                a["net"].to("cpu")
        return s2, wr4, pw4, top2, top4, g2

    def _reground(self, net, worlds, n=None, ground_members=False):
        """One recalibration: ground ratings (blended; first=raw), maybe promote (DUAL-format
        dominance), then VALIDATE watchdogs and run the MEASURED eviction. Scripted-anchor eviction
        fires at the very first recal (start/resume) and at every recal thereafter; the neural
        league-player pass runs only on the first/full recal (``first or ground_members``)."""
        cfg = self.cfg
        n = n or cfg.ELO_RECAL_ENVS
        first = not getattr(self, "_grounded", False)
        self._grounded = True
        s2, wr4, pw4, top2, top4, g2 = self._ground_ratings(net, worlds, n, ground_members, first)
        promoted = False
        if (s2 >= PROMOTE_S2) and (wr4 >= PROMOTE_WR4):            # DUAL-format dominance gate
            R2 = _recal_link(cfg, top2["elo"], s2)
            R4 = _recal_link(cfg, top4["elo4"], pw4) if cfg.FOURP_ENABLED else R2
            mp = self._freeze_anchor(net, R2, R4)
            promoted = True
            d4 = (R4 - top4["elo4"]) if cfg.FOURP_ENABLED else 0.0
            print("    [PROMOTE] %s: s2 %.2f wr4 %.2f pw4 %.2f -> R2 %.0f (+%.0f) R4 %.0f (+%.0f)"
                  % (mp["label"], s2, wr4, pw4, R2, R2 - top2["elo"], R4, d4))
        wd = "" if g2 >= GREEDY_WATCHDOG else "  [WATCHDOG] vs greedy %.2f < %.2f -- COLLAPSE?" % (g2, GREEDY_WATCHDOG)
        print("    [recal] elo2 %.0f (s2 %.2f vs %s) elo4 %.0f (pw4 %.2f) | greedy %.2f | anchors %d%s%s"
              % (self.learner_elo, s2, top2["label"], self.learner_elo4, pw4, g2,
                 len(self._self_anchors()), " [first=raw]" if first else "", wd))
        if promoted:                                               # fresh worlds + full league recal per anchor
            self._promo_ctr = getattr(self, "_promo_ctr", 0) + 1
            fresh = make_world_pool(cfg, n, base_seed=cfg.SEED + 80_000_000 + 7919 * self._promo_ctr)
            self._ground_ratings(net, fresh, n, True, first=False)
            self._force_resample = True                            # ask the training loop for fresh training worlds
            print("    [promotion] new anchor #%d -> fresh worlds + full league recal" % self._promo_ctr)
        self.validate_watchdogs(net, worlds, n)                    # rating check -> activate failing watchdogs
        # measured eviction: scripted anchors every recal; the heavier neural-player pass only on the
        # full/first recal (where the league is already being fully re-grounded).
        self.evict_mastered(net, worlds, n, neural=(first or ground_members))
        return promoted

    def recalibrate_elo(self, net, worlds, n_envs=None, all_anchors=False):
        return self._reground(net, worlds, n_envs, ground_members=True)

    def recalibrate_elo4(self, net, worlds, n_envs=None):
        return None                                               # 4p is grounded inside _reground

    def recalibrate_learner(self, net, worlds, n_envs=None):
        return self._reground(net, worlds, n_envs, ground_members=False)

    # ---------------- watchdog calibration validators + measured eviction ----------------
    def validate_watchdogs(self, net, worlds, n=None):
        """Each DORMANT watchdog (random/greedy) scores the learner vs its own Elo prediction;
        if the learner underperforms (actual < expected - WATCHDOG_TOL) the rating is inflated /
        the policy regressed, so the watchdog ENTERS the league as a live training opponent."""
        cfg = self.cfg
        n = n or cfg.ELO_RECAL_ENVS
        for m in self.members:
            if m["net"] is not None or m["kind"] not in cfg.WATCHDOG_KINDS or m.get("wd_active", False):
                continue
            actual = _eval_score(cfg, net, m["kind"], worlds, n)
            expected = self._p_beat(m, 2)
            if actual < expected - cfg.WATCHDOG_TOL:
                m["wd_active"] = True
                print("    [watchdog] %-8s FAILED validation: 2p %.2f < expected %.2f - %.2f -> ENTERS league"
                      % (m["kind"], actual, expected, cfg.WATCHDOG_TOL))

    def _newest_snapshot(self):
        """The most recently frozen learner snapshot (the current self) -- never evicted, so the
        league always keeps the freshest self-play opponent to train against."""
        autos = [m for m in self.members if m["kind"] == "snapshot" and not m["pinned"]]
        return autos[-1] if autos else None

    def _measure_mastery(self, net, m, worlds, n):
        """(s2, w4) of the learner vs member `m` on ONE standard for scripts and league players:
        s2 = seat-avg 2p score, w4 = 4p 1st-place rate vs THREE copies (0.0 when 4p is disabled)."""
        cfg = self.cfg
        if m["net"] is None:                                        # scripted anchor (on-GPU)
            s2 = _eval_score(cfg, net, m["kind"], worlds, n)
            w4 = _eval_winrate4(cfg, net, m["kind"], worlds, n) if cfg.FOURP_ENABLED else 0.0
        else:                                                       # neural league player (snapshot/invited)
            m["net"].to(cfg.device)
            s2 = _score2_vs(cfg, net, m["net"], worlds, n)
            w4 = _eval_winrate4_net(cfg, net, m["net"], worlds, n) if cfg.FOURP_ENABLED else 0.0
            m["net"].to("cpu")
        return s2, w4

    def evict_mastered(self, net, worlds, n=None, neural=True):
        """MEASURED eviction (start + every recal): remove every league member the learner has
        MASTERED under ONE standard -- 2p seat-avg score s2 and 4p 1st-place rate w4 vs three copies,
        mastered when ``s2 > MASTER_EVICT_2P_WR`` OR ``w4 > MASTER_EVICT_4P_WR`` (per-format; the 4p
        clause is skipped when 4p is disabled). Covers scripted anchors AND neural league players
        (snapshots + invited) -- the latter only when ``neural`` (gated by the caller to the heavier
        full-recal/start path). NEVER touched: the rolling self-anchors (the Elo reference) and the
        newest snapshot (the current self); keep-kinds random/greedy are never removed -- a re-mastered
        ACTIVE watchdog retires to DORMANT instead."""
        cfg = self.cfg
        n = n or cfg.ELO_RECAL_ENVS
        keep_newest = self._newest_snapshot()
        for m in list(self.members):
            is_script = (m["net"] is None and m["kind"] in _OPP_BY_KIND)
            is_player = (m["net"] is not None and m["kind"] in ("snapshot", "invited"))
            if not (is_script or (neural and is_player)) or m is keep_newest:
                continue                                            # self-anchors / freshest self: never
            s2, w4 = self._measure_mastery(net, m, worlds, n)
            mastered = (s2 > cfg.MASTER_EVICT_2P_WR) or (cfg.FOURP_ENABLED and w4 > cfg.MASTER_EVICT_4P_WR)
            m["mastered"] = bool(mastered)
            if not mastered:
                continue
            if m["kind"] in cfg.MASTER_KEEP_KINDS:                  # random/greedy: retire watchdog, never remove
                if m["kind"] in cfg.WATCHDOG_KINDS and m.get("wd_active", False):
                    m["wd_active"] = False                          # re-mastered -> calibration-only
                    print("    [watchdog] %-8s re-mastered (2p=%.2f 4p_win=%.2f) -> back to DORMANT"
                          % (m["kind"], s2, w4))
            else:
                self.members.remove(m)
                who = "script" if is_script else m["kind"]
                print("    [evict] %-14s (%-7s) mastered (2p=%.2f 4p_win=%.2f) -> removed from league"
                      % (m["label"], who, s2, w4))

    def _pfsp_weight(self, p):
        cfg = self.cfg
        if cfg.PFSP_MODE == "hard":
            return (1.0 - p) ** cfg.PFSP_POWER + cfg.PFSP_FLOOR   # focus on opponents you can't yet beat
        if cfg.PFSP_MODE == "even":
            return p * (1.0 - p) + cfg.PFSP_FLOOR             # focus on evenly-matched opponents
        return 1.0                                        # uniform

    def _mastered(self, m, fmt, thr):
        n = m["n"] if fmt == 2 else m.get("n4", 0)
        wr = m.get("wr") if fmt == 2 else m.get("wr4")
        return (n >= self.cfg.SCRIPT_BOOST_MIN_GAMES) and ((wr if wr is not None else 0.0) >= thr)

    def sample(self, net=None, fmt=2):
        """PFSP-sample ONE opponent using the per-format ratings + boosts."""
        ws = self._weights(fmt, net)
        tot = sum(ws)
        if tot <= 0.0:
            return self.rng.choice(self.members)
        r = self.rng.random() * tot
        c = 0.0
        for m, w in zip(self.members, ws):
            c += w
            if r <= c:
                return m
        return self.members[-1]

    def update_elo(self, m, learner_score):
        """2p result: learner_score in [0,1] = win + 0.5*draw over the rollout. Anchors stay
        fixed. The 2p delta also drags BOTH 4p ratings by ELO_COUPLING (correlated ratings)."""
        cfg = self.cfg
        exp = self._p_beat(m, 2)
        delta = cfg.ELO_K * (learner_score - exp)
        self.learner_elo += delta
        self.learner_elo4 += cfg.ELO_COUPLING * delta
        m["n"] += 1
        m["wr"] = learner_score if m.get("wr") is None else (1.0 - cfg.WR_EMA_BETA) * m["wr"] + cfg.WR_EMA_BETA * learner_score
        if not m["anchor"]:
            m["elo"] -= delta
            m["elo4"] -= cfg.ELO_COUPLING * delta

    def update_elo_4p(self, seats, seat_scores):
        """4p FFA result as len(seats) pairwise updates at ELO_K4/len each (standard multiplayer
        decomposition). Coupled back into the 2p ratings by ELO_COUPLING."""
        cfg = self.cfg
        kf = cfg.ELO_K4 / max(1, len(seats))
        for m, sc in zip(seats, seat_scores):
            exp = self._p_beat(m, 4)
            delta = kf * (sc - exp)
            self.learner_elo4 += delta
            self.learner_elo += cfg.ELO_COUPLING * delta
            m["n4"] = m.get("n4", 0) + 1
            m["wr4"] = sc if m.get("wr4") is None else (1.0 - cfg.WR_EMA_BETA) * m["wr4"] + cfg.WR_EMA_BETA * sc
            if not m["anchor"]:
                m["elo4"] -= delta
                m["elo"] -= cfg.ELO_COUPLING * delta

    def leaderboard(self):
        cfg = self.cfg
        rows = sorted(self.members, key=lambda m: -m["elo"])
        s = "  league | learner elo2 %.0f elo4 %.0f | share4p %.2f | %d members\n" % (
            self.learner_elo, self.learner_elo4, self.share_4p(), len(self.members))
        for m in rows:
            tag = "[anchor]" if m["anchor"] else ("[pinned]" if m["pinned"] else "")
            if m["kind"] == "invited":
                tag = "[invited %s%s]" % ("M" if self._mastered(m, 2, cfg.INVITE_MASTER_WR) else "-",
                                          "M" if self._mastered(m, 4, cfg.INVITE_MASTER_WR) else "-")
            elif m["net"] is None:
                boosted = not self._mastered(m, 2, cfg.SCRIPT_BOOST_DROP_WR)
                tag += " x%.1f" % cfg.SCRIPT_MATCH_BOOST if boosted else " (mastered)"
            wr = m.get("wr"); wr4 = m.get("wr4")
            s += "    %-22s elo2 %6.0f (n=%-3d wr %s)  elo4 %6.0f (n4=%-3d wr %s) %s\n" % (
                m["label"], m["elo"], m["n"], ("%.2f" % wr) if wr is not None else " -- ",
                m["elo4"], m.get("n4", 0), ("%.2f" % wr4) if wr4 is not None else " -- ", tag)
        return s

    def _probe_obs(self):
        """Fingerprint probe = REAL encoded game states (on-manifold), cached once."""
        if getattr(self, '_probe', None) is None:
            env = GpuEnv(self.cfg)
            env.reset(make_world_pool(self.cfg, 16, base_seed=0xC0FFEE % 100000))
            with torch.no_grad():
                for _t in range(20):
                    e, m, a, g = env_encode(env, 0)
                    rnd = torch.softmax(torch.randn(env.B, PLANET_CAP, PLANET_CAP + 1, device=self.cfg.device), -1)
                    env_step(self.cfg, env, rnd, 1, None, step_idx=_t)
            e, m, a, g = env_encode(env, 0)
            self._probe = (e.cpu(), m.cpu(), a.cpu(), g.cpu())
        return self._probe

    def _fingerprint(self, net):
        """Behavioral fingerprint = mean (WHERE allocation, gate fire/hold) on the fixed probe batch."""
        ent, em, am, gl = self._probe_obs()
        dev = next(net.parameters()).device
        with torch.no_grad():
            dest_logits, gate_logits, _, _ = net(ent.to(dev), em.to(dev), am.to(dev), gl.to(dev))
            dest = torch.softmax(dest_logits, -1).reshape(-1, dest_logits.shape[-1]).mean(0)  # (E,)
            gp = torch.sigmoid(gate_logits).reshape(-1).float().mean()                         # mean fire-prob
            gate = torch.stack([gp, 1.0 - gp])                                                 # (2,) fire/hold
        return dest.cpu(), gate.cpu()

    def calibrate_anchor_elo4(self, net, worlds, n_envs=None):
        """Measure the scripted anchors' TRUE 4p pairwise strength (2 lineups x 3 seat
        rotations, ego = current learner) and re-pin every scripted elo4 on one scale with
        greedy fixed at ELO_GREEDY. recalibrate_elo4/recalibrate_learner then ground the 4p
        scale on the measured GAUNTLET4_SEATS base instead of the 2p-copied constants."""
        cfg = self.cfg
        n = n_envs or cfg.ELO_RECAL_ENVS
        acc, cnt = {}, {}
        def run(trio):
            for r in range(3):
                tr = list(trio[r:]) + list(trio[:r])
                ps = _eval_pair_scores4(cfg, net, tr, worlds, n)
                for (a, b), s in ps.items():
                    ka, kb = tr[a - 1], tr[b - 1]
                    acc[(ka, kb)] = acc.get((ka, kb), 0.0) + s;         cnt[(ka, kb)] = cnt.get((ka, kb), 0) + 1
                    acc[(kb, ka)] = acc.get((kb, ka), 0.0) + (1.0 - s); cnt[(kb, ka)] = cnt.get((kb, ka), 0) + 1
        run(list(GAUNTLET4_SEATS))                 # starter / greedy / intermediate
        run(["medium", "greedy", "random"])        # greedy shared -> one common scale
        def rel(k):
            ss = [acc[(k, o)] / cnt[(k, o)] for o in _CAL4_KINDS if (k, o) in acc]
            s = min(max(sum(ss) / len(ss), 0.05), 0.95)
            return cfg.ELO_SCALE * math.log10(s / (1.0 - s))
        r = {k: rel(k) for k in _CAL4_KINDS if any((k, o) in acc for o in _CAL4_KINDS)}
        out = {k: cfg.ELO_GREEDY + r[k] - r["greedy"] for k in r}
        for k, e in out.items():
            try:
                self.anchor(k)["elo4"] = float(e)
            except StopIteration:
                pass                               # anchor already evicted (resume) -- skip
        self._anc4_base = sum(out[k] for k in GAUNTLET4_SEATS) / float(len(GAUNTLET4_SEATS))
        print("    [anc4] measured 4p anchors: " + " ".join("%s=%.0f" % (k, out[k]) for k in sorted(out))
              + " | trio base %.0f" % self._anc4_base)

    def calibrate_invited(self, worlds, n_envs=None):
        """Measure every invited member's elo2 (vs the ground anchors) and elo4 (vs the 4p trio,
        on the measured base). Filename/meta stamps carry old run-internal scales -- anchoring to
        them poisons the expected-score model AND matchmaking. Frozen agents: measuring once per
        train() start is cheap and exact."""
        cfg = self.cfg
        n = n_envs or cfg.ELO_RECAL_ENVS
        inv = [m for m in self.members if m["kind"] == "invited"]
        if not inv:
            return
        ground = [m for m in self.members if m["anchor"] and m["net"] is None
                  and m["kind"] in cfg.ELO_GROUND_KINDS]
        if not ground:
            ground = [m for m in self.members if m["anchor"] and m["net"] is None]
        base4 = getattr(self, "_anc4_base", None)
        if base4 is None:
            base4 = (cfg.ELO_STARTER + cfg.ELO_GREEDY + cfg.ELO_INTERMEDIATE) / 3.0
        def _est(base, sc):
            sc = min(max(sc, 0.02), 0.98)
            return base + cfg.ELO_SCALE * math.log10(sc / (1.0 - sc))
        for m in inv:
            old2, old4 = m["elo"], m["elo4"]
            m["net"].to(cfg.device)
            m["elo"] = sum(_est(a["elo"], _eval_score(cfg, m["net"], a["kind"], worlds, n))
                           for a in ground) / len(ground)
            m["elo4"] = (_est(base4, _eval_score4(cfg, m["net"], GAUNTLET4_SEATS, worlds, n))
                         if cfg.FOURP_ENABLED else m["elo"])
            m["net"].to("cpu")
            print("    [inv cal] %-26s elo2 %6.0f -> %6.0f | elo4 %6.0f -> %6.0f"
                  % (m["label"], old2, m["elo"], old4, m["elo4"]))

    def state_dict(self):
        members = []
        for m in self.members:
            members.append({"label": m["label"], "kind": m["kind"], "elo": m["elo"],
                            "elo4": m.get("elo4", m["elo"]), "anchor": m["anchor"], "pinned": m["pinned"],
                            "n": m["n"], "n4": m.get("n4", 0), "wr": m.get("wr"), "wr4": m.get("wr4"),
                            "cfg": m.get("cfg"),
                            "model": (_unwrap(m["net"]).state_dict() if m["net"] is not None else None)})
        return {"learner_elo": self.learner_elo, "learner_elo4": self.learner_elo4,
                "advanced": self.advanced, "members": members}

    def load_state_dict(self, sd):
        self.learner_elo = sd["learner_elo"]
        self.learner_elo4 = sd.get("learner_elo4", sd["learner_elo"])
        self.advanced = sd.get("advanced", self.advanced)
        self.members = []
        for e in sd["members"]:
            net = None
            if e["model"] is not None:
                net = build_from_cfg(self.cfg, e["cfg"]) if e.get("cfg") else build_policy(self.cfg)
                net.load_state_dict(e["model"])
                net = _freeze_snapshot(_wrap_if_legacy(net, e.get("cfg")))
            m = {"label": e["label"], "kind": e["kind"], "net": net, "elo": e["elo"],
                 "elo4": e.get("elo4", e["elo"]), "anchor": e["anchor"], "pinned": e["pinned"],
                 "n": e["n"], "n4": e.get("n4", 0)}
            if e.get("cfg") is not None:
                m["cfg"] = e["cfg"]
            if e.get("wr") is not None:
                m["wr"] = e["wr"]
            if e.get("wr4") is not None:
                m["wr4"] = e["wr4"]
            self.members.append(m)

## `orbit_wars_v12.train`

Elo-league-driven training loop (notebook cells 30 + 35).

:class:`Trainer` owns the :class:`~orbit_wars_v12.config.Config` and runs the loop: PFSP-sample an
opponent (2p or 4p per the dual-Elo schedule), roll out, PPO-update, update Elo, periodic recals +
measured eviction, and best-by-mixed-gauntlet checkpointing. There are NO hand-coded stages -- the
league IS the curriculum. The notebook's runtime globals are gone: the annealed entropy coefficient
and the value-warmup policy coefficient are computed per-iter and passed into ``ppo_update``.

In [ ]:
def anneal_ent_coef(cfg, it):
    if cfg.ENT_DECAY_ITERS <= 0:
        return cfg.ENT_COEF_END
    f = min(1.0, it / cfg.ENT_DECAY_ITERS)
    return cfg.ENT_COEF_START + (cfg.ENT_COEF_END - cfg.ENT_COEF_START) * f


class Trainer:
    """Holds the config and drives one or more train() calls. Returns (net, hist, league)."""

    def __init__(self, cfg):
        self.cfg = cfg

    def train(self, total_iters=None, log_every=1, resume_from=None):
        cfg = self.cfg
        total_iters = cfg.TOTAL_ITERS if total_iters is None else total_iters
        resume_from = cfg.RESUME_FROM if resume_from is None else resume_from

        net = build_policy(cfg)
        try:
            opt = torch.optim.Adam(net.parameters(), lr=cfg.LR, eps=cfg.ADAM_EPS, fused=(cfg.device.type == 'cuda'))
        except Exception:
            opt = torch.optim.Adam(net.parameters(), lr=cfg.LR, eps=cfg.ADAM_EPS)
        net_fwd = torch.compile(net, mode=cfg.COMPILE_MODE) if cfg.USE_COMPILE else net   # compile LEARNER forward only;
        #          snapshots (deepcopy), recals and saves keep the eager `net` to avoid recompile churn.
        maybe_compile_encode(cfg)   # also compile the pure obs encoder (_encode_core) -> fused rollout encode
        env = GpuEnv(cfg)
        rng = random.Random(cfg.SEED * 2654435761 + 12345)
        start_it = 0
        best_wr = -1.0

        league = League(cfg, rng)
        for ck in cfg.LEAGUE_INIT_CKPTS:
            try:
                m = league.add_checkpoint(ck)
                print("league seeded with %s  <- %s" % (m["label"], ck))
            except Exception as e:
                print("  !! skipped league ckpt %s -> %r" % (ck, e))

        if resume_from:
            start_it, best_wr = load_train_state(cfg, net, opt, league, resume_from)
            print("resumed from %s -> global iter %d, best_wr %.2f, %d league members"
                  % (resume_from, start_it, best_wr, len(league.members)))

        # GRPO reference policy for the optional KL penalty: freeze the (post-resume/BC) net once.
        ref_net = None
        if cfg.ALGO == "grpo" and cfg.GRPO_KL_COEF > 0.0:
            ref_net = _freeze_snapshot(copy.deepcopy(net))
            print("  [grpo] KL-to-reference enabled (beta=%.3g) -> froze current policy as reference" % cfg.GRPO_KL_COEF)

        if cfg.ALGO == "grpo":   # report the ACTIVE advantage estimator (the flags are mutually exclusive!)
            a2, ignored = grpo_adv_mode(cfg, 2)
            line = "  [grpo] advantage estimator = %s" % a2
            if cfg.FOURP_ENABLED and grpo_adv_mode(cfg, 4)[0] != a2:
                line += "  (2p) / %s (4p)" % grpo_adv_mode(cfg, 4)[0]
            comp = [t for t, on in (("clip-hi", cfg.CLIP_HI > 0), ("len-norm", cfg.RATIO_LENGTHNORM),
                                    ("rb-gate", cfg.GRPO_RB_GATE), ("aux-value", cfg.GRPO_AUX_VALUE),
                                    ("opp-base", cfg.GRPO_OPP_BASELINE_W > 0), ("target-rms", cfg.ADV_TARGET_RMS > 0),
                                    ("LOO", cfg.GRPO_LOO), ("KL-ref", cfg.GRPO_KL_COEF > 0), ("CRN", cfg.GRPO_CRN)) if on]
            print(line + ("  + " + ",".join(comp) if comp else ""))
            if ignored:
                print("  [grpo] !! WARNING: estimators [%s] are ALSO enabled but IGNORED (mutually exclusive) -- "
                      "ONLY %s is active. Enable exactly ONE advantage estimator." % (", ".join(ignored), a2))

        hist = {"iter": [], "return": [], "win_rate": [],
                "lnch_per_step": [], "approx_kl": [], "clipfrac": [], "sigma": [], "grad_norm": [], "auxr": [],
                "r_outcome": [], "r_capture": [], "r_milestone": [], "r_launch": [], "r_shape": [], "r_alive": [], "r_win_bet": [],
                "learner_elo": [], "learner_elo4": [], "share_4p": [], "fmt": [],
                "adv_absmax": [], "adv_std": []}   # anti-collapse advantage-health curves

        pool, pool_round = None, -1
        # held-out gauntlet worlds: FIXED seed range never touched by training rounds, so the
        # best-ckpt score measures generalization and stays comparable across the whole run.
        heldout_pool = make_world_pool(cfg, cfg.ELO_RECAL_ENVS, base_seed=cfg.SEED + 999_999_937)
        if cfg.FOURP_ENABLED:
            league.calibrate_anchor_elo4(net, heldout_pool)   # v9.3: measured (not 2p-copied) 4p anchors
        if cfg.INVITE_MEASURE:
            league.calibrate_invited(heldout_pool)   # v9.4: measured invited anchors (stamps are stale-scale)
        acc4 = 0.0       # error-diffusion accumulator -> the realized 2p:4p ratio equals share_4p
        for step in range(total_iters):
            it = start_it + step
            rnd = (it // cfg.WORLD_RESAMPLE_EVERY) if cfg.WORLD_RESAMPLE_EVERY > 0 else 0
            _force = getattr(league, "_force_resample", False)   # v9.9: a promotion asked for fresh worlds
            if rnd != pool_round or _force:
                if _force:
                    league._force_resample = False
                    league._resample_salt = getattr(league, "_resample_salt", 0) + 1
                _salt = getattr(league, "_resample_salt", 0)
                pool = make_world_pool(cfg, cfg.N_WORLDS, base_seed=cfg.SEED + rnd * cfg.N_WORLDS + 7919 * _salt)
                cursor, pool_round = 0, rnd
                print("  [worlds] pool round %d @ it%d (base_seed %d)%s" % (
                    rnd, it, cfg.SEED + rnd * cfg.N_WORLDS + 7919 * _salt,
                    " [promotion resample #%d]" % _salt if _force else ""))
            if cfg.ELO_RECAL_EVERY > 0 and step > 0 and it % cfg.ELO_RECAL_EVERY == 0:  # re-ground BOTH rating scales
                league.recalibrate_elo(net, pool)
                if cfg.FOURP_ENABLED:
                    league.recalibrate_elo4(net, pool)
            elif cfg.LEARNER_RECAL_EVERY > 0 and step > 0 and it % cfg.LEARNER_RECAL_EVERY == 0:
                league.recalibrate_learner(net, pool)   # v9.2: cheap learner-only re-ground (blended)
            t0 = time.time()
            ent_coef = anneal_ent_coef(cfg, it)   # entropy anneal (anti collapse -> exploit)
            # v7 guardrails: linear LR warmup + critic-only warmup (both per train() CALL, not global iter)
            # GRPO has no critic to warm up -> always full policy weight.
            policy_coef = 1.0 if cfg.ALGO in ("grpo", "mc") else (0.0 if step < cfg.VALUE_WARMUP_ITERS else 1.0)
            if cfg.LR_WARMUP_ITERS > 0:
                wf = min(1.0, (step + 1) / float(cfg.LR_WARMUP_ITERS))
                for pg in opt.param_groups:
                    pg['lr'] = cfg.LR * wf

            # grow the pool: snapshot the learner (after a short anchors-only warmup) every SELFPLAY_REFRESH iters
            if (it >= cfg.SNAPSHOT_WARMUP) and (step == 0 or it % max(1, cfg.SELFPLAY_REFRESH) == 0):
                league.add_learner_snapshot(net, it + 1)

            # ---- v8 match-type scheduler: dual-Elo gap -> bounded 2p/4p mix (1:1 .. 1:6) ----
            s4 = league.share_4p() if cfg.FOURP_ENABLED else 0.0
            acc4 += s4
            if acc4 >= 1.0:
                fmt = 4; acc4 -= 1.0
            else:
                fmt = 2

            # pick the opponent seat(s) for this rollout: PFSP over the whole league per format
            if fmt == 2:
                seats = [league.sample(net, fmt=2)]
            else:
                seats = league.sample_seats(net, k=3, fmt=4)
            for m in seats:
                if m["net"] is not None:
                    m["net"].to(cfg.device)

            opp_exp = None
            if cfg.ALGO == "grpo" and cfg.GRPO_OPP_BASELINE_W > 0.0:   # [E1] league Elo-expected outcome vs this foe
                opp_exp = (league._p_beat(seats[0], 2) if fmt == 2
                           else sum(league._p_beat(m, 4) for m in seats) / max(1, len(seats)))
            tb, rs, cursor = collect_ppo(cfg, env, net_fwd, pool, cursor, rng, seats,
                                         n_players=fmt, n_envs=(cfg.B if fmt == 2 else cfg.B_4P), opp_exp=opp_exp)
            if cfg.ALGO in ("grpo", "mc"):   # both critic-free: same clipped-surrogate update (no value loss)
                us = grpo_update(cfg, net_fwd, opt, tb, ent_coef, policy_coef, ref_net=ref_net)
            else:
                us = ppo_update(cfg, net_fwd, opt, tb, ent_coef, policy_coef)

            # dual Elo: 2p = single update; 4p = pairwise vs every seat (cross-coupled inside)
            if fmt == 2:
                league.update_elo(seats[0], rs["score"])
            else:
                league.update_elo_4p(seats, rs["seat_scores"])
            if (not league.advanced) and league.learner_elo >= cfg.ADVANCE_TRIGGER_ELO:   # curriculum: confirm with an all-anchor recal (fresh seeds)
                league.recalibrate_elo(net, make_world_pool(cfg, cfg.ELO_RECAL_ENVS, base_seed=cfg.SEED + 104729 + 7919 * (it + 1)), all_anchors=True)
                if league.learner_elo >= cfg.ADVANCE_CONFIRM_ELO:
                    league.advanced = True
                    print("  [advance] learner hit %.0f ELO; all-anchor recal %.0f >= %d -> UNLOCK greedy/medium" % (cfg.ADVANCE_TRIGGER_ELO, league.learner_elo, cfg.ADVANCE_CONFIRM_ELO))
                else:
                    print("  [advance] learner hit %.0f ELO but all-anchor recal %.0f < %d -> stay in base regime" % (cfg.ADVANCE_TRIGGER_ELO, league.learner_elo, cfg.ADVANCE_CONFIRM_ELO))
            for m in seats:
                if m["net"] is not None:
                    m["net"].to("cpu")           # keep only the live policy resident on the GPU

            dt = time.time() - t0
            sps = rs["transitions"] / max(dt, 1e-9)

            hist["iter"].append(it + 1)
            hist["return"].append(rs["mean_return"]); hist["win_rate"].append(rs["win_rate"])
            hist["lnch_per_step"].append(rs["lnch_per_step"]); hist["approx_kl"].append(us["approx_kl"])
            hist["clipfrac"].append(us["clipfrac"]); hist["sigma"].append(us["sigma"]); hist["grad_norm"].append(us["grad_norm"]); hist["auxr"].append(us.get("auxr", 0.0))
            hist["r_outcome"].append(rs["r_outcome"]); hist["r_capture"].append(rs["r_capture"])
            hist["r_milestone"].append(rs["r_milestone"]); hist["r_launch"].append(rs["r_launch"])
            hist["r_shape"].append(rs.get("r_shape", 0.0))
            hist["r_alive"].append(rs.get("r_alive", 0.0)); hist["r_win_bet"].append(rs.get("r_win_bet", 0.0))
            hist["learner_elo"].append(league.learner_elo)
            hist["learner_elo4"].append(league.learner_elo4)
            hist["share_4p"].append(s4); hist["fmt"].append(fmt)
            hist["adv_absmax"].append(rs.get("adv_absmax", 0.0)); hist["adv_std"].append(rs.get("adv_std", 0.0))

            # v7 tripwire: gate-collapse detector (launch rate decaying under healthy-looking KL)
            _l = hist["lnch_per_step"]
            if len(_l) >= cfg.TRIPWIRE_BASE_ITERS and (it + 1) % 10 == 0:
                _base = sum(_l[:cfg.TRIPWIRE_BASE_ITERS]) / cfg.TRIPWIRE_BASE_ITERS
                _ema = sum(_l[-5:]) / len(_l[-5:])
                if _base > 0 and _ema < cfg.TRIPWIRE_FRAC * _base:
                    print("  [TRIPWIRE] launch/st EMA %.2f < %.0f%% of early baseline %.2f -> passivity ratchet suspected; check gauntlet+elo before continuing" % (_ema, cfg.TRIPWIRE_FRAC * 100, _base))

            if (it + 1) % log_every == 0 or step == 0 or step == total_iters - 1:
                # heartbeat: R[o c p ln (sh) al (wb)] = outcome, capture, prod-milestone, launch, (shaping), alive-survival, (win-bet)
                opp_lab = "+".join(m["label"] for m in seats)
                _wb = (" wb %5.1f" % rs.get("r_win_bet", 0.0)) if cfg.AUX_WIN_BET_REWARD else ""
                _sh = (" sh %6.1f" % rs.get("r_shape", 0.0)) if cfg.USE_POTENTIAL_SHAPING else ""
                print("it%4d %dp | ret %8.2f wr %.2f sc %.2f | R[o %7.1f c %6.1f p %5.1f ln %5.1f%s al %6.1f%s] | "
                      "lnch/st %.2f | kl %.3f cf %.2f gn %.1f aMx %5.1f | loss %7.3f (pol %.3f vf %.3f%s) | "
                      "elo2 %5.0f elo4 %5.0f s4 %.2f vs %-22s | sps %5.0f"
                      % (it + 1, fmt, rs["mean_return"], rs["win_rate"], rs["score"],
                         rs["r_outcome"], rs["r_capture"], rs["r_milestone"], rs["r_launch"], _sh, rs["r_alive"], _wb,
                         rs["lnch_per_step"], us["approx_kl"], us["clipfrac"], us["grad_norm"], rs["adv_absmax"],
                         us["total"], us["policy"], us["vf"],
                         (" bet %+.3f" % us["auxr"] if cfg.AUX_WIN_BET else (" ar %.3f" % us["auxr"] if cfg.AUX_REWARD_PRED else "")),
                         league.learner_elo, league.learner_elo4, s4, opp_lab[:22], sps))

            # checkpoint: periodic + best-by-MIXED-GAUNTLET (2p + 4p; fixed opponents, held-out worlds)
            if ((it + 1) % max(1, cfg.CKPT_EVERY) == 0) or (step == total_iters - 1):
                save_ckpt(cfg, net, cfg.CKPT_PATH, meta={"iter": it + 1, "win_rate": rs["win_rate"]})
                save_train_state(net, opt, league, it + 1, best_wr, cfg.TRAIN_STATE_PATH)
                g2 = eval_gauntlet(cfg, net, heldout_pool)
                g4 = eval_gauntlet4(cfg, net, heldout_pool) if cfg.FOURP_ENABLED else g2
                gscore = (1.0 - cfg.GAUNTLET_4P_W) * g2 + cfg.GAUNTLET_4P_W * g4
                hist.setdefault("gauntlet", []).append(gscore)
                hist.setdefault("gauntlet2", []).append(g2)
                hist.setdefault("gauntlet4", []).append(g4)
                if gscore > best_wr:
                    best_wr = gscore
                    save_ckpt(cfg, net, cfg.BEST_CKPT_PATH, meta={"iter": it + 1, "gauntlet": float(gscore),
                                                                  "gauntlet2": float(g2), "gauntlet4": float(g4)})
                    print("  [best] mixed gauntlet %.3f (2p %.3f / 4p %.3f) -> saved best" % (gscore, g2, g4))
                print(league.leaderboard(), end="")

        save_ckpt(cfg, net, cfg.CKPT_PATH, meta={"iter": start_it + total_iters, "win_rate": hist["win_rate"][-1]})
        save_train_state(net, opt, league, start_it + total_iters, best_wr, cfg.TRAIN_STATE_PATH)
        print("saved final checkpoint ->", cfg.CKPT_PATH, "| best mixed gauntlet %.2f ->" % best_wr, cfg.BEST_CKPT_PATH)
        print(league.leaderboard(), end="")
        json.dump({k: [float(x) for x in v] for k, v in hist.items()},
                  open(os.path.join(cfg.CKPT_DIR, "metrics.json"), "w"))
        print("saved metrics ->", os.path.join(cfg.CKPT_DIR, "metrics.json"))
        return net, hist, league


def train(cfg, total_iters=None, log_every=1, resume_from=None):
    """Convenience: build a Trainer and run one train() call. Returns (net, hist, league)."""
    return Trainer(cfg).train(total_iters=total_iters, log_every=log_every, resume_from=resume_from)


def save_league_agents(cfg, league):
    """Save every league member that carries weights as a standalone .pt (notebook cell 35)."""
    out = os.path.join(cfg.CKPT_DIR, "league_agents")
    os.makedirs(out, exist_ok=True)
    saved = 0
    for m in sorted(league.members, key=lambda mm: -mm["elo"]):
        if m["net"] is None:                       # scripted anchors (random/starter) -> no weights
            continue
        safe = "".join(c if c.isalnum() else "_" for c in m["label"])
        path = os.path.join(out, "%s_elo%04d.pt" % (safe, round(m["elo"])))
        save_ckpt(cfg, m["net"], path,
                  meta={"elo": m["elo"], "elo4": m.get("elo4"), "label": m["label"],
                        "games": m["n"], "games4": m.get("n4", 0)})
        saved += 1
        print("saved %-12s elo %6.0f  n=%-4d -> %s" % (m["label"], m["elo"], m["n"], path))
    print("saved %d league agents -> %s" % (saved, out))
    return saved

## `orbit_wars_v12.mcts`

AlphaZero-style MCTS for Orbit Wars -- proof of concept (NOT wired into training).

The v12 stack already gives us everything AlphaZero needs except the search itself: a
bit-exact, fast forward model (:class:`~orbit_wars_v12.env.GpuEnv`) and a policy net that
emits a prior + a value head (:mod:`orbit_wars_v12.policy`). This module adds the missing
piece -- a tree search that uses the net as a **prior over sampled candidate actions** and a
bounded **leaf value** -- so a (small) net can spend inference compute to play better, and so
search visit distributions can later serve as policy-improvement targets (BC warm-start ->
self-play distillation; that loop is the *next* step, not this file).

Scope / simplifications for the PoC:

* **Opponent = a fixed deterministic script** (``opponent_action`` codes). With the opponent
  fixed, the game is a *deterministic single-agent MDP* from the ego's view, so plain PUCT is
  sound -- we deliberately dodge the simultaneous-move subtlety here. The real fix for
  self-play (decoupled-UCT / regret-matching at each node, 2p zero-sum first) comes once we
  have a BC-warm-started net to self-play.
* **Single env (B=1).** A node stores a full state snapshot; expansion restores it into one
  shared work env and steps once. Batched-across-games search is a later optimization.
* **Leaf value = bounded ship/production margin heuristic** by default (``value="net"`` uses
  the value head). SUBMISSION.md found the PPO critic is policy-biased and lost to exactly this
  heuristic for leaf eval; the AlphaZero loop is what later de-biases the head.

Determinism (required for correct snapshot/restore + reproducible search) holds when comets are
official (``COMET_OFFICIAL=True``, waypoint playback -- no ``torch.rand``) and the scripted
opponent is deterministic (codes 1/3/4/5; avoid 0=random). Candidate ego actions are sampled
once and stored on the node, so the transitions themselves never re-sample.

In [ ]:
from __future__ import annotations
# Mutable per-env state cloned by a snapshot. The static official-comet arrays
# (c_paths / c_len / c_ships) are read-only during stepping, so they are shared by
# reference (bound once on the work env) rather than cloned; only c_slot mutates.
_SNAP_ATTRS = (
    "p_alive", "p_owner", "p_x", "p_y", "p_radius", "p_ships", "p_prod", "p_is_comet",
    "p_init_x", "p_init_y", "p_rotates", "p_comet_vx", "p_comet_vy",
    "p_x_prev", "p_y_prev", "p_x_prev2", "p_y_prev2",   # motion history -> finite-diff velocity/curvature (v13)
    "f_alive", "f_owner", "f_x", "f_y", "f_angle", "f_ships", "f_seq",
    "ang_vel", "step_ct", "done", "c_slot",
)

PROD_WEIGHT = 20.0       # heuristic: 1 production ~ PROD_WEIGHT ships of long-run value
VALUE_SCALE = 200.0      # logistic scale for the strength-margin leaf value


def snapshot_env(env) -> dict:
    """Clone the mutable state of a GpuEnv into a plain dict (B and n_players included)."""
    snap = {a: (getattr(env, a).clone() if torch.is_tensor(getattr(env, a, None)) else None)
            for a in _SNAP_ATTRS}
    snap["_B"] = env.B
    snap["_n_players"] = int(getattr(env, "n_players", 2))
    return snap


def restore_env(env, snap: dict) -> None:
    """Overwrite ``env``'s mutable state from a snapshot. Clones on the way in so later in-place
    env_step writes (e.g. ``c_slot[...] =``) never corrupt the stored snapshot."""
    for a in _SNAP_ATTRS:
        v = snap.get(a)
        if torch.is_tensor(v):
            setattr(env, a, v.clone())
    env.B = snap["_B"]
    env.n_players = snap["_n_players"]


def heuristic_value(env, ego: int = 0) -> float:
    """Bounded position value in [0,1] for ``ego`` (B=1): logistic of the ship + fleet +
    PROD_WEIGHT*production margin against all opponents pooled. Matches settle_n's accounting."""
    alive = env.p_alive > 0.5
    owner = env.p_owner
    fa = env.f_alive > 0.5
    mine = (owner == float(ego)) & alive
    opp = (owner >= 0.0) & (owner != float(ego)) & alive
    fo = env.f_owner
    my = ((env.p_ships * mine).sum() + (env.f_ships * ((fo == float(ego)) & fa)).sum()
          + PROD_WEIGHT * (env.p_prod * mine).sum())
    op = ((env.p_ships * opp).sum() + (env.f_ships * ((fo != float(ego)) & fa)).sum()
          + PROD_WEIGHT * (env.p_prod * opp).sum())
    return float(torch.sigmoid((my - op) / VALUE_SCALE).item())


def criticality(env, ego: int = 0) -> float:
    """How pivotal this turn is, in [0,1] (B=1). High when the game is materially CLOSE -- a
    swing turn worth spending the overage bank on. Cheap (one strength-margin closeness)."""
    alive = env.p_alive > 0.5
    owner = env.p_owner
    fa = env.f_alive > 0.5
    mine = (owner == float(ego)) & alive
    opp = (owner >= 0.0) & (owner != float(ego)) & alive
    fo = env.f_owner
    my = float(((env.p_ships * mine).sum() + (env.f_ships * ((fo == float(ego)) & fa)).sum()).item())
    op = float(((env.p_ships * opp).sum() + (env.f_ships * ((fo != float(ego)) & fa)).sum()).item())
    tot = my + op
    if tot <= 0.0:
        return 1.0
    return max(0.0, 1.0 - abs(my - op) / tot)            # 1 = dead even -> most critical


class _Node:
    """One tree node: a state snapshot + K candidate edges (prior P, visits N, value sum W)."""
    __slots__ = ("snap", "cands", "P", "N", "W", "children", "terminal", "tv")

    def __init__(self, snap, cands, P, terminal=False, tv=0.0):
        K = len(cands)
        self.snap = snap
        self.cands = cands          # list of K action bundles (1, E, E+1)
        self.P = P                  # (K,) prior over candidates
        self.N = torch.zeros(K)
        self.W = torch.zeros(K)
        self.children = [None] * K
        self.terminal = terminal
        self.tv = tv                # terminal value (ego perspective) when terminal


def _candidates(cfg, net, env, ego, K):
    """K candidate full-turn actions for ``ego`` = the policy-greedy move (index 0, a safe
    fallback) + K-1 samples, with prior P = softmax of their policy log-probs.

    ONE net forward: the K candidates share an IDENTICAL observation, so the policy distribution is
    built once and sampled K times. The old code re-ran ``act`` (hence the whole trunk) per
    candidate -- K x the per-node cost, i.e. K x the entire MCTS budget -- for a distributionally
    identical result. This dedup is the difference between a real search and a depth-1 stub."""
    ent, em, am, gl = env_encode(env, ego)
    dist, _, _ = _make_dist(cfg, net, ent, em, am, gl)        # ONE trunk+heads forward for all K
    cands = [dist.greedy()] + [dist.sample() for _ in range(max(0, K - 1))]
    # Tree bookkeeping (P/N/W/Q/U) lives on CPU: it is tiny (K,) and per-sim GPU kernels +
    # .item() syncs would dominate. Candidate ACTION tensors stay on the net's device for stepping.
    P = torch.softmax(torch.stack([dist.log_prob(a).reshape(()) for a in cands]), 0).cpu()
    return cands, P


def _leaf_value(cfg, net, env, value):
    if value == "net":
        ent, em, am, gl = env_encode(env, 0)
        _, _, v = act(cfg, net, ent, em, am, gl, greedy=True)
        return float(torch.sigmoid(v.reshape(()) / VALUE_SCALE).item())
    return heuristic_value(env, 0)


def _terminal_value(s0, s1, al0, al1):
    if al0 and not al1:
        return 1.0
    if al1 and not al0:
        return 0.0
    if (not al0) and (not al1):
        return 0.5
    return 1.0 if float(s0) > float(s1) else 0.0 if float(s0) < float(s1) else 0.5


def _select(node, c_puct):
    """PUCT: argmax Q + c*P*sqrt(sum N)/(1+N); unvisited edges get a neutral FPU of 0.5."""
    N = node.N
    ntot = float(N.sum().item())
    Q = torch.where(N > 0, node.W / N.clamp_min(1.0), torch.full_like(N, 0.5))
    U = c_puct * node.P * math.sqrt(ntot + 1e-8) / (1.0 + N)
    return int(torch.argmax(Q + U).item())


def _expand_child(cfg, net, work, node, a, opp_code, K, value):
    """Apply edge ``a`` (ego candidate + scripted opponent) from ``node``'s state; return
    (backup_value, child_node). The child is terminal (game decided / horizon) or a fresh
    leaf with its own snapshot + candidates."""
    restore_env(work, node.snap)
    step_idx = int(work.step_ct[0].item())
    env_step(cfg, work, node.cands[a], opp_code, None, step_idx=step_idx)
    s0, s1, al0, al1 = settle(work)
    al0, al1 = bool(al0[0].item()), bool(al1[0].item())
    horizon = float(work.step_ct[0].item()) >= float(work.T)
    if (not al0) or (not al1) or horizon:
        v = _terminal_value(s0[0].item(), s1[0].item(), al0, al1)
        return v, _Node(None, [], None, terminal=True, tv=v)
    v = _leaf_value(cfg, net, work, value)
    child = _Node(snapshot_env(work), *_candidates(cfg, net, work, 0, K))
    return v, child


@torch.no_grad()
def mcts_search(cfg, net, work, root_snap, opp_code, n_sims=32, c_puct=1.5, K=6,
                value="heuristic", time_budget=None):
    """Run up to ``n_sims`` PUCT simulations from ``root_snap`` (ego = seat 0, opponent = scripted
    ``opp_code``) using ``work`` as the single restore-and-step env. Stops early if ``time_budget``
    (seconds) is exceeded -- the deploy wall-clock guard; the most-visited edge so far is always a
    safe answer (greedy is edge 0). Returns (best_action, root_node, stats)."""
    t0 = time.perf_counter()
    restore_env(work, root_snap)
    root = _Node(root_snap, *_candidates(cfg, net, work, 0, K))
    ran = 0
    for _ in range(n_sims):
        ran += 1
        path, node = [], root
        while True:
            a = _select(node, c_puct)
            path.append((node, a))
            ch = node.children[a]
            if ch is None:
                v, child = _expand_child(cfg, net, work, node, a, opp_code, K, value)
                node.children[a] = child
                break
            if ch.terminal:
                v = ch.tv
                break
            node = ch
        for n, a in path:
            n.N[a] += 1.0
            n.W[a] += v
        if time_budget is not None and (time.perf_counter() - t0) > time_budget:
            break
    best = int(torch.argmax(root.N).item())
    visits = root.N
    nz = visits[visits > 0]
    probs = nz / nz.sum() if nz.numel() else nz
    stats = {
        "visits": visits.tolist(),
        "sims_run": ran,
        "best_idx": best,
        "switched": best != 0,                                   # search preferred non-greedy
        "root_q": float((root.W.sum() / root.N.sum().clamp_min(1.0)).item()),
        "visit_entropy": float(-(probs * probs.clamp_min(1e-9).log()).sum().item()) if nz.numel() else 0.0,
        "elapsed": time.perf_counter() - t0,
    }
    return root.cands[best], root, stats


# (until_step, bank-seconds spent ACROSS this phase). The game is decided in the opening, so the
# per-game compute bank is front-loaded there and tapers to greedy. Per-turn budget within a phase =
# phase_seconds / phase_length: 30/20=1.5 s (steps 0-20), 20/40=0.5 s (20-60), 5/20=0.25 s (60-80),
# then greedy. 30+20+5 = 55 s of the 60 s bank; the remaining 5 s is the turbulence reserve.
DEFAULT_SCHEDULE = ((20, 30.0), (60, 20.0), (80, 5.0))


class MctsAgent:
    """Seat-0 agent that plays the most-visited MCTS move under a STEP-SCHEDULED time budget.

    ``schedule`` is a tuple of ``(until_step, phase_seconds)`` consumed in order; the per-turn
    wall-clock budget inside a phase is ``phase_seconds / phase_length`` (so :data:`DEFAULT_SCHEDULE`
    spends 1.5 / 0.5 / 0.25 s per turn over steps 0-20 / 20-60 / 60-80). Past the last phase -- or
    once ``bank_s - reserve_s`` is spent -- it plays PURE greedy (one forward, ``base_budget_s``).
    Alternatively ``crit_schedule=(every, until, secs)`` runs a SPARSE critical-step budget -- a deep
    ``secs``-second search only on steps that are a multiple of ``every`` and ``< until`` (e.g.
    ``(10, 50, 10.0)`` -> 10 s on steps 0/10/20/30/40, ~50 s of the bank), greedy on every other turn;
    it takes precedence over ``schedule``.
    ``reserve_s`` is never spent by the schedule: it absorbs inference turbulence (latency spikes) so
    one slow turn can't blow the game's total time. ``schedule=None`` -> a flat ``turn_budget_s`` every
    turn (eval/measurement). Reuses one work env; binds the static official-comet arrays on first use."""

    def __init__(self, cfg, net, opp_kind="greedy", K=6, c_puct=1.5, value="heuristic",
                 schedule=DEFAULT_SCHEDULE, crit_schedule=None, turn_budget_s=0.9, base_budget_s=0.05,
                 bank_s=60.0, reserve_s=5.0, max_sims=4096, n_sims=None,
                 crit_budget_s=None, crit_sims=None, crit_thresh=None):   # last 3: deprecated, ignored
        self.cfg = cfg
        self.net = net.eval()
        self.opp_code = _OPP_BY_KIND[opp_kind]
        self.K, self.c_puct, self.value = K, c_puct, value
        self.schedule = tuple(schedule) if schedule else None
        self.crit_schedule = tuple(crit_schedule) if crit_schedule else None   # (every, until, secs): deep search only on step%every==0 & step<until
        self.turn_budget_s, self.base_budget_s = turn_budget_s, base_budget_s
        self.bank_s, self.reserve_s = bank_s, reserve_s
        self.max_sims = int(n_sims) if n_sims is not None else int(max_sims)   # n_sims = compat alias for the cap
        self.spent_s = 0.0
        self.work = GpuEnv(cfg)
        self.work.c_paths = None
        self.last_stats = None

    def _turn_budget(self, step):
        """Scheduled per-turn wall-clock budget at ``step``, clamped so the schedule never dips into
        the turbulence reserve; ``base_budget_s`` past the schedule or once the bank is spent."""
        if self.crit_schedule is not None:               # SPARSE critical-step mode: deep search only on
            every, until, secs = self.crit_schedule      # steps that are a multiple of `every` and < `until`,
            per_turn = secs if (step < until and step % every == 0) else self.base_budget_s   # greedy elsewhere
        elif self.schedule is None:
            per_turn = self.turn_budget_s
        else:
            per_turn, lo = self.base_budget_s, 0
            for hi, secs in self.schedule:
                if step < hi:
                    per_turn = secs / max(1, hi - lo); break
                lo = hi
        avail = max(0.0, self.bank_s - self.reserve_s - self.spent_s)   # the reserve is untouchable
        return per_turn if per_turn <= avail else max(self.base_budget_s, avail)

    def act(self, game_env, step_idx=None):
        if getattr(self.work, "c_paths", None) is None and getattr(game_env, "c_paths", None) is not None:
            self.work.c_paths, self.work.c_len, self.work.c_ships = (
                game_env.c_paths, game_env.c_len, game_env.c_ships)
        step = int(step_idx if step_idx is not None else game_env.step_ct[0].item())
        budget = self._turn_budget(step)
        if budget <= self.base_budget_s:                       # past the schedule / bank spent -> pure greedy (1 fwd)
            ent, em, am, gl = env_encode(game_env, 0)
            a, _, _ = act(self.cfg, self.net, ent, em, am, gl, greedy=True)
            self.last_stats = {"step": step, "budget": 0.0, "sims_run": 0, "greedy": True,
                               "spent_s": self.spent_s, "bank_left": max(0.0, self.bank_s - self.spent_s)}
            return a, self.last_stats
        snap = snapshot_env(game_env)
        a, _root, stats = mcts_search(self.cfg, self.net, self.work, snap, self.opp_code,
                                      n_sims=self.max_sims, c_puct=self.c_puct, K=self.K,
                                      value=self.value, time_budget=budget)
        self.spent_s += max(0.0, stats["elapsed"])
        stats.update(step=step, budget=budget, greedy=False,
                     spent_s=self.spent_s, bank_left=max(0.0, self.bank_s - self.spent_s))
        self.last_stats = stats
        return a, stats


@torch.no_grad()
def play_2p_vs_script(cfg, act_fn, world, opp_kind="greedy", max_steps=None):
    """Play seat 0 (driven by ``act_fn(game_env, t) -> action``) vs a scripted seat-1 bot on one
    world. Returns (score, (s0, s1), n_steps); score = 1 win / 0.5 draw / 0 loss for seat 0."""
    opp_code = _OPP_BY_KIND[opp_kind]
    env = GpuEnv(cfg)
    env.reset([world], n_players=2)
    T = env.T if max_steps is None else min(max_steps, env.T)
    steps = 0
    for t in range(T):
        a0 = act_fn(env, t)
        env_step(cfg, env, a0, opp_code, None, step_idx=t)
        steps = t + 1
        _, _, al0, al1 = settle(env)
        if not (bool(al0[0].item()) and bool(al1[0].item())):
            break
    s0, s1, _, _ = settle(env)
    s0, s1 = float(s0[0].item()), float(s1[0].item())
    score = 1.0 if s0 > s1 else 0.0 if s0 < s1 else 0.5
    return score, (s0, s1), steps


def greedy_act_fn(cfg, net):
    """An ``act_fn`` for play_2p_vs_script that plays the raw policy-greedy move (no search)."""
    net = net.eval()

    @torch.no_grad()
    def _fn(game_env, t):
        ent, em, am, gl = env_encode(game_env, 0)
        a, _, _ = act(cfg, net, ent, em, am, gl, greedy=True)
        return a
    return _fn

## `orbit_wars_v12.plotting`

Training-health curves (notebook cell 37).

``plot_training_health(hist)`` draws each metric on its own panel (raw faint + EMA bold) from the
``hist`` dict returned by :meth:`orbit_wars_v12.train.Trainer.train`.

In [ ]:
def _ema(y, a=0.15):
    out, m = [], None
    for v in y:
        m = float(v) if m is None else (1 - a) * m + a * float(v)
        out.append(m)
    return out


def plot_training_health(hist, show=True, save_path=None):
    """Render the 3x4 training-health grid. Returns the matplotlib Figure."""
    import matplotlib.pyplot as plt

    it = hist["iter"]

    def _panel(ax, key, title, pct=False, symlog=False):
        y = hist[key]
        ax.plot(it, y, color="tab:blue", alpha=0.25, lw=0.8)
        ax.plot(it, _ema(y), color="tab:blue", lw=1.8)
        ax.set_title(title, fontsize=9); ax.grid(alpha=0.3, lw=0.5)
        if pct:
            ax.set_ylim(-0.02, 1.02)
        if symlog:
            ax.set_yscale("symlog", linthresh=1e-3)

    fig, ax = plt.subplots(3, 4, figsize=(20, 10))
    _panel(ax[0, 0], "return", "episode return (real units)")
    _panel(ax[0, 1], "win_rate", "win rate (vs sampled opp)", pct=True)
    _panel(ax[0, 2], "learner_elo", "learner Elo (2p)")
    _panel(ax[1, 0], "sigma", "exploration (sigma / mean entropy)")
    _panel(ax[1, 1], "clipfrac", "PPO clip fraction")
    _panel(ax[1, 2], "approx_kl", "approx KL (symlog)", symlog=True)
    _panel(ax[0, 3], "grad_norm", "grad norm (pre-clip, symlog)", symlog=True)
    _panel(ax[1, 3], "learner_elo4", "learner Elo (4p)")
    _panel(ax[2, 3], "share_4p", "share of 4p iterations", pct=True)
    _panel(ax[2, 0], "lnch_per_step", "launches / step")
    _panel(ax[2, 1], "r_outcome", "reward: outcome channel")
    for k, c in (("r_capture", "tab:green"), ("r_launch", "tab:orange"), ("r_milestone", "tab:purple"),
                 ("r_alive", "tab:blue"), ("r_win_bet", "tab:red")):
        if not hist.get(k) or not any(hist[k]):   # skip channels that are absent (old hist) or all-zero (feature off)
            continue
        ax[2, 2].plot(it, hist[k], color=c, alpha=0.20, lw=0.8)
        ax[2, 2].plot(it, _ema(hist[k]), color=c, lw=1.6, label=k.replace("r_", ""))
    ax[2, 2].set_title("reward: dense channels"); ax[2, 2].grid(alpha=0.3, lw=0.5); ax[2, 2].legend(fontsize=7)
    for a in ax[-1]:
        a.set_xlabel("iter")
    fig.suptitle("Orbit Wars v12 -- training health (faint = raw, bold = EMA)", fontsize=12)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=110, bbox_inches="tight")
    if show:
        plt.show()
    return fig

## MCTS deploy strategy (critical-step time budget)

**Trunk:** LEAN/fast: 28 ResNet-MLP + 2 cross-planet attention, h256 (~256x32-class) -- max sims/turn -- 30 blocks, ~5.0M params. Measured **B=1 forward ~ 6.0 ms** (2-core CPU, `torch.set_num_threads(2)`, fp32, no `torch.compile`).

**Budget.** The game is decided in the opening, so the ~60 s compute bank is spent on a few EARLY CRITICAL turns and the rest of the game plays greedy. Critical turns = every 10th step in the first 50 (steps 0/10/20/30/40 -- five turns); **each runs a deep ~10 s MCTS** (5x10 = 50 s of the 60 s bank, leaving a 10 s turbulence reserve). Every other turn is a single greedy forward (free against the bank). Leaf value = the bounded **heuristic** by default; with PPO+GAE the **critic (value head) is trained**, so `value="net"` is now available too -- A/B it (the policy-biased critic historically lost to the heuristic for leaf eval). Wire it as:

```python
agent = MctsAgent(cfg, net, opp_kind="greedy", value="heuristic", K=8,
                  crit_schedule=(10, 50, 10.0), bank_s=60.0, reserve_s=10.0)
```

At **10 s/critical turn** this net does **~1667 PUCT sims** (1 sim ~ 1 forward + 1 env step; the real count is lower by the snapshot/step overhead the PoC still pays). With a prior-pruned breadth of `b` candidates/node, depth `D ~ sims/b`:

| breadth b | 4 | 8 | 16 |
|---|---|---|---|
| depth D ~ | 416 | 208 | 104 |

A leaner/faster trunk trades per-eval quality for MORE sims/turn -- run the **deep** and **lean** notebooks side by side to see which wins under the same 10 s budget. fp16 + a lean single-state stepper would roughly 1.5-2x the sims.

## Run training

In [ ]:
# ==================== RUN SETTINGS: block-sequence trunk + PPO+GAE ====================
# Trunk (h=256, 8 heads): LEAN/fast: 28 ResNet-MLP + 2 cross-planet attention, h256 (~256x32-class) -- max sims/turn
#   res = pre-LN ResNet-MLP block ; attn = PURE cross-planet multi-head self-attention (no MLP).
# Training = PPO + GAE (the shipped default; SMOOTHER than the critic-free MC path it replaced). The
#   critic is a small SHARED-TRUNK HEAD, NOT a separate net: masked-mean pool the trunk tokens ->
#   concat globals -> val_in -> VALUE_RES_BLOCKS pre-LN ResNet-MLP blocks -> scalar, PopArt-normalized.
# GPU: A100/H100 (bf16). Colab triton crash -> `pip install triton==3.6.0` then RESTART.
SMOKE       = False     # True -> tiny net + few iters (sanity run; OVERRIDES ignored)
RESUME_FROM = None
OVERRIDES   = dict(
    ARCH="blockseq", TRUNK_SPEC="res8,attn1,res12,attn1,res8", HIDDEN=256, N_HEADS=8,
    ALGO="ppo",                                      # PPO+GAE (learned critic) -- smoother than the old ALGO="mc"
    VALUE_RES_BLOCKS=2,                              # critic HEAD depth off the shared trunk (try 1-4; deeper trunk -> smaller head)
    VALUE_WARMUP_ITERS=5,                            # critic-only warmup after BC (val-grad only) before joint PPO
    VF_COEF=0.5,                                     # actor/critic grad balance on the shared trunk (lower to ~0.25 if value distorts the policy)
    # (the MC-era DENSE_REWARD_SCALE=0.5 knob is dropped: the real critic + GAE now do the credit
    #  assignment.) Optional aux: AUX_WIN_BET=True adds a win/loss representation head; pair it with
    #  AUX_WIN_BET_REWARD=True (+ _COEF) to ALSO feed a ONE-SIDED confident-WIN bet reward into the
    #  advantage (coef*relu(tanh(bet))*max(z,0); zero on a loss -> no give-up trap). Heartbeat: R[... wb].
    GRAD_CHECKPOINT=True, USE_COMPILE=True, AMP_DTYPE=torch.bfloat16,
    NUM_GROUPS=16, GROUP_SIZE=16, TOTAL_ITERS=2000, BC_ENABLED=True,
)
cfg = Config.create(smoke=SMOKE, **(OVERRIDES if not SMOKE else {}))
print("device=%s  SMOKE=%s  ARCH=%s  HIDDEN=%d  ITERS=%d  B=%d  4p=%s"
      % (cfg.device, cfg.SMOKE, cfg.ARCH, cfg.HIDDEN, cfg.TOTAL_ITERS, cfg.B, cfg.FOURP_ENABLED))

In [ ]:
# Optional BC warm-start: clone the medium bot into the gated-alloc heads (off unless
# cfg.BC_ENABLED). On a Run-All, the train cell then warm-starts from the saved BC init.
if cfg.BC_ENABLED:
    bc_pretrain(cfg)
    if RESUME_FROM is None:
        RESUME_FROM = cfg.BC_CKPT_PATH

In [ ]:
net, hist, league = train(cfg, total_iters=cfg.TOTAL_ITERS, log_every=1, resume_from=RESUME_FROM)

## Save every league agent as weights

In [ ]:
save_league_agents(cfg, league)

## Plot training-health curves

In [ ]:
plot_training_health(hist)

## Critical-step MCTS demo (deploy strategy on the trained net)

In [ ]:
# Critical-step MCTS demo on the freshly trained net -- proves the deploy strategy runs end-to-end.
# DEPLOY uses crit_schedule=(10, 50, 10.0): a deep 10 s search on steps 0/10/20/30/40, greedy elsewhere.
# Here the critical budget + bank are SCALED DOWN so the in-notebook demo stays quick (fresh agent/world so
# each game gets the full bank). Leaf value = bounded heuristic (try value="net" too -- PPO+GAE trains the critic). B=1 = a demo.
_crit_budget = 0.2 if cfg.SMOKE else 1.0        # DEPLOY: 10.0 s/critical turn
_greedy_fn = greedy_act_fn(cfg, net)
for _i, _w in enumerate(make_world_pool(cfg, 2 if cfg.SMOKE else 4, base_seed=99991)):
    _agent = MctsAgent(cfg, net, opp_kind="greedy", value="heuristic", K=6,
                       crit_schedule=(10, 50, _crit_budget), bank_s=20.0, reserve_s=2.0)
    _sm, _scm, _ = play_2p_vs_script(cfg, lambda e, t: _agent.act(e, t)[0], _w, "greedy", max_steps=60)
    _sg, _scg, _ = play_2p_vs_script(cfg, _greedy_fn, _w, "greedy", max_steps=60)
    print("world %d  crit-MCTS %.1f (%.0f-%.0f)  |  greedy %.1f (%.0f-%.0f)  bank_left=%.1fs"
          % (_i, _sm, _scm[0], _scm[1], _sg, _scg[0], _scg[1], max(0.0, _agent.bank_s - _agent.spent_s)))